# GamaX1 / Aetherion — Complete Self-Contained Colab Training Notebook

This is the **master Colab notebook**. It contains the complete project source inside the notebook instead of depending on a single training command or a separate local checkout.

### Included project files
All **20 non-notebook project files** from the final rechecked bundle are embedded below:
- all `gamax1/*.py` modules
- `compare_dense.py`
- `prepare_large_corpus.py`
- `setup.py`
- `requirements.txt`
- math/mechanism manifest and correction log
- README/checklist/version metadata

### Training workflow
1. GPU + Drive
2. Write all project files into `/content/GamaX1_Aetherion`
3. Validate/compile every module
4. Inspect the four real corpus sources
5. Build/resume the large BPE token cache
6. Run a short GPU smoke test
7. **START FULL TRAINING** with 500-step checkpoints
8. Resume after Colab disconnect
9. Inspect timing/metrics/plots
10. Generate/chat
11. Optional instruction fine-tuning
12. Sparse-vs-dense controlled comparison
13. Mathematical/mechanism audit

The research foundation is unchanged. The notebook only makes the full implementation and training workflow self-contained and observable.


## 1. GPU + Google Drive


In [ ]:
!nvidia-smi
import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

from google.colab import drive
drive.mount("/content/drive")


## 2. Paths and master configuration


In [ ]:
from pathlib import Path
import os, json, time

PROJECT_ROOT = Path("/content/GamaX1_Aetherion")
DATA_ROOT = Path("/content/drive/MyDrive/Aetherion_GamaX1/data")

# Large-corpus BPE cache. This is persistent on Drive so a Colab reconnect
# does not force the entire corpus to be encoded again.
BULK_CACHE = Path("/content/drive/MyDrive/Aetherion_GamaX1/cache/bulk_bpe")

# Persistent model checkpoints and experiment logs.
CKPT_DIR = Path("/content/drive/MyDrive/Aetherion_GamaX1/checkpoints/gamax1_final_v7")
EXPERIMENT_DIR = Path("/content/drive/MyDrive/Aetherion_GamaX1/experiments")

# Full training configuration from the final v7 workflow.
MODEL = {
    "d_model": 768,
    "n_heads": 12,
    "n_layers": 12,
    "n_features": 3072,
    "block_size": 512,
    "batch_size": 16,
    "bpe_vocab_size": 16000,
    "max_steps": 30000,
    "eval_interval": 200,
    "checkpoint_interval": 500,
    "lr": 3e-4,
    "dropout": 0.1,
}

# A short smoke test uses smaller settings only to prove the pipeline works.
SMOKE = {
    "d_model": 64,
    "n_heads": 2,
    "n_layers": 2,
    "n_features": 256,
    "block_size": 128,
    "batch_size": 4,
    "max_steps": 10,
    "eval_interval": 5,
    "checkpoint_interval": 5,
}

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_ROOT:", DATA_ROOT)
print("BULK_CACHE:", BULK_CACHE)
print("CKPT_DIR:", CKPT_DIR)
print("EXPERIMENT_DIR:", EXPERIMENT_DIR)


## 3. Create the complete project — all 20 source/config files


The next 20 cells are deliberately separate: each cell creates **one exact project file** inside the Colab runtime. This makes the notebook self-contained and lets you inspect/run each file independently.

The two notebook files from the ZIP are intentionally not embedded recursively; this notebook is the master notebook.


In [ ]:
# FILE: README_COMMENTED.md
from pathlib import Path

path = PROJECT_ROOT / 'README_COMMENTED.md'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text("# GamaX1 — commented/fixed code bundle\n\nThis bundle is the readable working copy of the GamaX1/Aetherion training\npipeline. Comments are intentionally explanatory rather than decorative:\nthey document **why** a piece exists, what it saves, when it resumes, and what\ncan break if it is changed.\n\n## Checkpoint meanings\n\n- **Every 500 encoded files** → corpus checkpoint (`encode_progress.json` + flushed token stream).\n- **Every 500 training steps** → model checkpoint (`gamax1_step_N.pt` + `gamax1_latest.pt`).\n\nThese are different checkpoints and must not be confused.\n\n## Important architecture note\n\nThe current GamaX1 implementation uses standard causal self-attention for\ntoken mixing and applies the Aetherion sparse-superposition mechanism to the\nFFN/feed-forward part. It does **not** claim that the present code has already\nreplaced Transformer attention with DICE.\n\n## Suggested order\n\n1. Verify/import package.\n2. Verify the four corpus sources.\n3. Build/reuse the persistent BPE tokenizer.\n4. Encode the corpus incrementally; let 500-file checkpoints accumulate.\n5. Run a small real-data training smoke test.\n6. Start full pretraining; resume from `gamax1_latest.pt` after interruption.\n7. Evaluate separately on held-out data.\n8. Fine-tune with instruction/Q&A data only after the base model is stable.\n9. Run sparse-vs-dense comparison as a controlled experiment.\n\n## Generation change\n\nRepetition penalty is applied to **generated answer tokens only**. The user's\nprompt is context and is not itself penalized. This keeps the decoding control\nfrom treating words in the question as undesirable merely because they occur\nin the prompt.\n\n\n## Final v7 additions\n\n- `gamax1/experiment_tracker.py`: append-only per-run metrics, summaries, plots, and run comparison.\n- Training records loss, validation loss, perplexity, LR, sparsity/activity, local compute proxy, and checkpoint intervals.\n- `checkpoint_timing.jsonl` records measured seconds between 500-step checkpoints; it does not pretend to predict GPU speed from a formula.\n- `encoding_checkpoint_timing.jsonl` records measured seconds/files-per-second/tokens-per-second between 500-file corpus checkpoints plus an ETA estimate for the current batch.\n- The ETA is explicitly an estimate derived from the latest measured interval, not a guaranteed completion time.\n- Dynamic sparsity now clamps restored/current `k` to the declared `[k_min, k_max]` interval.\n- `__pycache__` artifacts are excluded from the deliverable ZIP.\n\n## What remains deliberately unchanged\n\nThe Aetherion research question/foundation is unchanged. Current GamaX1 still uses causal self-attention for token mixing and applies the Aetherion sparse-superposition mechanism in the FFN path. No benchmark result is upgraded to a claim of Transformer superiority without a measured controlled experiment.\n", encoding="utf-8")
print("Wrote:", path)
print("Bytes:", path.stat().st_size)


In [ ]:
# FILE: requirements.txt
from pathlib import Path

path = PROJECT_ROOT / 'requirements.txt'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text('torch>=2.0.0\n', encoding="utf-8")
print("Wrote:", path)
print("Bytes:", path.stat().st_size)


In [ ]:
# FILE: setup.py
from pathlib import Path

path = PROJECT_ROOT / 'setup.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text('from setuptools import setup, find_packages\n\nsetup(\n    name="gamax1",\n    version="2.0.0",\n    description="GamaX1 -- first working version of the Aetherion architecture as a real NLP language model.",\n    author="MrRoy",\n    packages=find_packages(include=["gamax1", "gamax1.*"]),\n    install_requires=["torch>=2.0.0"],\n    python_requires=">=3.9",\n)\n', encoding="utf-8")
print("Wrote:", path)
print("Bytes:", path.stat().st_size)


In [ ]:
# FILE: MATH_CORRECTION_LOG.md
from pathlib import Path

path = PROJECT_ROOT / 'MATH_CORRECTION_LOG.md'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text('# Mathematical / Mechanism Correction Log\nRecord Date, Component, Original rule, Observed inconsistency, Corrected rule,\nCode change, Unit test, Re-run result, and affected paper sections.\nCorrections are made for mathematical/mechanistic correctness, never just for score improvement.\n', encoding="utf-8")
print("Wrote:", path)
print("Bytes:", path.stat().st_size)


In [ ]:
# FILE: FINAL_RECHECK_CHECKLIST.md
from pathlib import Path

path = PROJECT_ROOT / 'FINAL_RECHECK_CHECKLIST.md'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text('# FINAL RECHECK CHECKLIST — v7\n\n- [x] 500-file corpus checkpoint\n- [x] corpus checkpoint timing + files/s + tokens/s + ETA estimate\n- [x] 500-step model checkpoint default\n- [x] training checkpoint timing + steps/s\n- [x] atomic checkpoint writes\n- [x] resume model/optimizer/sparsity/PTM/scaler/RNG\n- [x] generated-token-only repetition penalty\n- [x] chat role tokens + EOS support\n- [x] six stable BPE special-token IDs\n- [x] sparse k clamped to [k_min, k_max]\n- [x] boundary-safe hex neighbor mask\n- [x] mathematical mechanism audit\n- [x] per-run metrics.jsonl + config + summary + plots\n- [x] separate base pretraining and instruction/SFT flow\n- [x] sparse-vs-dense controlled comparison\n- [x] clean Colab notebook with smoke/full/resume/generation/audit sections\n- [x] no __pycache__ artifacts in deliverable\n- [x] no change to Aetherion research foundation\n', encoding="utf-8")
print("Wrote:", path)
print("Bytes:", path.stat().st_size)


In [ ]:
# FILE: prepare_large_corpus.py
from pathlib import Path

path = PROJECT_ROOT / 'prepare_large_corpus.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text('"""\nprepare_large_corpus.py (v2 -- multi-book combiner)\n======================================================\nDownloads a curated list of long, well-known public-domain novels from\nProject Gutenberg, strips each one\'s boilerplate, and concatenates\nthem into a single large training corpus for GamaX1.\n\nRun this on your own machine (needs internet access):\n    python prepare_large_corpus.py                     # default range: Gutenberg IDs 1-400\n    python prepare_large_corpus.py --gutenberg_id 100   # single book, unchanged v1 behavior\n    python prepare_large_corpus.py --ids 2600 100 1400 2701   # your own list of IDs\n\nA default curated list of long public-domain novels/plays is provided\nbelow, chosen for length and lexical variety (mixed authors/genres\nhelps a language model generalize better than many similar texts).\n"""\n\nimport argparse\nimport re\nimport time\nimport urllib.request\n\nSTART_MARKERS = [\n    re.compile(r"\\*\\*\\*\\s*START OF (THE|THIS) PROJECT GUTENBERG EBOOK.*?\\*\\*\\*", re.IGNORECASE | re.DOTALL),\n]\nEND_MARKERS = [\n    re.compile(r"\\*\\*\\*\\s*END OF (THE|THIS) PROJECT GUTENBERG EBOOK.*", re.IGNORECASE | re.DOTALL),\n]\n\n# Curated list of long, well-known, public-domain novels (Gutenberg IDs),\n# chosen for length + author/genre variety. Rough combined size: several\n# million words when all succeed.\n# which one is actually used by default.\nCURATED_BOOK_IDS = [\n    100,    # Complete Works of William Shakespeare\n    98,     # A Tale of Two Cities\n    84,     # Frankenstein\n    76,     # Adventures of Huckleberry Finn\n    1342,   # Pride and Prejudice\n    1400,   # Great Expectations\n    145,    # Middlemarch\n    1661,   # The Adventures of Sherlock Holmes\n    1952,   # The Yellow Wallpaper\n    2554,   # Crime and Punishment\n    2600,   # War and Peace\n    2701,   # Moby-Dick\n    4300,   # Ulysses\n    5200,   # Metamorphosis\n    1260,   # Jane Eyre\n    1497,   # The Republic\n    2000,   # Don Quixote\n    74,     # The Adventures of Tom Sawyer\n    11,     # Alice\'s Adventures in Wonderland\n    16,     # Peter Pan\n    35,     # The Time Machine\n    36,     # The War of the Worlds\n    43,     # The Strange Case of Dr Jekyll and Mr Hyde\n    46,     # A Christmas Carol\n    55,     # The Wonderful Wizard of Oz\n    73,     # The Red Badge of Courage\n    120,    # Treasure Island\n    1232,   # The Prince\n    158,    # Emma\n    161,    # Sense and Sensibility\n    209,    # The Turn of the Screw\n    2148,   # The Odyssey\n    219,    # Heart of Darkness\n    236,    # The Jungle Book\n    244,    # A Study in Scarlet\n    27827,  # The Kama Sutra\n    28054,  # The Brothers Karamazov\n    2814,   # Dubliners\n    30254,  # Beyond Good and Evil\n    3207,   # Leviathan\n    3300,   # The Bible, King James Version\n    345,    # Dracula\n    34901,  # On Liberty\n    3600,   # The Scarlet Letter\n    408,    # The Souls of Black Folk\n    4217,   # A Portrait of the Artist as a Young Man\n    4363,   # Beyond Good and Evil (alt edition)\n    514,    # Little Women\n    521,    # Paradise Lost\n    6130,   # The Iliad\n    6133,   # The Aeneid\n    730,    # Oliver Twist\n    768,    # Wuthering Heights\n    829,    # Gulliver\'s Travels\n    863,    # The Mysterious Affair at Styles\n    8800,   # Siddhartha\n    996,    # Don Juan\n    10,     # The King James Bible\n    45,     # Anne of Green Gables\n    1080,   # A Modest Proposal\n    108,    # The Return of Sherlock Holmes\n    1399,   # Anna Karenina\n    174,    # The Picture of Dorian Gray\n    1998,   # Thus Spoke Zarathustra\n    205,    # Walden\n    2147,   # The Works of Edgar Allan Poe\n    26184,  # Simple Sabotage Field Manual\n    27805,  # The Art of War\n    3206,   # The Federalist Papers\n    3825,   # Pygmalion\n    5000,   # The Notebooks of Leonardo da Vinci\n    55201,  # Meditations\n    61,     # The Communist Manifesto\n    74,     # Tom Sawyer\n    768,    # Wuthering Heights\n    902,    # The Happy Prince\n    962,    # The Last of the Mohicans\n    103,    # Around the World in Eighty Days\n    164,    # Twenty Thousand Leagues Under the Seas\n    2781,   # The Divine Comedy\n    4363,   # Nietzsche Collection\n    7370,   # Second Treatise of Government\n    7371,   # Essay Concerning Human Understanding\n    7372,   # Civil Government\n    500,    # The Moonstone\n    5740,   # The Art of Money Getting\n    5744,   # Essays of Michel de Montaigne\n    4368,   # The Golden Bough\n    600,    # Notes from Underground\n    2448,   # Candide\n    1934,   # The Secret Garden\n    215,    # The Call of the Wild\n    910,    # White Fang\n]\n\n# The plain-range preset: Gutenberg IDs 1-400. Unavailable or non-book IDs\n# are skipped by the downloader below and reported in the final summary.\nRANGE_BOOK_IDS = list(range(1, 401))\n\n# This IS the actual default used when neither --ids nor --preset is given\n# (see --preset\'s choices/default below) -- explicit, not a silent\n# overwrite of the curated list.\nDEFAULT_PRESET = "range_1_400"\nPRESETS = {"curated": CURATED_BOOK_IDS, "range_1_400": RANGE_BOOK_IDS}\n\n\ndef strip_gutenberg_boilerplate(text: str) -> str:\n    for pattern in START_MARKERS:\n        m = pattern.search(text)\n        if m:\n            text = text[m.end():]\n            break\n    for pattern in END_MARKERS:\n        m = pattern.search(text)\n        if m:\n            text = text[:m.start()]\n            break\n    return text.strip()\n\n\ndef download_book(gutenberg_id: int) -> str:\n    urls_to_try = [\n        f"https://www.gutenberg.org/cache/epub/{gutenberg_id}/pg{gutenberg_id}.txt",\n        f"https://www.gutenberg.org/files/{gutenberg_id}/{gutenberg_id}-0.txt",\n        f"https://www.gutenberg.org/files/{gutenberg_id}/{gutenberg_id}.txt",\n    ]\n    for url in urls_to_try:\n        try:\n            req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})\n            with urllib.request.urlopen(req, timeout=30) as response:\n                return response.read().decode("utf-8", errors="ignore")\n        except Exception:\n            continue\n    return None\n\n\ndef main():\n    parser = argparse.ArgumentParser(description="Download and combine multiple Gutenberg books for GamaX1 training.")\n    parser.add_argument("--gutenberg_id", type=int, default=None,\n                         help="Download a single book by ID (overrides --ids/--preset).")\n    parser.add_argument("--ids", type=int, nargs="+", default=None,\n                         help="Download and combine a custom list of Gutenberg IDs (overrides --preset).")\n    parser.add_argument("--preset", choices=sorted(PRESETS), default=DEFAULT_PRESET,\n                         help=f"Which built-in ID list to use when --ids/--gutenberg_id are not given: "\n                              f"\'curated\' ({len(PRESETS[\'curated\'])} hand-picked long, well-known novels) or "\n                              f"\'range_1_400\' (plain Gutenberg IDs 1-400, unfiltered). Default: \'{DEFAULT_PRESET}\'.")\n    parser.add_argument("--out", type=str, default="data/sample_corpus_combined.txt")\n    parser.add_argument("--delay", type=float, default=1.0,\n                         help="Seconds to wait between downloads, polite to Gutenberg\'s servers (default: 1.0).")\n    args = parser.parse_args()\n\n    if args.gutenberg_id is not None:\n        book_ids = [args.gutenberg_id]\n    elif args.ids is not None:\n        book_ids = args.ids\n    else:\n        book_ids = PRESETS[args.preset]\n        print(f"Using preset \'{args.preset}\' ({len(book_ids)} ids). Pass --preset to choose the other one.")\n\n    combined_parts = []\n    succeeded, failed = [], []\n\n    for i, gid in enumerate(book_ids):\n        print(f"[{i+1}/{len(book_ids)}] Downloading Gutenberg ID {gid} ...")\n        raw = download_book(gid)\n        if raw is None:\n            print(f"  FAILED to download ID {gid} (skipping)")\n            failed.append(gid)\n            continue\n        cleaned = strip_gutenberg_boilerplate(raw)\n        if len(cleaned) < 1000:\n            print(f"  WARNING: cleaned text for ID {gid} looks suspiciously short ({len(cleaned)} chars), skipping")\n            failed.append(gid)\n            continue\n        combined_parts.append(cleaned)\n        succeeded.append(gid)\n        print(f"  OK: {len(cleaned):,} characters")\n        if i < len(book_ids) - 1:\n            time.sleep(args.delay)\n\n    if not combined_parts:\n        print("\\nNo books downloaded successfully. Check your internet connection, or download")\n        print("plain-text (.txt) files manually from https://www.gutenberg.org/ and run:")\n        print("  python prepare_large_corpus.py --ids <space-separated Gutenberg IDs>")\n        return\n\n    combined_text = "\\n\\n".join(combined_parts)\n    with open(args.out, "w", encoding="utf-8") as f:\n        f.write(combined_text)\n\n    print(f"\\n{\'=\'*70}")\n    print(f"Combined {len(succeeded)}/{len(book_ids)} books successfully.")\n    if failed:\n        print(f"Failed IDs (skipped): {failed}")\n    print(f"Saved combined corpus to {args.out}")\n    print(f"Total size: {len(combined_text):,} characters (~{len(combined_text.split()):,} words)")\n    print(f"{\'=\'*70}")\n    print(f"\\nNow train with, e.g.:")\n    print(f"  python -m gamax1.train --tokenizer word --data {args.out} --auto_size_model --max_steps 3000")\n\n\nif __name__ == "__main__":\n    main()\n', encoding="utf-8")
print("Wrote:", path)
print("Bytes:", path.stat().st_size)


In [ ]:
# FILE: MATH_MECHANISM_MANIFEST.json
from pathlib import Path

path = PROJECT_ROOT / 'MATH_MECHANISM_MANIFEST.json'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text('{\n  "version": "aetherion-math-audit-v1",\n  "foundation_invariants": [\n    "0 <= k <= n_features",\n    "exact top-k has k active features without nudge",\n    "active_fraction = active_features / total_feature_capacity",\n    "sparse/dense local work ratio = k / n_features"\n  ],\n  "policy": {\n    "do_not_change_formula_to_improve_score": true,\n    "record_failed_audits": true,\n    "paper_update_after_retest": true\n  }\n}', encoding="utf-8")
print("Wrote:", path)
print("Bytes:", path.stat().st_size)


In [ ]:
# FILE: compare_dense.py
from pathlib import Path

path = PROJECT_ROOT / 'compare_dense.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text('"""Train matched sparse and dense GamaX1 variants for a small fair comparison."""\n\nimport argparse\nimport os\n\nimport torch\n\nfrom gamax1.model import GamaX1Model\nfrom gamax1.tokenizer import BPETokenizer, CharTokenizer, WordTokenizer\nfrom gamax1.train import get_batch, perplexity\n\n\ndef evaluate(model, data, block_size, batch_size, device, eval_batches=20, seed=1337):\n    """Deterministic multi-batch evaluation so one lucky random batch cannot decide the result."""\n    model.eval()\n    generator = torch.Generator(device="cpu").manual_seed(seed)\n    losses = []\n    with torch.no_grad():\n        for _ in range(max(1, eval_batches)):\n            ix = torch.randint(len(data) - block_size, (batch_size,), generator=generator)\n            xb = torch.stack([data[int(i):int(i) + block_size] for i in ix]).to(device=device, dtype=torch.long)\n            yb = torch.stack([data[int(i) + 1:int(i) + 1 + block_size] for i in ix]).to(device=device, dtype=torch.long)\n            _, loss = model(xb, targets=yb, k=model.sparsity_ctrl.k, use_ptm=False)\n            losses.append(float(loss))\n    return sum(losses) / len(losses)\n\n\ndef train_pair(sparse, dense, train_data, val_data, args, device):\n    """Train both models on identical sampled batches and optimizer settings."""\n    sparse_opt = torch.optim.AdamW(sparse.parameters(), lr=args.lr)\n    dense_opt = torch.optim.AdamW(dense.parameters(), lr=args.lr)\n    last_losses = {"sparse": float("nan"), "dense": float("nan")}\n    # PTM is intentionally disabled here: this script compares the sparse FFN\n    # mechanism itself against dense, not the extra bookkeeping mechanism.\n    # Dropout is also disabled by default in main() for a paired mechanism test.\n    for _ in range(args.steps):\n        xb, yb = get_batch(train_data, args.block_size, args.batch_size, device)\n        for name, model, optimizer in (("sparse", sparse, sparse_opt), ("dense", dense, dense_opt)):\n            model.train()\n            _, loss = model(xb, targets=yb, k=model.sparsity_ctrl.k, use_ptm=False)\n            optimizer.zero_grad(set_to_none=True)\n            loss.backward()\n            optimizer.step()\n            last_losses[name] = loss.detach().item()\n    return {\n        "sparse": (last_losses["sparse"], evaluate(sparse, val_data, args.block_size, args.batch_size, device, args.eval_batches)),\n        "dense": (last_losses["dense"], evaluate(dense, val_data, args.block_size, args.batch_size, device, args.eval_batches)),\n    }\n\n\ndef main():\n    parser = argparse.ArgumentParser(description="Compare GamaX1 sparse FFN against a matched dense FFN.")\n    parser.add_argument("--data", default=os.path.join(os.path.dirname(__file__), "..", "data", "sample_corpus.txt"))\n    parser.add_argument("--tokenizer", choices=("char", "word", "bpe"), default="char")\n    parser.add_argument("--bpe_vocab_size", type=int, default=8000)\n    parser.add_argument("--steps", type=int, default=300)\n    parser.add_argument("--d_model", type=int, default=64)\n    parser.add_argument("--n_heads", type=int, default=2)\n    parser.add_argument("--n_layers", type=int, default=2)\n    parser.add_argument("--n_features", type=int, default=256)\n    parser.add_argument("--block_size", type=int, default=64)\n    parser.add_argument("--batch_size", type=int, default=16)\n    parser.add_argument("--lr", type=float, default=3e-4)\n    parser.add_argument("--dropout", type=float, default=0.0,\n                        help="Dropout for the comparison; 0 keeps sparse/dense paired and deterministic.")\n    parser.add_argument("--eval_batches", type=int, default=20)\n    parser.add_argument("--device", default=None)\n    args = parser.parse_args()\n    device = args.device or ("cuda" if torch.cuda.is_available() else "cpu")\n    with open(args.data, encoding="utf-8") as f:\n        text = f.read()\n    if args.tokenizer == "bpe":\n        tok = BPETokenizer(text, vocab_size=args.bpe_vocab_size, sample_chars=3_000_000)\n    else:\n        tok = (WordTokenizer if args.tokenizer == "word" else CharTokenizer)(text)\n    data = torch.tensor(tok.encode(text), dtype=torch.long)\n    n = int(0.9 * len(data))\n    train_data, val_data = data[:n], data[n:]\n\n    common = dict(vocab_size=tok.vocab_size, d_model=args.d_model, n_heads=args.n_heads,\n                  n_layers=args.n_layers, n_features=args.n_features, max_seq_len=args.block_size,\n                  sparsity_k_init=max(1, args.n_features // 2), sparsity_k_min=max(1, args.n_features // 8),\n                  dropout=args.dropout)\n    torch.manual_seed(0)\n    sparse = GamaX1Model(**common).to(device)\n    torch.manual_seed(0)\n    dense = GamaX1Model(**common, dense_mode=True).to(device)\n    results = train_pair(sparse, dense, train_data, val_data, args, device)\n\n    sparse_compute = sparse.active_units_per_token()\n    dense_compute = dense.active_units_per_token()\n    ratio = dense_compute / max(sparse_compute, 1)\n    sparse_val, dense_val = results["sparse"][1], results["dense"][1]\n    sparse_ppl = perplexity(sparse_val)\n    dense_ppl = perplexity(dense_val)\n    ppl_ratio = sparse_ppl / max(dense_ppl, 1e-12)\n    print("\\nFinal comparison")\n    print("model  | train_loss | val_loss | val_ppl | parameters | active_units/token")\n    for name, model in (("sparse", sparse), ("dense ", dense)):\n        train_loss, val_loss = results[name.strip()]\n        print(f"{name} | {train_loss:10.4f} | {val_loss:8.4f} | {perplexity(val_loss):7.2f} | "\n              f"{sum(p.numel() for p in model.parameters()):10,d} | {model.active_units_per_token():18,d}")\n    print(f"\\nCompute ratio: {ratio:.2f}x (sparse uses {sparse_compute / dense_compute * 100:.1f}% of dense compute)")\n    print(f"Validation PPL ratio (sparse / dense): {ppl_ratio:.3f}x "\n          f"(1.0 means equal PPL; lower is better).")\n\n\nif __name__ == "__main__":\n    main()\n\n# The printed table remains the human-readable result; JSON output is written by the run\n# command only when --output is supplied in future versions. Keep this script free of\n# benchmark claims: it reports measured sparse/dense values only.\n', encoding="utf-8")
print("Wrote:", path)
print("Bytes:", path.stat().st_size)


In [ ]:
# FILE: VERSION.txt
from pathlib import Path

path = PROJECT_ROOT / 'VERSION.txt'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text('GamaX1 Aetherion final consolidated recheck v7\nCreated: 2026-09-26\n', encoding="utf-8")
print("Wrote:", path)
print("Bytes:", path.stat().st_size)


In [ ]:
# FILE: gamax1/__init__.py
from pathlib import Path

path = PROJECT_ROOT / 'gamax1/__init__.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text('"""GamaX1 runtime package.\n\nThis file is intentionally tiny. The real implementation lives in the\nmodules next to it. Keeping the package marker explicit makes both\n`import gamax1` and `python -m gamax1.train` predictable in Colab and Linux.\n"""\n', encoding="utf-8")
print("Wrote:", path)
print("Bytes:", path.stat().st_size)


In [ ]:
# FILE: gamax1/tokenizer.py
from pathlib import Path

path = PROJECT_ROOT / 'gamax1/tokenizer.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text('"""\ngamax1/tokenizer.py\n====================\nDependency-free tokenizers for GamaX1.\n\n- ``CharTokenizer``: character-level baseline.\n- ``WordTokenizer``: word/punctuation level with ``<unk>`` and an\n  optional frequency cap.\n- ``BPETokenizer``: a GPT-2-style byte-level BPE (subword) tokenizer.\n  This is the recommended choice for real corpora: it needs no\n  external dependency, never emits ``<unk>`` (any byte sequence is\n  representable), and produces far more training tokens per byte than\n  word tokenization, which is what actually drives the quality of\n  next-token language modeling on a corpus like the bundled one.\n\nAll tokenizers expose the same interface (``encode``/``decode``/\n``vocab_size``/``save``/``load``) so swapping between them is a one-line\nchange, and ``generate`` restores the right one from the checkpoint.\n"""\n\nimport heapq\nimport json\nimport re\nfrom collections import Counter\n\n\ndef word_tokenizer_warning(tokenizer_type: str, token_count: int, vocab_size: int):\n    """Return a useful warning when a word vocabulary lacks training signal.\n\n    A tiny word corpus gives most words too few repeated contexts to learn a\n    next-word distribution. Character tokenization remains a better default in\n    that situation, but this is deliberately advisory rather than a blocker.\n    """\n    if tokenizer_type == "word" and (token_count < 2000 or vocab_size < 200):\n        return (f"[WARNING] Word-level tokenizer built from only {token_count} word occurrences "\n                f"and a vocabulary of {vocab_size} words. This is likely too small for word-level "\n                "generation to produce coherent text -- consider using --tokenizer char instead, "\n                "or a larger corpus (--corpus large / a bigger --data file).")\n    return None\n\n\nclass CharTokenizer:\n    """Simple character-level tokenizer.\n\n    Every character in the training vocabulary maps to one token id.\n    Unlike BPE, this tokenizer does not use byte merges or special-token\n    machinery. It is intentionally simple and deterministic for baseline\n    and smoke-test training.\n    """\n\n    def __init__(self, text: str = None, vocab: list = None):\n        if vocab is not None:\n            self.chars = list(vocab)\n        else:\n            if text is None:\n                raise ValueError("text is required when vocab is not supplied")\n            self.chars = sorted(set(text))\n\n        if not self.chars:\n            raise ValueError("character vocabulary cannot be empty")\n\n        self.stoi = {ch: i for i, ch in enumerate(self.chars)}\n        self.itos = {i: ch for i, ch in enumerate(self.chars)}\n\n    @property\n    def vocab_size(self):\n        return len(self.chars)\n\n    def encode(self, text: str):\n        """Encode each character into its vocabulary id."""\n        try:\n            return [self.stoi[ch] for ch in text]\n        except KeyError as exc:\n            ch = exc.args[0]\n            raise ValueError(\n                f"Character {ch!r} is not present in the tokenizer vocabulary"\n            ) from None\n\n    def decode(self, ids):\n        """Decode token ids back into characters."""\n        try:\n            return "".join(self.itos[int(i)] for i in ids)\n        except KeyError as exc:\n            raise ValueError(\n                f"Token id {exc.args[0]} is not present in the tokenizer vocabulary"\n            ) from None\n\n    def save(self, path: str):\n        with open(path, "w") as f:\n            json.dump(self.chars, f, ensure_ascii=False)\n\n    @classmethod\n    def load(cls, path: str):\n        with open(path) as f:\n            chars = json.load(f)\n        return cls(vocab=chars)\n\n\nclass WordTokenizer:\n    """Small dependency-free word/punctuation tokenizer.\n\n    Keeping punctuation as its own token lets generated text retain readable\n    sentence boundaries while ``<unk>`` makes inference safe for prompts that\n    contain vocabulary not seen during training.\n    """\n\n    unk_token = "<unk>"\n    _pattern = re.compile(r"\\w+|[^\\w\\s]", re.UNICODE)\n\n    def __init__(self, text: str = None, vocab: list = None, max_vocab_size: int = None):\n        """Build a vocabulary, optionally capped with ``<unk>`` included.\n\n        ``max_vocab_size`` counts every entry, including the reserved ``<unk>``\n        token. The remaining slots retain the most frequent source tokens;\n        frequency ties are resolved lexically for reproducible checkpoints.\n        """\n        if vocab is not None:\n            self.tokens = vocab\n        else:\n            if text is None:\n                raise ValueError("text is required when vocab is not supplied")\n            if max_vocab_size is not None and max_vocab_size < 1:\n                raise ValueError("max_vocab_size must be at least 1 when provided")\n            counts = Counter(token for token in self._tokenize(text) if token != self.unk_token)\n            ranked_tokens = sorted(counts, key=lambda token: (-counts[token], token))\n            if max_vocab_size is not None:\n                ranked_tokens = ranked_tokens[:max_vocab_size - 1]\n            self.tokens = [self.unk_token] + ranked_tokens\n        if self.unk_token not in self.tokens:\n            self.tokens.insert(0, self.unk_token)\n        self.stoi = {token: i for i, token in enumerate(self.tokens)}\n        self.itos = {i: token for i, token in enumerate(self.tokens)}\n\n    @classmethod\n    def _tokenize(cls, text: str):\n        return cls._pattern.findall(text)\n\n    @property\n    def vocab_size(self):\n        return len(self.tokens)\n\n    def encode(self, text: str):\n        unk_id = self.stoi[self.unk_token]\n        return [self.stoi.get(token, unk_id) for token in self._tokenize(text)]\n\n    def decode(self, ids):\n        tokens = [self.itos.get(int(i), self.unk_token) for i in ids]\n        text = ""\n        no_space_before = set(".,!?;:%)]}")\n        no_space_after = set("([{")\n        for token in tokens:\n            if not text or token in no_space_before or text[-1] in no_space_after:\n                text += token\n            else:\n                text += " " + token\n        return text\n\n    def save(self, path: str):\n        with open(path, "w") as f:\n            json.dump(self.tokens, f)\n\n    @classmethod\n    def load(cls, path: str):\n        with open(path) as f:\n            tokens = json.load(f)\n        return cls(vocab=tokens)\n\n\nclass BPETokenizer:\n    """Dependency-free byte-level BPE tokenizer, GPT-2 style.\n\n    Text is pre-tokenized into words and punctuation (whitespace kept\n    attached to the following word), each unit is split into UTF-8\n    bytes, and the most frequent adjacent byte pairs are merged until\n    ``vocab_size`` is reached. Merges never cross word boundaries,\n    which keeps encoding deterministic and reproducible.\n\n    The base vocabulary is the 256 byte values; every merge adds one\n    token that subsumes a byte sequence. Because any byte sequence is\n    representable, ``decode`` never falls back to ``<unk>`` -- the\n    dominant failure mode of word-level generation on a capped\n    vocabulary.\n\n    ``sample_chars`` caps how much of the training text is scanned for\n    pair statistics. Larger values produce a slightly better vocabulary\n    but cost linear time in Python, so the default of a few megabytes is\n    a deliberate speed/quality trade-off.\n    """\n\n    _pattern = re.compile(r"\\s*\\d|\\s*[^\\W\\d]+|\\s+|[^\\w\\s]+", re.UNICODE)\n    min_pair_count = 2\n    progress_interval_chars = 50_000_000\n\n    # Reserved ids, in order, immediately after the last BPE merge id.\n    # See ``special_token_ids``/``eos_id``/``user_id``/``assistant_id``/\n    # ``pad_id`` below. Order matters for id stability across\n    # save/load -- append new entries at the end, never reorder or\n    # remove an existing one (that would silently reassign an id a\n    # trained checkpoint already relies on).\n    SPECIAL_TOKENS = ("<|eos|>", "<|user|>", "<|assistant|>", "<|pad|>", "<|think|>", "<|/think|>")\n\n    def __init__(self, text: str = None, vocab_size: int = 8000, merges: list = None,\n                 sample_chars: int = 3_000_000):\n        if merges is not None:\n            self.merges = [tuple(m) for m in merges]\n        else:\n            if text is None:\n                raise ValueError("text is required when merges are not supplied")\n            if vocab_size < 256:\n                raise ValueError("vocab_size must be at least 256 for byte-level BPE")\n            self.merges = self._train(text, vocab_size, sample_chars)\n        self._build_vocab()\n\n    # -- vocabulary construction -------------------------------------------\n\n    def _train(self, text: str, vocab_size: int, sample_chars: int) -> list:\n        if sample_chars is not None and len(text) > sample_chars:\n            text = text[:sample_chars]\n        units = [list(unit.encode("utf-8")) for unit in self._pattern.findall(text)]\n        if not units:\n            return []\n\n        counts = Counter()\n        occurrences = {}\n        for idx, ids in enumerate(units):\n            for a, b in zip(ids, ids[1:]):\n                counts[(a, b)] += 1\n                occurrences.setdefault((a, b), set()).add(idx)\n\n        heap = []\n        for pair, count in counts.items():\n            heapq.heappush(heap, (-count, pair))\n\n        merges = []\n        next_id = 256\n        while len(merges) < vocab_size - 256:\n            pair = None\n            while heap:\n                neg_count, candidate = heapq.heappop(heap)\n                current = counts.get(candidate, 0)\n                if current != -neg_count:\n                    continue  # stale heap entry\n                if current < self.min_pair_count:\n                    pair = None\n                    break\n                pair = candidate\n                break\n            if pair is None:\n                break\n            a, b = pair\n            counts.pop(pair, None)\n            new_id = next_id\n            next_id += 1\n            merges.append(pair)\n\n            for idx in list(occurrences.get(pair, ())):\n                old_ids = units[idx]\n                new_ids = []\n                i = 0\n                while i < len(old_ids):\n                    if i + 1 < len(old_ids) and old_ids[i] == a and old_ids[i + 1] == b:\n                        new_ids.append(new_id)\n                        i += 2\n                    else:\n                        new_ids.append(old_ids[i])\n                        i += 1\n                units[idx] = new_ids\n                old_counts = Counter(zip(old_ids, old_ids[1:]))\n                new_counts = Counter(zip(new_ids, new_ids[1:]))\n                for p, delta in (new_counts - old_counts).items():\n                    counts[p] = counts.get(p, 0) + delta\n                    occurrences.setdefault(p, set()).add(idx)\n                    heapq.heappush(heap, (-counts[p], p))\n                for p, delta in (old_counts - new_counts).items():\n                    counts[p] -= delta\n                    if counts[p] <= 0:\n                        counts.pop(p, None)\n                    else:\n                        heapq.heappush(heap, (-counts[p], p))\n                    occ = occurrences.get(p)\n                    if occ is not None:\n                        occ.discard(idx)\n                        if not occ:\n                            del occurrences[p]\n            occurrences.pop(pair, None)\n        return merges\n\n    def _build_vocab(self):\n        self.token_bytes = {i: bytes([i]) for i in range(256)}\n        for new_id, (a, b) in enumerate(self.merges, start=256):\n            self.token_bytes[new_id] = self.token_bytes[a] + self.token_bytes[b]\n        self.byte_to_id = {b: i for i, b in self.token_bytes.items()}\n        # Trie of the merged token bytes, with -1 as the terminal marker\n        # (byte values are 0..255, so -1 cannot collide). Greedy longest-match\n        # trie walk is equivalent to the classic merged-vocabulary regex but\n        # runs at tens of MB/s in pure Python.\n        self._merge_trie = {}\n        for token_id, token_bytes in self.token_bytes.items():\n            node = self._merge_trie\n            for byte in token_bytes:\n                node = node.setdefault(byte, {})\n            node[-1] = token_id\n\n    @property\n    def vocab_size(self):\n        # 256 base bytes + learned merges + reserved special tokens\n        # (eos/user/assistant/pad -- see SPECIAL_TOKENS). Derived, not\n        # stored, so it is automatically present for every tokenizer --\n        # including ones loaded from a tokenizer.json saved before these\n        # tokens existed.\n        return 256 + len(self.merges) + len(self.SPECIAL_TOKENS)\n\n    @property\n    def special_token_ids(self):\n        """{name: id} for every reserved special token, in a stable order\n        right after the last BPE merge id. Adding/removing an entry in\n        ``SPECIAL_TOKENS`` changes ``vocab_size``, which correctly forces\n        a cache/checkpoint rebuild via the existing tokenizer-identity\n        check in bulk_corpus.py -- these ids are never silently reused\n        for a different meaning."""\n        base = 256 + len(self.merges)\n        return {name: base + i for i, name in enumerate(self.SPECIAL_TOKENS)}\n\n    @property\n    def eos_id(self):\n        """Reserved id marking a genuine document/conversation boundary.\n\n        Not part of the byte-merge vocabulary (no ``token_bytes`` entry),\n        so it can never be produced by ``encode()`` on ordinary text and\n        is unambiguous in the token stream -- unlike a plain "\\\\n\\\\n",\n        which a model cannot distinguish from an ordinary paragraph break.\n        """\n        return self.special_token_ids["<|eos|>"]\n\n    @property\n    def user_id(self):\n        """Reserved id marking the start of a user turn in a dialogue\n        source with genuinely known speaker roles (e.g. "User:"/\n        "Assistant:" transcripts). Never inserted for sources where the\n        real role is ambiguous (e.g. "Speaker 0/1") -- see bulk_corpus.py."""\n        return self.special_token_ids["<|user|>"]\n\n    @property\n    def assistant_id(self):\n        """Reserved id marking the start of an assistant turn. See ``user_id``."""\n        return self.special_token_ids["<|assistant|>"]\n\n    @property\n    def pad_id(self):\n        """Reserved id for padding variable-length sequences into a batch.\n\n        Not used anywhere in the current bulk-pretraining pipeline (every\n        training window is a fixed ``block_size`` slice of one continuous\n        token stream, so there is nothing to pad) -- reserved in advance\n        for a future instruction-tuning stage where individual examples\n        have different lengths and need padding to batch together.\n        """\n        return self.special_token_ids["<|pad|>"]\n\n    @property\n    def think_id(self):\n        """Reserved id marking the start of a reasoning/chain-of-thought\n        trace within an assistant turn (from literal \'<think>\' markup in\n        some sources\' already-tagged data). Not a turn/role boundary --\n        appears inside an assistant turn, not between turns."""\n        return self.special_token_ids["<|think|>"]\n\n    @property\n    def think_end_id(self):\n        """Reserved id marking the end of a reasoning trace (from literal\n        \'</think>\' markup). Some sources\' records never close the tag\n        (the reasoning trace runs to the end of the assistant turn) --\n        that\'s fine, this id simply never appears in those records."""\n        return self.special_token_ids["<|/think|>"]\n\n    @property\n    def tokens(self):\n        ids_to_name = {v: k for k, v in self.special_token_ids.items()}\n        out = []\n        for i in range(self.vocab_size):\n            if i in ids_to_name:\n                out.append(ids_to_name[i])\n            else:\n                out.append(self.token_bytes[i].decode("latin-1"))\n        return out\n\n    # -- encode / decode ---------------------------------------------------\n\n    def encode(self, text: str):\n        ids = []\n        processed_chars = 0\n        # Reserved markup is converted only when it appears literally and\n        # exactly; ordinary text continues through the byte-BPE path.\n        special_re = re.compile(r"(<\\|eos\\|>|<\\|user\\|>|<\\|assistant\\|>|<\\|pad\\|>|<\\|think\\|>|<\\|/think\\|>)")\n        special_map = {\n            "<|eos|>": self.eos_id,\n            "<|user|>": self.user_id,\n            "<|assistant|>": self.assistant_id,\n            "<|pad|>": self.pad_id,\n            "<|think|>": self.think_id,\n            "<|/think|>": self.think_end_id,\n        }\n        segments = special_re.split(text)\n        text_segments = []\n        for segment in segments:\n            if segment in special_map:\n                ids.append(special_map[segment])\n            elif segment:\n                text_segments.append(segment)\n        if len(segments) > 1:\n            text = "".join(text_segments)\n\n        processed_chars = 0\n        next_progress = self.progress_interval_chars\n        total_chars = len(text)\n        for unit in self._pattern.findall(text):\n            b = unit.encode("utf-8")\n            i = 0\n            n = len(b)\n            while i < n:\n                node = self._merge_trie\n                j = i\n                last_id = self.byte_to_id[b[i:i + 1]]\n                last_j = i + 1\n                while j < n:\n                    nxt = node.get(b[j])\n                    if nxt is None:\n                        break\n                    node = nxt\n                    j += 1\n                    term = node.get(-1)\n                    if term is not None:\n                        last_id = term\n                        last_j = j\n                ids.append(last_id)\n                i = last_j\n            processed_chars += len(unit)\n            if processed_chars >= next_progress:\n                percent = 100.0 * processed_chars / total_chars if total_chars else 100.0\n                print(f"Encoded {processed_chars:,} / {total_chars:,} chars ({percent:.1f}%)")\n                next_progress += self.progress_interval_chars\n        return ids\n\n    def decode(self, ids):\n        # Special ids (eos/user/assistant/pad) have no byte representation\n        # (they never appear inside real text); render them as empty\n        # instead of doing a dict lookup that would KeyError, so decoding\n        # a sequence that contains them never crashes.\n        special_ids = set(self.special_token_ids.values())\n        parts = []\n        for i in ids:\n            i = int(i)\n            if i in special_ids:\n                parts.append(b"")\n            else:\n                parts.append(self.token_bytes[i])\n        return b"".join(parts).decode("utf-8", errors="replace")\n\n    def decode_with_boundaries(self, ids, boundary_marker: str = None):\n        """Like ``decode``, but renders each special token as a visible\n        marker instead of silently dropping it -- useful for inspecting\n        whether a cache or a generation actually contains the boundaries/\n        role tags you expect. If ``boundary_marker`` is given, it is used\n        for every special token (legacy single-marker behavior); otherwise\n        each special token gets its own readable tag, e.g. "<|user|>"."""\n        ids_to_name = {v: k for k, v in self.special_token_ids.items()}\n        parts = []\n        for i in ids:\n            i = int(i)\n            if i in ids_to_name:\n                tag = boundary_marker if boundary_marker is not None else f"\\n\\n{ids_to_name[i]}\\n\\n"\n                parts.append(tag)\n            else:\n                parts.append(self.token_bytes[i].decode("latin-1"))\n        # token_bytes pieces are latin-1 (1 byte <-> 1 char); re-encode then\n        # decode as utf-8 to correctly join multi-byte sequences, matching\n        # decode()\'s behavior. Marker text is plain ASCII so this is lossless.\n        raw = "".join(parts).encode("latin-1", errors="ignore")\n        return raw.decode("utf-8", errors="replace")\n\n    def save(self, path: str):\n        with open(path, "w") as f:\n            json.dump({"merges": [list(m) for m in self.merges]}, f)\n\n    @classmethod\n    def load(cls, path: str):\n        with open(path) as f:\n            payload = json.load(f)\n        return cls(merges=payload["merges"])', encoding="utf-8")
print("Wrote:", path)
print("Bytes:", path.stat().st_size)


In [ ]:
# FILE: gamax1/instruction_data.py
from pathlib import Path

path = PROJECT_ROOT / 'gamax1/instruction_data.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text('"""\ngamax1/instruction_data.py\n===========================\nLoads instruction/Q&A-style training data (JSON or JSONL files) for the\nfine-tuning stage, as opposed to bulk_corpus.py\'s plain-.txt continuous-\nstream pretraining pipeline.\n\nKey differences from bulk pretraining that this module exists to handle:\n  1. Each record is a DISCRETE example (a question + its answer), not a\n     slice of one long continuous document -- so examples are padded\n     into batches rather than windowed out of a token stream.\n  2. Loss must be computed ONLY on the answer/assistant tokens, never on\n     the question/prompt tokens -- otherwise the model spends capacity\n     learning to predict the *question*, which is not the training goal\n     and dilutes the (already scarce) signal we\'re trying to concentrate\n     into it. This is the reason for `loss_mask` throughout this file.\n\nFORMAT DETECTION\n-----------------\nYour data\'s actual key names haven\'t been confirmed yet, so this loader\ntries several common conventions, in this order, per record:\n  1. {"messages": [{"role": "user", "content": "..."},\n                    {"role": "assistant", "content": "..."}, ...]}\n     -- OpenAI/ChatML-style multi-turn. Every assistant turn becomes one\n     training example, with all prior turns in that conversation as its\n     prompt context (so a 4-turn conversation yields 2 training examples:\n     one predicting the first assistant reply, one predicting the second\n     with both prior turns as context).\n  2. {"instruction": "...", "input": "...", "output": "..."}\n     -- Alpaca-style. "input" is optional; if present it\'s appended to\n     "instruction" (matching the standard Alpaca prompt template).\n  3. {"question": "...", "answer": "..."}  or\n     {"prompt": "...", "response": "..."}  or\n     {"prompt": "...", "completion": "..."}\n     -- Plain Q&A pairs, whichever key names your files use.\n\nIf a record matches NONE of these, it\'s skipped and counted, and the\nfinal summary tells you how many were skipped along with the key names\nseen -- so a real format mismatch is loud and diagnosable, not a silent\nzero-example dataset.\n\nIf your actual file uses different key names than all of the above,\ntell me the exact keys and I\'ll add a fourth pattern rather than you\nhaving to reshape the data.\n"""\n\nimport json\nimport os\nimport random\n\nimport torch\n\n\ndef _iter_records(path: str):\n    """Yield dict records from a single .json or .jsonl file.\n\n    .json: either a single object, or a list of objects.\n    .jsonl: one JSON object per non-empty line.\n    """\n    ext = os.path.splitext(path)[1].lower()\n    with open(path, encoding="utf-8") as f:\n        if ext == ".jsonl":\n            for line_no, line in enumerate(f, start=1):\n                line = line.strip()\n                if not line:\n                    continue\n                try:\n                    yield json.loads(line)\n                except json.JSONDecodeError as e:\n                    print(f"[WARNING] {path}:{line_no}: skipping malformed JSON line ({e})")\n        else:\n            data = json.load(f)\n            if isinstance(data, list):\n                for item in data:\n                    yield item\n            elif isinstance(data, dict):\n                # Some exports wrap the list under a top-level key, e.g.\n                # {"data": [...]} or {"examples": [...]}. Try the common\n                # ones before giving up and treating the dict as one record.\n                for key in ("data", "examples", "records", "conversations"):\n                    if key in data and isinstance(data[key], list):\n                        for item in data[key]:\n                            yield item\n                        return\n                yield data\n            else:\n                raise ValueError(f"{path}: top-level JSON must be an object or a list")\n\n\ndef iter_files(data_path: str):\n    """Yield every .json/.jsonl file under data_path (a file or a directory,\n    recursive)."""\n    if os.path.isfile(data_path):\n        yield data_path\n        return\n    for root, _dirs, files in os.walk(data_path):\n        for name in sorted(files):\n            if name.lower().endswith((".json", ".jsonl")):\n                yield os.path.join(root, name)\n\n\ndef _extract_pairs(record: dict):\n    """Return a list of (prompt_text, answer_text) pairs from one record,\n    per the format-detection rules documented in the module docstring.\n    Empty list if the record matches no known format.\n    """\n    if not isinstance(record, dict):\n        return []\n\n    # -- 1. ChatML-style multi-turn messages -------------------------------\n    messages = record.get("messages")\n    if isinstance(messages, list) and messages:\n        pairs = []\n        context = []\n        for msg in messages:\n            role = str(msg.get("role", "")).lower()\n            content = str(msg.get("content", ""))\n            if role == "user":\n                context.append(("user", content))\n            elif role == "assistant":\n                if context:  # only train on replies that have a preceding prompt\n                    prompt_text = "\\n".join(c for _, c in context)\n                    pairs.append((prompt_text, content))\n                context.append(("assistant", content))\n            # "system" (or anything else) is folded into context text but\n            # not itself a role tag we emit -- GamaX1\'s tokenizer only\n            # reserves <|user|>/<|assistant|>, not a system tag.\n            elif role == "system":\n                context.append(("system", content))\n        return pairs\n\n    # -- 2. Alpaca-style instruction/input/output ---------------------------\n    if "instruction" in record and "output" in record:\n        instruction = str(record["instruction"])\n        extra_input = str(record.get("input", "") or "")\n        prompt_text = f"{instruction}\\n\\n{extra_input}" if extra_input else instruction\n        return [(prompt_text, str(record["output"]))]\n\n    # -- 3. Plain Q&A pairs, several common key-name conventions ------------\n    key_pairs = (\n        ("question", "answer"),\n        ("prompt", "response"),\n        ("prompt", "completion"),\n        ("input", "output"),\n    )\n    for prompt_key, answer_key in key_pairs:\n        if prompt_key in record and answer_key in record:\n            return [(str(record[prompt_key]), str(record[answer_key]))]\n\n    return []\n\n\ndef load_pairs(data_path: str):\n    """Walk data_path (file or directory) and return (pairs, stats).\n\n    pairs: list of (prompt_text, answer_text) strings, ready to encode.\n    stats: dict with counts, useful to sanity-check a format mismatch\n    before spending time encoding/training on zero real examples.\n    """\n    pairs = []\n    seen_files = 0\n    total_records = 0\n    skipped_records = 0\n    unmatched_keys_seen = set()\n\n    for path in iter_files(data_path):\n        seen_files += 1\n        for record in _iter_records(path):\n            total_records += 1\n            record_pairs = _extract_pairs(record)\n            if not record_pairs:\n                skipped_records += 1\n                if isinstance(record, dict):\n                    unmatched_keys_seen.add(tuple(sorted(record.keys())))\n                continue\n            pairs.extend(record_pairs)\n\n    stats = {\n        "files": seen_files,\n        "records": total_records,\n        "pairs": len(pairs),\n        "skipped_records": skipped_records,\n        "unmatched_key_sets": list(unmatched_keys_seen)[:5],  # a few examples, not all\n    }\n    return pairs, stats\n\n\ndef encode_example(tokenizer, prompt_text: str, answer_text: str, max_len: int):\n    """Build one training example: token ids plus a same-length loss mask.\n\n    ids  = [<|user|>] + encode(prompt) + [<|assistant|>] + encode(answer) + [<|eos|>]\n    mask = [0]*(len up to and including <|assistant|>) + [1]*(answer + eos)\n\n    mask==1 marks exactly the tokens the loss should be computed on --\n    the assistant\'s own answer and the closing eos, never the question\n    or the role tags themselves. Truncation, if the example is too long\n    for max_len, always removes from the START of the PROMPT first (the\n    least important tokens to keep), never from the answer -- an\n    example is only answer-truncated (with a printed warning) if the\n    answer plus both role tags and eos alone still exceeds max_len.\n    """\n    user_id = tokenizer.user_id\n    assistant_id = tokenizer.assistant_id\n    eos_id = tokenizer.eos_id\n\n    prompt_ids = tokenizer.encode(prompt_text)\n    answer_ids = tokenizer.encode(answer_text)\n\n    fixed_overhead = 3  # <|user|>, <|assistant|>, <|eos|>\n    answer_budget = max(max_len - fixed_overhead, 0)\n    if len(answer_ids) > answer_budget:\n        print(f"[WARNING] answer alone ({len(answer_ids)} tokens) exceeds --max_len={max_len}; "\n              "truncated. Consider raising --max_len for this dataset.")\n        answer_ids = answer_ids[:answer_budget]\n\n    prompt_budget = max_len - fixed_overhead - len(answer_ids)\n    if len(prompt_ids) > prompt_budget:\n        prompt_ids = prompt_ids[-max(prompt_budget, 0):]  # keep the END of the prompt (nearest the question)\n\n    ids = [user_id] + prompt_ids + [assistant_id] + answer_ids + [eos_id]\n    mask = [0] * (2 + len(prompt_ids)) + [1] * (len(answer_ids) + 1)\n    return ids, mask\n\n\nclass InstructionDataset(torch.utils.data.Dataset):\n    """Wraps a list of (prompt_text, answer_text) pairs, encoding lazily\n    (on __getitem__) so a huge dataset doesn\'t need every example\n    tokenized up front."""\n\n    def __init__(self, pairs, tokenizer, max_len: int):\n        self.pairs = pairs\n        self.tokenizer = tokenizer\n        self.max_len = max_len\n\n    def __len__(self):\n        return len(self.pairs)\n\n    def __getitem__(self, idx):\n        prompt_text, answer_text = self.pairs[idx]\n        ids, mask = encode_example(self.tokenizer, prompt_text, answer_text, self.max_len)\n        return ids, mask\n\n\ndef make_collate_fn(pad_id: int):\n    """Right-pads a batch of variable-length (ids, mask) examples to the\n    batch\'s own max length (not a fixed max_len), which keeps compute\n    proportional to what\'s actually in the batch. Returns xb, yb, loss_mask\n    -- all (batch, seq-1) since xb/yb are the standard next-token shift.\n    Padded positions get loss_mask==0 automatically (pad_id tokens are\n    never real answer content), so they contribute nothing to the loss\n    without needing a separate attention-padding mask: GamaX1\'s attention\n    is plain causal self-attention with no padding-mask input, so a\n    padded key position CAN be attended to by real tokens before it in\n    the same row -- harmless here only because every pad token is placed\n    strictly after that row\'s real content (right-padding) and pad\n    positions never contribute to the loss themselves, so their influence\n    on earlier positions\' predictions is the only leakage. Left-padding\n    would leak in a way that matters and must not be used with this\n    collate function.\n    """\n\n    def collate(batch):\n        max_len_in_batch = max(len(ids) for ids, _ in batch)\n        batch_x, batch_y, batch_mask = [], [], []\n        for ids, mask in batch:\n            pad_len = max_len_in_batch - len(ids)\n            padded_ids = ids + [pad_id] * pad_len\n            padded_mask = mask + [0] * pad_len\n            batch_x.append(padded_ids[:-1])\n            batch_y.append(padded_ids[1:])\n            batch_mask.append(padded_mask[1:])  # mask aligns with the TARGET (yb) position\n        return (\n            torch.tensor(batch_x, dtype=torch.long),\n            torch.tensor(batch_y, dtype=torch.long),\n            torch.tensor(batch_mask, dtype=torch.bool),\n        )\n\n    return collate\n\n\ndef split_train_val(pairs, val_fraction: float, seed: int = 0):\n    """Deterministic shuffle + split, so repeated runs on the same file\n    see the same held-out set (useful for comparing fine-tune runs)."""\n    pairs = list(pairs)\n    random.Random(seed).shuffle(pairs)\n    n_val = max(1, int(len(pairs) * val_fraction)) if pairs else 0\n    return pairs[n_val:], pairs[:n_val]\n', encoding="utf-8")
print("Wrote:", path)
print("Bytes:", path.stat().st_size)


In [ ]:
# FILE: gamax1/bulk_corpus.py
from pathlib import Path

path = PROJECT_ROOT / 'gamax1/bulk_corpus.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text('"""Incremental token-cache support for large directories of text books.\n\nThe normal trainer is intentionally simple and reads one text file into RAM.\nThis module is the large-corpus path: it samples text to train a tokenizer,\nthen encodes each book one at a time into an int32 binary file.  The finished\nfile is memory mapped, so training batches do not require an 11 GB Python\nstring or an equally large list of token IDs.\n\nThis version supports multiple source categories (e.g. books, wiki, qna) held\nin separate sub-directories under one root data directory.  Each source\'s\nfiles are tracked separately in the metadata so downstream tooling can\ncompute per-category token counts and, later, per-category evaluation.\n\nEncoding is tracked per file (path + size), not just as one\nall-or-nothing corpus manifest. This means adding a brand-new source folder\n(or a handful of new files to an existing one) only encodes the new/changed\nfiles and appends them to the token stream -- files that were already\nencoded, under the same tokenizer, are left untouched. This matters most on\nfree-tier Colab, where re-encoding tens of thousands of already-done files\njust because one new folder was added would burn most of the session\'s\ncompute budget on repeated work. A tokenizer change (different vocab size or\nmerges) still forces a full rebuild, since every existing token ID would be\nwrong under new merges -- nothing can be salvaged in that case.\n\nEncoding itself is also resumable within one incremental batch. Every\n``PROGRESS_INTERVAL`` files, the current progress (which files in this\nbatch are done, and the full per-file token index so far) is written to a\nsmall progress JSON file. If the process is interrupted (e.g. a Colab\ndisconnect) and re-launched against the same data/cache directories with the\nsame new/changed file set, encoding picks up right after the last completed\nfile instead of starting the batch over.\n"""\n\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport mmap\nimport os\nimport re\nimport time\nfrom array import array\nfrom pathlib import Path\nfrom typing import Optional\n\nimport torch\n\nfrom .tokenizer import BPETokenizer\n\n\n# Explicit dataset paths used by the default corpus build.\nDATASETS = {\n    "books_cleaned_v1": "/content/drive/MyDrive/Aetherion_GamaX1/data/books_cleaned_v1",\n    "Math_Reasoning_train": "/content/drive/MyDrive/Aetherion_GamaX1/data/Math_Reasoning/train/books",\n    "Conversations-200k_clean": "/content/drive/MyDrive/Aetherion_GamaX1/data/Conversations-200k_clean",\n    "Q&A": "/content/drive/MyDrive/Aetherion_GamaX1/data/QnA",\n}\n\nDEFAULT_SOURCE_DIRS = DATASETS\n\n# -- Per-source content format ----------------------------------------------\n#\n# Different source folders hold genuinely different kinds of text, and\n# treating them identically at encoding time is itself a source of the\n# "model drifts into an unrelated pattern mid-generation" problem: a plain\n# document-boundary token is not enough when a single *file* also glues\n# together several unrelated mini-conversations (observed in\n# Discord-Dialogues, "---"-separated), or when the file\'s own speaker\n# labels don\'t map to a real user/assistant role (observed in\n# Reddit-Constructive\'s "Speaker 0:"/"Speaker 1:" format, where role\n# identity isn\'t fixed across threads).\n#\n# Three formats are supported, chosen deliberately per source rather than\n# guessed from content:\n#   "prose"         -- continuous text (books/wiki). Gutenberg license\n#                      boilerplate is stripped (it is legal filler, not\n#                      content, and was otherwise the single most-repeated\n#                      pattern across thousands of book files). One\n#                      eos_id-bounded unit per file.\n#   "user_assistant" -- text with genuine, known "User:"/"Assistant:"\n#                      labels. Each turn is wrapped with the tokenizer\'s\n#                      dedicated <|user|>/<|assistant|> ids instead of the\n#                      literal text "User:"/"Assistant:" (which would just\n#                      be ordinary, spoofable BPE tokens). A file that\n#                      glues multiple unrelated exchanges together with a\n#                      standalone "---" line is first split on that\n#                      separator, and each resulting exchange gets its own\n#                      eos_id boundary -- fixing the file-level-only\n#                      boundary\'s blind spot for glued-together turns.\n#   "generic_turns" -- dialogue with ambiguous/unfixed speaker identity\n#                      (e.g. "Speaker 0:"/"Speaker 1:", where the same\n#                      label doesn\'t reliably mean the same role across\n#                      threads). Deliberately does NOT fabricate\n#                      <|user|>/<|assistant|> role tags here -- an\n#                      incorrect role label would be a worse training\n#                      signal than no role label. Still split on a\n#                      standalone "---" line, with eos_id between each\n#                      resulting turn/exchange, and text encoded plainly.\n#\n# A source not listed here defaults to "prose" (the safe, unopinionated\n# choice) with a one-time warning -- see _resolve_format().\nSOURCE_FORMAT_PROSE = "prose"\nSOURCE_FORMAT_USER_ASSISTANT = "user_assistant"\nSOURCE_FORMAT_GENERIC_TURNS = "generic_turns"\nCORPUS_FORMAT_VERSION = "v4-schema-aware-json-fingerprint"\n\nDEFAULT_SOURCE_FORMATS = {\n    "books_cleaned_v1": SOURCE_FORMAT_PROSE,\n    "Math_Reasoning_train": SOURCE_FORMAT_USER_ASSISTANT,\n    "Conversations-200k_clean": SOURCE_FORMAT_USER_ASSISTANT,\n    "Q&A": SOURCE_FORMAT_USER_ASSISTANT,\n}\n\n_unknown_source_format_warned: set[str] = set()\n\n\ndef _resolve_format(source_name: str, source_formats: dict) -> str:\n    fmt = source_formats.get(source_name)\n    if fmt is not None:\n        return fmt\n    if source_name not in _unknown_source_format_warned:\n        print(\n            f"[WARNING] No content format configured for source \'{source_name}\' -- "\n            f"defaulting to \'{SOURCE_FORMAT_PROSE}\' (plain text, one eos_id-bounded "\n            "unit per file, no speaker tags). Add it to DEFAULT_SOURCE_FORMATS if "\n            "it actually contains dialogue."\n        )\n        _unknown_source_format_warned.add(source_name)\n    return SOURCE_FORMAT_PROSE\n\n\n# Matches a "---" (or longer) line on its own, with only whitespace around\n# it -- the separator observed gluing unrelated Discord/Reddit exchanges\n# together. Deliberately requires the WHOLE line to be dashes so it does\n# not fire on a literal "---" appearing mid-sentence in real prose.\n_STANDALONE_SEP_RE = re.compile(r"(?m)^[ \\t]*-{3,}[ \\t]*$")\n\n# Recognizes a "User:"/"Assistant:" turn label at the start of a line (or\n# start of text) and captures everything up to the next such label. Case\n# matches the observed corpus convention exactly; extend the alternation\n# here if a source uses different capitalization.\n_TURN_RE = re.compile(\n    r"(?m)^[ \\t]*(User|Assistant)\\s*:\\s*(.*?)(?=(?:\\n[ \\t]*(?:User|Assistant)\\s*:)|\\Z)",\n    re.DOTALL,\n)\n\n# Standard Project Gutenberg boilerplate markers (same pattern used in\n# prepare_large_corpus.py). Keeping content strictly between these two\n# markers removes the repeated "The Project Gutenberg eBook of ... This\n# eBook is for the use of anyone..." license preamble/footer that would\n# otherwise be the single most over-represented pattern across a\n# multi-thousand-book corpus.\n_GUTENBERG_START_RE = re.compile(\n    r"\\*\\*\\*\\s*START OF (THE|THIS) PROJECT GUTENBERG EBOOK.*?\\*\\*\\*", re.IGNORECASE | re.DOTALL\n)\n_GUTENBERG_END_RE = re.compile(\n    r"\\*\\*\\*\\s*END OF (THE|THIS) PROJECT GUTENBERG EBOOK.*", re.IGNORECASE | re.DOTALL\n)\n\n\ndef _strip_gutenberg_boilerplate(text: str) -> str:\n    m = _GUTENBERG_START_RE.search(text)\n    if m:\n        text = text[m.end():]\n    m = _GUTENBERG_END_RE.search(text)\n    if m:\n        text = text[:m.start()]\n    return text.strip()\n\n\ndef _split_on_separator(text: str) -> list[str]:\n    """Split on a standalone "---" line into non-empty, stripped chunks."""\n    parts = [p.strip() for p in _STANDALONE_SEP_RE.split(text)]\n    return [p for p in parts if p]\n\n\n\ndef _json_records(raw_text: str, suffix: str) -> list[dict]:\n    """Parse JSON/JSONL objects conservatively and tolerate common wrappers."""\n    records: list[dict] = []\n    if suffix.lower() == ".jsonl":\n        for line in raw_text.splitlines():\n            line = line.strip()\n            if not line:\n                continue\n            try:\n                obj = json.loads(line)\n            except json.JSONDecodeError:\n                continue\n            if isinstance(obj, dict):\n                records.append(obj)\n            elif isinstance(obj, list):\n                records.extend(x for x in obj if isinstance(x, dict))\n        return records\n\n    try:\n        obj = json.loads(raw_text)\n    except json.JSONDecodeError:\n        return []\n    if isinstance(obj, dict):\n        # Common dataset wrappers: {"data": [...]} / {"examples": [...]} etc.\n        for key in ("data", "records", "examples", "items", "rows"):\n            value = obj.get(key)\n            if isinstance(value, list) and all(isinstance(x, dict) for x in value):\n                return value\n        return [obj]\n    if isinstance(obj, list):\n        return [x for x in obj if isinstance(x, dict)]\n    return []\n\n\ndef _content_to_text(content) -> str:\n    """Normalize string or common multimodal-content representations."""\n    if isinstance(content, str):\n        return content.strip()\n    if isinstance(content, list):\n        parts = []\n        for item in content:\n            if isinstance(item, str):\n                parts.append(item)\n            elif isinstance(item, dict):\n                text = item.get("text")\n                if isinstance(text, str):\n                    parts.append(text)\n        return "\\n".join(parts).strip()\n    return ""\n\n\ndef _message_pairs_from_record(record: dict) -> list[tuple[str, str]]:\n    """Extract user/assistant pairs from common conversation/Q&A schemas."""\n    messages = record.get("messages")\n    if isinstance(messages, list):\n        pairs: list[tuple[str, str]] = []\n        pending_user: str | None = None\n        for message in messages:\n            if not isinstance(message, dict):\n                continue\n            role = str(message.get("role", "")).strip().lower()\n            content = _content_to_text(message.get("content", ""))\n            if not content:\n                continue\n            if role in {"user", "human", "question"}:\n                pending_user = content\n            elif role in {"assistant", "bot", "answer", "gpt", "model"} and pending_user is not None:\n                pairs.append((pending_user, content))\n                pending_user = None\n        if pairs:\n            return pairs\n\n    # Common single-turn Q&A schemas, including Johnson-style datasets.\n    user_keys = ("question", "prompt", "query", "instruction", "input", "user", "human")\n    assistant_keys = ("answer", "response", "output", "completion", "assistant", "bot", "target")\n    user_text = next((_content_to_text(record.get(k)) for k in user_keys if _content_to_text(record.get(k))), "")\n    assistant_text = next((_content_to_text(record.get(k)) for k in assistant_keys if _content_to_text(record.get(k))), "")\n    if user_text and assistant_text:\n        return [(user_text, assistant_text)]\n\n    # Some datasets store an explicit two-turn list under conversation/dialog.\n    for key in ("conversation", "dialog", "dialogue", "turns"):\n        turns = record.get(key)\n        if not isinstance(turns, list):\n            continue\n        pending_user = None\n        pairs = []\n        for turn in turns:\n            if not isinstance(turn, dict):\n                continue\n            role = str(turn.get("role", turn.get("speaker", ""))).strip().lower()\n            content = _content_to_text(turn.get("content", turn.get("text", turn.get("value", ""))))\n            if not content:\n                continue\n            if role in {"user", "human", "question"}:\n                pending_user = content\n            elif role in {"assistant", "bot", "answer", "model", "gpt"} and pending_user is not None:\n                pairs.append((pending_user, content))\n                pending_user = None\n        if pairs:\n            return pairs\n    return []\n\n\ndef _extract_json_training_text(raw_text: str, suffix: str) -> list[str]:\n    """Extract actual textual training content for BPE sampling."""\n    chunks: list[str] = []\n    for record in _json_records(raw_text, suffix):\n        pairs = _message_pairs_from_record(record)\n        if pairs:\n            for user_text, assistant_text in pairs:\n                chunks.extend((user_text, assistant_text))\n            continue\n        # For prose-like JSON records, use explicit text/content fields only.\n        for key in ("text", "content", "document", "body"):\n            value = _content_to_text(record.get(key))\n            if value:\n                chunks.append(value)\n                break\n    return chunks\n\n\ndef _encode_json_messages(tokenizer: BPETokenizer, raw_text: str, suffix: str) -> list[int]:\n    """Encode JSON/JSONL Q&A records as reserved user/assistant role tokens."""\n    ids: list[int] = []\n    for record in _json_records(raw_text, suffix):\n        for user_text, assistant_text in _message_pairs_from_record(record):\n            if ids:\n                ids.append(tokenizer.eos_id)\n            ids.append(tokenizer.user_id)\n            ids.extend(tokenizer.encode(user_text))\n            ids.append(tokenizer.assistant_id)\n            ids.extend(tokenizer.encode(assistant_text))\n    return ids\n\n\ndef _encode_user_assistant_block(tokenizer: BPETokenizer, block: str) -> list[int]:\n    """Encode one exchange, replacing literal "User:"/"Assistant:" labels\n    with the tokenizer\'s dedicated role ids. Falls back to plain prose\n    encoding if no recognizable turn label is found at all, since forcing\n    a role tag onto untagged text would be a fabricated signal."""\n    matches = list(_TURN_RE.finditer(block))\n    if not matches:\n        return tokenizer.encode(block)\n\n    ids = []\n    preamble = block[:matches[0].start()].strip()\n    if preamble:\n        ids.extend(tokenizer.encode(preamble))\n    for m in matches:\n        role, content = m.group(1), m.group(2).strip()\n        ids.append(tokenizer.user_id if role == "User" else tokenizer.assistant_id)\n        if content:\n            ids.extend(tokenizer.encode(content))\n    return ids\n\n\ndef _encode_source_file(\n    tokenizer: BPETokenizer, source_name: str, raw_text: str, source_formats: dict,\n    is_first_emission: bool, suffix: str = "",\n) -> tuple[list[int], bool]:\n    """Encode one file\'s text according to its source\'s content format.\n\n    Returns ``(token_ids, is_first_emission)`` where the returned\n    ``is_first_emission`` has been updated for the caller\'s next file --\n    threading it through this way keeps the "does this need a leading\n    eos_id" decision correct across BOTH file boundaries and any\n    within-file "---"-separated boundaries this function introduces,\n    without the caller needing to know how many sub-blocks a file split\n    into.\n    """\n    fmt = _resolve_format(source_name, source_formats)\n    ids: list[int] = []\n\n    def emit(block_ids: list[int]):\n        nonlocal is_first_emission\n        if not is_first_emission:\n            ids.append(tokenizer.eos_id)\n        is_first_emission = False\n        ids.extend(block_ids)\n\n    if fmt == SOURCE_FORMAT_PROSE:\n        emit(tokenizer.encode(_strip_gutenberg_boilerplate(raw_text)))\n    elif fmt == SOURCE_FORMAT_USER_ASSISTANT:\n        if suffix.lower() in {".json", ".jsonl"}:\n            json_ids = _encode_json_messages(tokenizer, raw_text, suffix)\n            if json_ids:\n                emit(json_ids)\n        else:\n            for block in (_split_on_separator(raw_text) or [raw_text]):\n                emit(_encode_user_assistant_block(tokenizer, block))\n    elif fmt == SOURCE_FORMAT_GENERIC_TURNS:\n        for block in (_split_on_separator(raw_text) or [raw_text]):\n            emit(tokenizer.encode(block))\n    else:  # pragma: no cover -- _resolve_format never returns anything else\n        emit(tokenizer.encode(raw_text))\n\n    return ids, is_first_emission\n\n\n# Write a progress checkpoint every this many files during encoding.\n# Corpus encoding checkpoint frequency.\n#\n# Why 500? Encoding thousands of files can take a long time on a Drive-mounted\n# Colab filesystem. Every 500 completed files we flush the token stream and\n# write `encode_progress.json`. If Colab disconnects after that point, the next\n# run can resume instead of starting the whole corpus again.\n#\n# This is a FILE checkpoint, not a neural-network training checkpoint.\nENCODE_CHECKPOINT_EVERY = 500\n\n# Backwards-compatible name used by older notebook/debugging code. Keeping the\n# alias avoids breaking a cell that still prints or inspects PROGRESS_INTERVAL.\nPROGRESS_INTERVAL = ENCODE_CHECKPOINT_EVERY\n\n# Every this many files, force an OS-level fsync (not just a Python-level\n# flush) and pause briefly while checking that the on-disk file size has\n# stopped changing -- a practical proxy for "Google Drive\'s background sync\n# has likely caught up". Meant for split sessions (e.g. a daily 4-hour GPU\n# quota against a 5-hour encode): stopping the runtime right after one of\n# these hard-sync points is much safer than stopping between them.\nHARD_SYNC_INTERVAL = 10_000\nHARD_SYNC_STABLE_CHECKS = 3   # consecutive stable size readings required\nHARD_SYNC_CHECK_DELAY_SEC = 10  # seconds between size checks\n\n\ndef _format_duration(seconds: float) -> str:\n    """Return compact human-readable elapsed time for progress logs."""\n    seconds = max(0, int(round(seconds)))\n    hours, remainder = divmod(seconds, 3600)\n    minutes, seconds = divmod(remainder, 60)\n    if hours:\n        return f"{hours}h {minutes:02d}m {seconds:02d}s"\n    if minutes:\n        return f"{minutes}m {seconds:02d}s"\n    return f"{seconds}s"\n\n\ndef _hard_sync_checkpoint(output, token_path: Path) -> None:\n    """Force an OS-level fsync and wait for the on-disk file size to settle.\n\n    ``output.flush()`` only pushes Python\'s buffer into the OS -- it does not\n    guarantee Google Drive\'s FUSE mount has pushed those bytes to the actual\n    cloud copy. ``os.fsync`` forces the OS to write its buffers to the mount,\n    and then re-checking the file size a few times a few seconds apart gives\n    a practical (not perfect) signal that Drive\'s background sync has caught\n    up: if the size is still climbing, Drive is still working through a\n    backlog and it is not a good time to disconnect.\n    """\n    output.flush()\n    os.fsync(output.fileno())\n\n    stable_count = 0\n    last_size = -1\n    for _ in range(HARD_SYNC_STABLE_CHECKS + 5):  # bounded, never hangs forever\n        current_size = token_path.stat().st_size\n        if current_size == last_size:\n            stable_count += 1\n            if stable_count >= HARD_SYNC_STABLE_CHECKS:\n                break\n        else:\n            stable_count = 0\n        last_size = current_size\n        time.sleep(HARD_SYNC_CHECK_DELAY_SEC)\n\n    print(\n        f"  [hard sync] fsync\'d and size stable at {last_size:,} bytes -- "\n        f"safe to stop the runtime now if you need to."\n    )\n\n\nclass BulkTokenStore:\n    """Read-only memory-mapped token storage with a torch tensor view."""\n\n    def __init__(self, token_path: Path, token_count: int):\n        self.token_path = Path(token_path)\n        self._file = self.token_path.open("rb")\n        # ACCESS_COPY gives torch a writable view without copying the entire\n        # token file into RAM; writes remain private and never touch the cache.\n        self._mapping = mmap.mmap(self._file.fileno(), 0, access=mmap.ACCESS_COPY)\n        self.tensor = torch.frombuffer(self._mapping, dtype=torch.int32, count=token_count)\n\n    def close(self):\n        # Release the tensor view before closing its backing mmap.\n        self.tensor = None\n        self._mapping.close()\n        self._file.close()\n\n    def __enter__(self):\n        return self\n\n    def __exit__(self, exc_type, exc, tb):\n        self.close()\n\n\ndef _collect_source_paths(data_dir: str | Path, source_dirs=DEFAULT_SOURCE_DIRS) -> dict[str, list[Path]]:\n    """Collect text files from explicit dataset paths or legacy subfolders."""\n    if isinstance(source_dirs, dict):\n        candidates = [(str(name), Path(path)) for name, path in source_dirs.items()]\n    else:\n        root = Path(data_dir)\n        if not root.is_dir():\n            raise FileNotFoundError(f"data directory does not exist: {root}")\n        candidates = [(str(name), root / name) for name in source_dirs]\n\n    sources: dict[str, list[Path]] = {}\n    for name, source_path in candidates:\n        if not source_path.is_dir():\n            continue\n        allowed_suffixes = {".txt", ".json", ".jsonl"}\n        paths = sorted(\n            (\n                p for p in source_path.rglob("*")\n                if p.is_file() and p.suffix.lower() in allowed_suffixes\n            ),\n            key=lambda p: str(p),\n        )\n        if paths:\n            sources[name] = paths\n\n    if not sources:\n        root = Path(data_dir)\n        if root.is_dir():\n            allowed_suffixes = {".txt", ".json", ".jsonl"}\n            flat_paths = sorted(\n                (\n                    p for p in root.rglob("*")\n                    if p.is_file() and p.suffix.lower() in allowed_suffixes\n                ),\n                key=lambda p: str(p),\n            )\n            if flat_paths:\n                sources["books"] = flat_paths\n\n    if not sources:\n        raise ValueError(\n            f"no supported source files (.txt/.json/.jsonl) found: {candidates}"\n        )\n    return sources\n\n\ndef book_paths(data_dir: str | Path) -> list[Path]:\n    """Return deterministic, recursive .txt input paths across all sources.\n\n    Kept for backward compatibility with callers that just want a flat list\n    of every file regardless of source category. Order is: source category\n    name order, then path order within each source.\n    """\n    sources = _collect_source_paths(data_dir)\n    paths: list[Path] = []\n    for name in sorted(sources):\n        paths.extend(sources[name])\n    return paths\n\n\ndef sample_book_text(paths: list[Path], sample_chars: int) -> str:\n    """Read actual textual content, not raw JSON syntax, for BPE training."""\n    if sample_chars <= 0:\n        raise ValueError("sample_chars must be positive")\n    pieces: list[str] = []\n    remaining = sample_chars\n    for path in paths:\n        if remaining <= 0:\n            break\n        with path.open("r", encoding="utf-8", errors="ignore") as handle:\n            raw = handle.read(remaining * 2 if path.suffix.lower() in {".json", ".jsonl"} else remaining)\n        if path.suffix.lower() in {".json", ".jsonl"}:\n            extracted = _extract_json_training_text(raw, path.suffix)\n            text = "\\n\\n".join(extracted)\n        else:\n            text = raw\n        if text:\n            text = text[:remaining]\n            pieces.append(text)\n            remaining -= len(text)\n    sample = "\\n\\n".join(pieces)\n    if not sample.strip():\n        raise ValueError("input files contain no readable training text")\n    return sample\n\n\ndef _file_stat(path: Path) -> dict:\n    """Return a stable content fingerprint without hashing entire huge files."""\n    stat = path.stat()\n    h = hashlib.blake2b(digest_size=16)\n    with path.open("rb") as f:\n        head = f.read(65536)\n        if stat.st_size > 131072:\n            f.seek(max(0, stat.st_size - 65536))\n            tail = f.read(65536)\n        else:\n            tail = b""\n    h.update(head)\n    h.update(tail)\n    h.update(str(stat.st_size).encode())\n    return {"size": stat.st_size, "fingerprint": h.hexdigest()}\n\n\ndef _file_stat(path: Path) -> dict:\n    """Cheap identity check for a source file: size only.\n\n    mtime is deliberately NOT used here. Google Drive\'s FUSE mount does not\n    reliably preserve file mtimes across a remount (e.g. after\n    drive.flush_and_unmount() at the end of a session) -- a file\'s reported\n    mtime can drift between sessions even though its content never changed.\n    That drift changes how many files look "new/changed" from one session\n    to the next, and _load_resumable_progress refuses to resume at all when\n    that count (``batch_size``) doesn\'t match the checkpoint -- silently\n    forcing a full restart of the batch even when the token cache on disk\n    is perfectly fine. Size-only detection sidesteps this: these are static\n    text files that don\'t change in place, so size alone is a reliable\n    enough signal, and it isn\'t affected by Drive\'s mtime drift.\n    """\n    stat = path.stat()\n    return {"size": stat.st_size}\n\n\ndef _file_index_path(cache_dir: Path) -> Path:\n    return cache_dir / "file_index.json"\n\n\ndef _progress_path(cache_dir: Path) -> Path:\n    return cache_dir / "encode_progress.json"\n\ndef _encoding_timing_path(cache_dir: Path) -> Path:\n    return cache_dir / "encoding_checkpoint_timing.jsonl"\n\n\ndef _load_file_index(cache_dir: Path, tokenizer_expected: dict) -> Optional[dict]:\n    """Load the per-file token index, or None if absent/unusable.\n\n    Only usable if the tokenizer identity (vocab size + BPE merges) matches\n    exactly -- reusing token IDs encoded under different merges would\n    silently corrupt the stream, so any tokenizer change forces a clean\n    rebuild rather than a partial reuse.\n    """\n    path = _file_index_path(cache_dir)\n    if not path.exists():\n        return None\n    try:\n        index = json.loads(path.read_text(encoding="utf-8"))\n    except (json.JSONDecodeError, OSError):\n        return None\n    if index.get("tokenizer") != tokenizer_expected:\n        return None\n    return index\n\n\ndef _save_file_index(cache_dir: Path, index: dict) -> None:\n    _atomic_write_text(\n        _file_index_path(cache_dir),\n        json.dumps(index, indent=2),\n    )\n\n\ndef _load_resumable_progress(\n    cache_dir: Path, token_path: Path, tokenizer_expected: dict, batch_size: int,\n) -> Optional[dict]:\n    """Return a valid in-progress checkpoint for the current incremental\n    batch of new/changed files, or None if none applies.\n\n    A checkpoint is only usable if it was written for the exact same\n    tokenizer AND the exact same number of files in this batch -- if the set\n    of new/changed files differs from what the checkpoint expected (e.g.\n    yet another folder was added mid-run), we refuse to resume and let the\n    caller restart this batch cleanly instead of risking a misaligned token\n    stream.\n    """\n    progress_file = _progress_path(cache_dir)\n    if not progress_file.exists() or not token_path.exists():\n        return None\n    try:\n        progress = json.loads(progress_file.read_text(encoding="utf-8"))\n    except (json.JSONDecodeError, OSError):\n        return None\n    if progress.get("tokenizer") != tokenizer_expected:\n        return None\n    # New files may have appeared after an interrupted run.  If the checkpoint\n    # contains its exact batch_paths, those paths define the resumable batch;\n    # do not reject the checkpoint merely because the current corpus has more\n    # files now.  Older checkpoints without batch_paths retain the old\n    # batch-size guard.\n    if "batch_paths" not in progress and progress.get("batch_size") != batch_size:\n        return None\n\n    expected_bytes = int(progress["token_count"]) * array("I").itemsize\n    actual_bytes = token_path.stat().st_size\n    if actual_bytes < expected_bytes:\n        return None\n    if actual_bytes > expected_bytes:\n        # A partial extra file may have been written after the last\n        # checkpoint before the interruption. Truncate back to the last\n        # confirmed-good checkpoint boundary so the token stream stays\n        # file-aligned, then resume from there.\n        with token_path.open("r+b") as handle:\n            handle.truncate(expected_bytes)\n\n    return progress\n\n\ndef _tokenizer_path(cache_dir: Path) -> Path:\n    return cache_dir / "tokenizer.json"\n\n\ndef _atomic_write_text(path: Path, text: str, encoding: str = "utf-8") -> None:\n    """Atomically replace a small metadata/tokenizer JSON file."""\n    tmp = path.with_suffix(path.suffix + ".tmp")\n    tmp.write_text(text, encoding=encoding)\n    os.replace(tmp, path)\n\n\ndef _load_or_create_persistent_bpe(\n    cache_dir: Path,\n    all_paths: list[Path],\n    bpe_vocab_size: int,\n    bpe_sample_chars: int,\n    supplied: Optional[BPETokenizer],\n) -> BPETokenizer:\n    """Load the cache-owned tokenizer, creating it exactly once if necessary.\n\n    Training settings such as batch size, LR, epochs/max_steps, dropout, etc.\n    never participate in tokenizer creation.  A persistent tokenizer is what\n    makes the token IDs stable across Colab sessions and across later runs.\n    """\n    path = _tokenizer_path(cache_dir)\n\n    if supplied is not None:\n        # A training checkpoint may carry the tokenizer.  Persist it so future\n        # runs can use the same tokenizer even without a model checkpoint.\n        if path.exists():\n            try:\n                loaded = BPETokenizer.load(path)\n                if loaded.merges != supplied.merges:\n                    raise ValueError(\n                        "Supplied checkpoint tokenizer differs from the tokenizer "\n                        "stored in the bulk cache. Use the cache tokenizer or explicitly "\n                        "rebuild the corpus cache with a new cache directory."\n                    )\n                return loaded\n            except ValueError:\n                raise\n            except (OSError, KeyError, json.JSONDecodeError):\n                pass\n        payload = {"merges": [list(pair) for pair in supplied.merges]}\n        _atomic_write_text(path, json.dumps(payload, indent=2))\n        return supplied\n\n    if path.exists():\n        try:\n            tok = BPETokenizer.load(path)\n            print(f"Using persistent BPE tokenizer: {path}")\n            return tok\n        except (OSError, ValueError, KeyError, json.JSONDecodeError) as exc:\n            print(f"[WARNING] Could not load {path}: {exc}. Rebuilding tokenizer.")\n\n    print("Training BPE tokenizer once for this bulk cache...")\n    tok = BPETokenizer(\n        sample_book_text(all_paths, bpe_sample_chars),\n        vocab_size=bpe_vocab_size,\n        sample_chars=bpe_sample_chars,\n    )\n    payload = {"merges": [list(pair) for pair in tok.merges]}\n    _atomic_write_text(path, json.dumps(payload, indent=2))\n    print(f"Saved persistent BPE tokenizer: {path}")\n    return tok\n\n\ndef _quarantine_file(path: Path, reason: str = "invalid") -> None:\n    """Move a suspect cache file aside instead of deleting it."""\n    if not path.exists():\n        return\n    stamp = time.strftime("%Y%m%d_%H%M%S")\n    target = path.with_name(f"{path.name}.{reason}.{stamp}.bak")\n    counter = 1\n    while target.exists():\n        target = path.with_name(f"{path.name}.{reason}.{stamp}.{counter}.bak")\n        counter += 1\n    path.replace(target)\n    print(f"[SAFE RECOVERY] Moved {path.name} to {target.name}")\n\n\ndef build_or_load_bulk_tokens(\n    data_dir: str | Path,\n    cache_dir: str | Path,\n    *,\n    bpe_vocab_size: int = 16000,\n    bpe_sample_chars: int = 3_000_000,\n    tokenizer: Optional[BPETokenizer] = None,\n    rebuild: bool = False,\n    source_dirs=DEFAULT_SOURCE_DIRS,\n    source_formats: dict = None,\n) -> tuple[BPETokenizer, BulkTokenStore, dict]:\n    """Build/reuse a persistent memory-mapped bulk token cache.\n\n    ``source_formats`` maps each source-folder name to one of\n    SOURCE_FORMAT_PROSE / SOURCE_FORMAT_USER_ASSISTANT /\n    SOURCE_FORMAT_GENERIC_TURNS (see the module-level comment above\n    DEFAULT_SOURCE_FORMATS for what each does and why). Defaults to\n    DEFAULT_SOURCE_FORMATS; pass a copy with overrides to customize\n    without editing this file.\n\n    IMPORTANT:\n      * Changing training settings does NOT invalidate this cache.\n      * Existing files are never re-encoded when their size is unchanged.\n      * New files are appended only.\n      * A changed/deleted previously-encoded file triggers a clean rebuild,\n        because this append-only token stream cannot safely replace bytes in\n        the middle.\n      * The tokenizer is persisted inside ``cache_dir/tokenizer.json`` and is\n        therefore stable across Colab sessions.\n      * ``encode_progress.json`` resumes an interrupted batch from its last\n        checkpoint, even after the runtime disappears.\n    """\n    if tokenizer is not None and not isinstance(tokenizer, BPETokenizer):\n        raise ValueError("bulk training requires a BPETokenizer")\n\n    if source_formats is None:\n        source_formats = DEFAULT_SOURCE_FORMATS\n\n    sources = _collect_source_paths(data_dir, source_dirs)\n    all_paths = [p for name in sorted(sources) for p in sources[name]]\n    path_to_source = {\n        str(p): name for name, paths in sources.items() for p in paths\n    }\n\n    cache = Path(cache_dir)\n    cache.mkdir(parents=True, exist_ok=True)\n    token_path = cache / "tokens.int32.bin"\n    meta_path = cache / "metadata.json"\n    progress_path = _progress_path(cache)\n\n    # The tokenizer belongs to the corpus cache, not to the training run.\n    tokenizer = _load_or_create_persistent_bpe(\n        cache, all_paths, bpe_vocab_size, bpe_sample_chars, tokenizer\n    )\n\n    tokenizer_expected = {\n        "tokenizer": "bpe",\n        "vocab_size": tokenizer.vocab_size,\n        "merges": [list(pair) for pair in tokenizer.merges],\n        "corpus_format_version": CORPUS_FORMAT_VERSION,\n        "source_formats": dict(sorted(source_formats.items())),\n    }\n\n    if rebuild:\n        # Explicit rebuild is non-destructive: quarantine old artifacts\n        # instead of deleting them.\n        for path in (token_path, _file_index_path(cache), progress_path):\n            _quarantine_file(path, "explicit_rebuild")\n        file_index = None\n    else:\n        file_index = _load_file_index(cache, tokenizer_expected)\n\n    # An invalid/missing index means the existing binary cannot be trusted.\n    # NEVER append to it: that would mix token IDs from different corpus/parser\n    # identities. Quarantine the binary and start a clean stream.\n    if file_index is None:\n        if token_path.exists():\n            _quarantine_file(token_path, "cache_identity_changed")\n        if progress_path.exists():\n            _quarantine_file(progress_path, "cache_identity_changed")\n        files_record: dict[str, dict] = {}\n        base_token_count = 0\n    else:\n        files_record = file_index["files"]\n        base_token_count = int(file_index["token_count"])\n\n        # Append-only storage cannot remove/replace a file in the middle.\n        # If an existing file changed or disappeared, rebuild cleanly.\n        current_keys = {str(p) for p in all_paths}\n        changed = [\n            key for key, record in files_record.items()\n            if key not in current_keys or\n            _file_stat(Path(key)) != {k: record.get(k) for k in ("size", "fingerprint")}\n        ]\n        if changed:\n            print(\n                f"[WARNING] {len(changed)} previously-encoded file(s) were "\n                "changed or removed. Rebuilding the bulk token stream so stale "\n                "tokens cannot remain mixed with new content."\n            )\n            _quarantine_file(token_path, "changed_files")\n            _quarantine_file(progress_path, "changed_files")\n            files_record = {}\n            base_token_count = 0\n\n    # Determine only genuinely new files.\n    to_encode = []\n    for path in all_paths:\n        key = str(path)\n        stat = _file_stat(path)\n        recorded = files_record.get(key)\n        if recorded is None or any(recorded.get(k) != stat.get(k) for k in ("size", "fingerprint")):\n            to_encode.append(path)\n\n    token_count = base_token_count\n\n    if to_encode:\n        # Resume uses the exact file list from the interrupted batch.  This is\n        # deliberately independent of the current total corpus file count:\n        # if new files appeared after a disconnect, finish the old batch first.\n        resume_progress = None if rebuild else _load_resumable_progress(\n            cache, token_path, tokenizer_expected, len(to_encode)\n        )\n\n        if resume_progress is not None:\n            saved_paths = resume_progress.get("batch_paths")\n            current_paths = [str(p) for p in to_encode]\n            if saved_paths is not None and saved_paths != current_paths:\n                # New files were added/reordered. Try to resume the exact old\n                # batch instead of throwing away already-encoded progress.\n                saved_set = set(saved_paths)\n                if not all(p in {str(x) for x in all_paths} for p in saved_paths):\n                    resume_progress = None\n                else:\n                    # The old batch must be contiguous from its saved order;\n                    # after it completes, a subsequent invocation will append\n                    # newly discovered files.\n                    to_encode = [Path(p) for p in saved_paths]\n                    resume_progress = _load_resumable_progress(\n                        cache, token_path, tokenizer_expected, len(to_encode)\n                    )\n\n        if resume_progress is not None:\n            start_index = int(resume_progress["files_done"])\n            token_count = int(resume_progress["token_count"])\n            files_record = resume_progress["files_record"]\n            file_mode = "r+b"\n            print(\n                f"Resuming incremental encode from file "\n                f"{start_index:,}/{len(to_encode):,} | "\n                f"{token_count:,} tokens in cache so far"\n            )\n        else:\n            start_index = 0\n            token_count = base_token_count\n            expected_bytes = base_token_count * array("I").itemsize\n            if token_path.exists():\n                actual_bytes = token_path.stat().st_size\n                if actual_bytes != expected_bytes:\n                    # Never append to an unverified byte boundary.\n                    with token_path.open("r+b") as handle:\n                        handle.truncate(expected_bytes)\n            file_mode = "ab" if token_path.exists() else "wb"\n\n        batch_paths = [str(p) for p in to_encode]\n        with token_path.open(file_mode) as output:\n            if file_mode == "r+b":\n                output.seek(0, 2)\n\n            encode_run_start_time = time.monotonic()\n            last_progress_time = encode_run_start_time\n            last_progress_index = start_index\n            last_progress_token_count = token_count\n            timing_path = _encoding_timing_path(cache)\n            # Timing is append-only and survives disconnects. A resumed run\n            # starts a new timing interval from its current checkpoint; it never\n            # fabricates the time spent before the runtime disappeared.\n            timing_records = []\n            if timing_path.exists():\n                for line in timing_path.read_text(encoding="utf-8").splitlines()[-20:]:\n                    try: timing_records.append(json.loads(line))\n                    except Exception: pass\n            previous_checkpoint_time = None\n            previous_checkpoint_index = start_index\n            # Tracks whether the very next emitted block (file, or\n            # "---"-separated sub-block within a file) needs a leading\n            # eos_id. False only for the very first block of the entire\n            # cache; True forever after -- including across a resume,\n            # since token_count/start_index > 0 there means something\n            # was already emitted in an earlier run or earlier file.\n            is_first_emission = (token_count == 0 and start_index == 0)\n\n            for index, path in enumerate(\n                to_encode[start_index:], start=start_index + 1\n            ):\n                with path.open("r", encoding="utf-8", errors="ignore") as handle:\n                    raw_text = handle.read()\n\n                source_name = path_to_source[str(path)]\n                encoded, is_first_emission = _encode_source_file(\n                    tokenizer, source_name, raw_text, source_formats,\n                    is_first_emission, path.suffix\n                )\n\n                values = array("I", encoded)\n                values.tofile(output)\n\n                key = str(path)\n                stat = _file_stat(path)\n                files_record[key] = {\n                    "source": path_to_source[key],\n                    "size": stat["size"],\n                    "fingerprint": stat["fingerprint"],\n                    "token_start": token_count,\n                    "token_count": len(encoded),\n                }\n                token_count += len(encoded)\n\n                if index % ENCODE_CHECKPOINT_EVERY == 0 or index == len(to_encode):\n                    output.flush()\n                    os.fsync(output.fileno())\n                    _atomic_write_text(\n                        progress_path,\n                        json.dumps(\n                            {\n                                "tokenizer": tokenizer_expected,\n                                "batch_size": len(to_encode),\n                                "batch_paths": batch_paths,\n                                "files_done": index,\n                                "token_count": token_count,\n                                "files_record": files_record,\n                                "last_checkpoint_wall_time": time.time(),\n                                "last_checkpoint_interval_sec": interval_elapsed,\n                                "last_checkpoint_files_per_sec": files_per_sec,\n                                "eta_seconds_to_batch_end": eta_sec,\n                            },\n                            indent=2,\n                        ),\n                        encoding="utf-8",\n                    )\n\n                    now = time.monotonic()\n                    interval_files = index - last_progress_index\n                    interval_tokens = token_count - last_progress_token_count\n                    interval_elapsed = now - last_progress_time\n                    run_elapsed = now - encode_run_start_time\n                    files_per_sec = (\n                        interval_files / interval_elapsed\n                        if interval_elapsed > 0 else 0.0\n                    )\n                    tokens_per_sec = (\n                        interval_tokens / interval_elapsed\n                        if interval_elapsed > 0 else 0.0\n                    )\n                    remaining_files = max(0, len(to_encode) - index)\n                    eta_sec = (remaining_files / files_per_sec) if files_per_sec > 0 else None\n                    timing_record = {\n                        "timestamp": time.time(), "checkpoint_files": index,\n                        "interval_files": interval_files, "interval_sec": interval_elapsed,\n                        "files_per_sec": files_per_sec, "tokens_per_sec": tokens_per_sec,\n                        "eta_seconds_to_batch_end": eta_sec,\n                        "estimated_batch_end_unix": (time.time()+eta_sec if eta_sec is not None else None),\n                    }\n                    with timing_path.open("a", encoding="utf-8") as tf:\n                        tf.write(json.dumps(timing_record, sort_keys=True) + "\\n")\n                    print(\n                        f"Encoded {index:,}/{len(to_encode):,} new files "\n                        f"| {token_count:,} tokens total "\n                        f"| last {interval_files:,} files: "\n                        f"{_format_duration(interval_elapsed)} "\n                        f"({files_per_sec:.2f} files/s, "\n                        f"{tokens_per_sec:,.0f} tok/s) "\n                        f"| run elapsed: {_format_duration(run_elapsed)} "\n                        f"| checkpoint saved"\n                    )\n                    last_progress_time = now\n                    last_progress_index = index\n                    last_progress_token_count = token_count\n\n                if index % HARD_SYNC_INTERVAL == 0 or index == len(to_encode):\n                    print(\n                        f"  [hard sync] {index:,} files done -- "\n                        "forcing fsync and checking Drive sync..."\n                    )\n                    _hard_sync_checkpoint(output, token_path)\n\n        if progress_path.exists():\n            progress_path.unlink()\n    else:\n        print(\n            f"Reusing bulk token cache: {token_path} "\n            f"({token_count:,} tokens) -- no new or changed files"\n        )\n\n    source_token_counts: dict[str, int] = {}\n    for record in files_record.values():\n        source_token_counts[record["source"]] = (\n            source_token_counts.get(record["source"], 0) + record["token_count"]\n        )\n\n    file_index = {\n        "tokenizer": tokenizer_expected,\n        "files": files_record,\n        "token_count": token_count,\n    }\n    _save_file_index(cache, file_index)\n\n    metadata = {\n        **tokenizer_expected,\n        "token_count": token_count,\n        "source_token_counts": source_token_counts,\n        "file_count": len(files_record),\n    }\n    _atomic_write_text(\n        meta_path,\n        json.dumps(metadata, indent=2),\n    )\n\n    print(\n        f"Bulk corpus ready: {len(files_record):,} files, "\n        f"{token_count:,} tokens | by source: {source_token_counts}"\n    )\n\n    final_expected_bytes = token_count * array("I").itemsize\n    final_actual_bytes = token_path.stat().st_size\n    if final_actual_bytes != final_expected_bytes:\n        raise RuntimeError(\n            f"Bulk token cache is corrupt: {token_path} is "\n            f"{final_actual_bytes:,} bytes, but the file index claims "\n            f"{token_count:,} tokens ({final_expected_bytes:,} bytes). "\n            "Automatic deletion is disabled; inspect the cache and make a backup before any explicit rebuild."\n        )\n\n    return tokenizer, BulkTokenStore(token_path, token_count), metadata', encoding="utf-8")
print("Wrote:", path)
print("Bytes:", path.stat().st_size)


In [ ]:
# FILE: gamax1/train.py
from pathlib import Path

path = PROJECT_ROOT / 'gamax1/train.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text('"""Training utilities and CLI for GamaX1."""\n\nimport argparse\nfrom dataclasses import dataclass\nimport hashlib\nimport json\nimport math\nimport os\nimport time\n\nimport torch\n\nfrom .model import GamaX1Model\nfrom .tokenizer import BPETokenizer, CharTokenizer, WordTokenizer, word_tokenizer_warning\nfrom .bulk_corpus import build_or_load_bulk_tokens\nfrom .experiment_tracker import ExperimentTracker\n\n\ndef perplexity(loss: float) -> float:\n    """Return exp(loss), reporting infinity instead of overflowing."""\n    loss = float(loss)\n    return math.exp(loss) if loss < math.log(float.fromhex("0x1.fffffffffffffp+1023")) else float("inf")\n\n\ndef get_lr_schedule(step: int, max_steps: int, base_lr: float, warmup_steps: int) -> float:\n    """Linear warmup followed by cosine decay to ten percent of base LR."""\n    if max_steps <= 0:\n        raise ValueError("max_steps must be positive")\n    warmup_steps = max(0, min(warmup_steps, max_steps))\n    if warmup_steps and step <= warmup_steps:\n        return base_lr * step / warmup_steps\n    progress = (step - warmup_steps) / max(1, max_steps - warmup_steps)\n    progress = min(max(progress, 0.0), 1.0)\n    return base_lr * (0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * progress)))\n\n\ndef is_overfitting(val_losses, train_losses, patience: int) -> bool:\n    """Detect consecutive validation regression while training still improves."""\n    if patience <= 0 or len(val_losses) < patience + 1 or len(train_losses) < patience + 1:\n        return False\n    recent_val = val_losses[-(patience + 1):]\n    recent_train = train_losses[-(patience + 1):]\n    return (all(b > a for a, b in zip(recent_val, recent_val[1:])) and\n            all(b <= a for a, b in zip(recent_train, recent_train[1:])))\n\n\ndef tokens_per_parameter(token_count: int, parameter_count: int) -> float:\n    """Return the corpus-size-to-model-capacity heuristic used by training."""\n    if token_count < 0:\n        raise ValueError("token_count must not be negative")\n    if parameter_count <= 0:\n        raise ValueError("parameter_count must be positive")\n    return token_count / parameter_count\n\n\ndef is_memorization_detected(\n    token_count: int,\n    parameter_count: int,\n    train_loss: float,\n    val_loss: float,\n    min_tokens_per_param: float = 10.0,\n    perplexity_memorization_floor: float = 1.5,\n) -> bool:\n    """Detect implausibly low loss for a corpus that is too small for the model.\n\n    Unlike ``is_overfitting``, this intentionally does not require validation\n    loss to rise: a small validation split drawn from the same tiny corpus can\n    be memorized along with training data.\n    """\n    if min_tokens_per_param < 0:\n        raise ValueError("min_tokens_per_param must not be negative")\n    if perplexity_memorization_floor <= 0:\n        raise ValueError("perplexity_memorization_floor must be positive")\n    corpus_ratio = tokens_per_parameter(token_count, parameter_count)\n    best_perplexity = min(perplexity(train_loss), perplexity(val_loss))\n    return corpus_ratio < min_tokens_per_param and best_perplexity < perplexity_memorization_floor\n\n\ndef create_optimizer(model: GamaX1Model, lr: float, weight_decay: float) -> torch.optim.AdamW:\n    """Create the training optimizer in one testable place."""\n    return torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)\n\n\n@dataclass(frozen=True)\nclass AutoSizeResult:\n    """Architecture and capacity assessment selected by ``auto_size_model``."""\n\n    d_model: int\n    n_heads: int\n    n_layers: int\n    n_features: int\n    parameter_count: int\n    tokens_per_param: float\n    status: str\n\n\ndef _parameter_count_for_config(\n    vocab_size: int, block_size: int, d_model: int, n_heads: int,\n    n_layers: int, n_features: int,\n) -> int:\n    """Count actual trainable parameters for an auto-size candidate."""\n    # Candidate construction initializes tensors; preserve the training RNG\n    # state so enabling auto-size does not silently change reproducibility.\n    with torch.random.fork_rng(devices=[]):\n        model = GamaX1Model(\n            vocab_size=vocab_size, d_model=d_model, n_heads=n_heads,\n            n_layers=n_layers, n_features=n_features, max_seq_len=block_size,\n            sparsity_k_init=max(1, n_features // 2),\n            sparsity_k_min=max(1, n_features // 8),\n        )\n        return sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)\n\n\ndef auto_size_model(\n    token_count: int, vocab_size: int, block_size: int, n_heads: int,\n    target_tokens_per_param: float = 40.0, min_tokens_per_param: float = 10.0,\n) -> AutoSizeResult:\n    """Select a viable model, scaling upward only when the token budget permits.\n\n    The hard floor preserves a meaningful attention width and a wide sparse\n    feature space. On very small corpora that floor may exceed the recommended\n    capacity budget; the correct response is a warning and more data, never a\n    degenerate model with one-dimensional heads or a four-unit feature space.\n    """\n    if token_count <= 0 or vocab_size <= 0 or block_size <= 0:\n        raise ValueError("token_count, vocab_size, and block_size must be positive")\n    if target_tokens_per_param <= 0 or min_tokens_per_param <= 0:\n        raise ValueError("token-per-parameter targets must be positive")\n\n    selected_heads = max(2, n_heads)\n    base_d_model = max(64, selected_heads)\n    base_d_model = math.ceil(base_d_model / selected_heads) * selected_heads\n    # Each tuple increases usable depth/width while keeping n_features at 4x\n    # d_model, the minimum wide sparse-superposition regime.\n    candidate_shapes = [\n        (base_d_model, 2),\n        (base_d_model, 3),\n        (base_d_model * 2, 3),\n        (base_d_model * 2, 4),\n        (base_d_model * 4, 4),\n        (base_d_model * 4, 6),\n        (base_d_model * 8, 6),\n    ]\n    candidates = []\n    target_parameter_budget = token_count / target_tokens_per_param\n    minimum_parameter_budget = token_count / min_tokens_per_param\n    for d_model, n_layers in candidate_shapes:\n        n_features = 4 * d_model\n        parameter_count = _parameter_count_for_config(\n            vocab_size, block_size, d_model, selected_heads, n_layers, n_features,\n        )\n        candidates.append((d_model, n_layers, n_features, parameter_count))\n        if parameter_count > target_parameter_budget:\n            break\n\n    selected = candidates[0]\n    if selected[3] <= target_parameter_budget:\n        for candidate in candidates:\n            if candidate[3] <= target_parameter_budget:\n                selected = candidate\n            else:\n                break\n\n    d_model, n_layers, n_features, parameter_count = selected\n    ratio = tokens_per_parameter(token_count, parameter_count)\n    if candidates[0][3] > minimum_parameter_budget:\n        status = "hard_floor"\n    elif ratio < target_tokens_per_param:\n        status = "above_minimum_below_target"\n    else:\n        status = "comfortably_above_target"\n    return AutoSizeResult(\n        d_model=d_model, n_heads=selected_heads, n_layers=n_layers,\n        n_features=n_features, parameter_count=parameter_count,\n        tokens_per_param=ratio, status=status,\n    )\n\n\ndef get_batch(\n    data: torch.Tensor, block_size: int, batch_size: int, device: str,\n    start_indices: torch.Tensor = None,\n):\n    """Sample next-token windows, optionally from a predefined split."""\n    if start_indices is None:\n        ix = torch.randint(len(data) - block_size, (batch_size,))\n    else:\n        if start_indices.numel() == 0:\n            raise ValueError("start_indices must not be empty")\n        ix = start_indices[torch.randint(len(start_indices), (batch_size,))]\n    x = torch.stack([data[i:i + block_size] for i in ix])\n    y = torch.stack([data[i + 1:i + 1 + block_size] for i in ix])\n    # Bulk caches are int32 to keep the on-disk footprint small. Convert only\n    # the sampled batch to the Long dtype required by cross-entropy.\n    return x.to(device=device, dtype=torch.long), y.to(device=device, dtype=torch.long)\n\n\ndef split_training_windows(\n    data: torch.Tensor, block_size: int, validation_fraction: float = 0.1,\n    strategy: str = "random_windows", seed: int = 1337,\n):\n    """Create either a legacy tail split or disjoint randomized text windows.\n\n    Random windows are block-sized non-overlapping segments sampled throughout\n    the corpus. They produce a representative validation distribution for a\n    multi-book corpus without leaking individual tokens between train and\n    validation windows. ``tail`` remains useful for a deliberately harder\n    final-book/domain-shift evaluation.\n    """\n    if not 0 < validation_fraction < 1:\n        raise ValueError("validation_fraction must be between 0 and 1")\n    if strategy == "tail":\n        split_at = int((1 - validation_fraction) * len(data))\n        return data[:split_at], data[split_at:], None, None\n    if strategy != "random_windows":\n        raise ValueError(f"unknown validation split strategy: {strategy}")\n\n    starts = torch.arange(0, len(data) - block_size, block_size + 1)\n    if len(starts) < 2:\n        raise ValueError("corpus is too short for a non-overlapping random-window split")\n    generator = torch.Generator().manual_seed(seed)\n    shuffled = starts[torch.randperm(len(starts), generator=generator)]\n    val_count = max(1, int(round(validation_fraction * len(shuffled))))\n    return data, data, shuffled[val_count:], shuffled[:val_count]\n\n\ndef evaluate_loss(\n    model: GamaX1Model, data: torch.Tensor, block_size: int, eval_batch_size: int,\n    eval_batches: int, device: str, k: int = None, start_indices: torch.Tensor = None,\n    use_amp: bool = False,\n) -> float:\n    """Return the mean loss across independent validation batches.\n\n    A larger sampled token population makes validation metrics substantially\n    less sensitive to which rare words happen to appear in one small batch.\n    """\n    if eval_batches <= 0 or eval_batch_size <= 0:\n        raise ValueError("eval_batches and eval_batch_size must be positive")\n    losses = []\n    with torch.no_grad():\n        for _ in range(eval_batches):\n            xb, yb = get_batch(data, block_size, eval_batch_size, device, start_indices)\n            with torch.autocast(device_type="cuda", dtype=torch.float16,\n                                enabled=use_amp and device.startswith("cuda")):\n                _, loss = model(xb, targets=yb, k=k)\n            losses.append(loss.item())\n    return sum(losses) / len(losses)\n\n\ndef tokenizer_class(name: str):\n    return {"word": WordTokenizer, "bpe": BPETokenizer}.get(name, CharTokenizer)\n\n\ndef checkpoint_dict(model, optimizer, tokenizer, config, step: int, scaler=None):\n    """Collect state needed for a robust cross-session training resume."""\n    if isinstance(tokenizer, BPETokenizer):\n        vocab = None\n        merges = tokenizer.merges\n    elif isinstance(tokenizer, WordTokenizer):\n        vocab = tokenizer.tokens\n        merges = None\n    else:\n        vocab = tokenizer.chars\n        merges = None\n    return {\n        "model_state": model.state_dict(),\n        "optimizer_state": optimizer.state_dict(),\n        "sparsity_controller_state": model.sparsity_ctrl.state_dict(),\n        "ptm_states": model.ptm_state_dicts(),\n        "step": step,\n        "vocab": vocab,\n        "merges": merges,\n        "config": config,\n        "scaler_state": scaler.state_dict() if scaler is not None else None,\n        "rng_state": torch.get_rng_state(),\n        "cuda_rng_state_all": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,\n    }\n\n\ndef save_checkpoint(path, model, optimizer, tokenizer, config, step, scaler=None):\n    """Atomically save a checkpoint so a Colab interruption cannot leave a half-file."""\n    os.makedirs(os.path.dirname(os.path.abspath(path)), exist_ok=True)\n    tmp_path = path + ".tmp"\n    torch.save(\n        checkpoint_dict(model, optimizer, tokenizer, config, step, scaler=scaler),\n        tmp_path,\n    )\n    os.replace(tmp_path, path)\n    print(f"Saved checkpoint to {path}")\n\n\ndef main():\n    parser = argparse.ArgumentParser(description="Train GamaX1 on a text corpus.")\n    parser.add_argument("--data", type=str, default=os.path.join(\n        os.path.dirname(__file__), "..", "data", "sample_corpus.txt"))\n    parser.add_argument("--data_dir", type=str, default=None,\n                        help="Recursive directory of .txt books for bulk training. Uses BPE and a memory-mapped token cache.")\n    parser.add_argument("--bulk_cache_dir", type=str, default="data/bulk_cache",\n                        help="Directory for the bulk int32 token cache (default: data/bulk_cache).")\n    parser.add_argument("--rebuild_bulk_cache", action="store_true",\n                        help="Re-encode all books even if the bulk token cache is reusable.")\n    parser.add_argument("--out_dir", type=str, default="checkpoints")\n    parser.add_argument("--d_model", type=int, default=256)\n    parser.add_argument("--n_heads", type=int, default=4)\n    parser.add_argument("--n_layers", type=int, default=4)\n    parser.add_argument("--n_features", type=int, default=1024)\n    parser.add_argument("--auto_size_model", action="store_true",\n                        help="Choose a viable model from corpus token count without going below hard architecture floors. "\n                             "A rough safety net; manually size and validate models for serious use.")\n    parser.add_argument("--auto_size_target_tokens_per_param", type=float, default=40.0,\n                        help="Target corpus tokens per parameter for --auto_size_model (default: 40).")\n    parser.add_argument("--block_size", type=int, default=128)\n    parser.add_argument("--batch_size", type=int, default=32)\n    parser.add_argument("--lr", type=float, default=3e-4)\n    parser.add_argument("--weight_decay", type=float, default=0.01,\n                        help="AdamW weight decay regularization (default: 0.01).")\n    parser.add_argument("--dropout", type=float, default=0.1,\n                        help="Dropout used by embeddings, attention, FFNs, and residual paths (default: 0.1).")\n    parser.add_argument("--max_steps", type=int, default=2000)\n    parser.add_argument("--warmup_steps", type=int, default=None)\n    parser.add_argument("--eval_interval", type=int, default=200)\n    parser.add_argument("--eval_batches", type=int, default=10,\n                        help="Independent validation batches averaged at each evaluation (default: 10).")\n    parser.add_argument("--eval_batch_size", type=int, default=None,\n                        help="Sequences per validation batch. Defaults to --batch_size if not given -- "\n                             "NOT a larger fixed value: an eval batch bigger than the training batch runs "\n                             "right after a training step, while its memory is still reserved, and is a "\n                             "common cause of a CUDA out-of-memory crash immediately after step 1 succeeds. "\n                             "Pass a larger value explicitly only if you\'ve confirmed the GPU has headroom.")\n    parser.add_argument("--validation_split", choices=("random_windows", "tail"), default="random_windows",\n                        help="Validation split: representative non-overlapping random windows (default) or final corpus tail.")\n    parser.add_argument("--validation_split_seed", type=int, default=1337,\n                        help="Random-window validation split seed (default: 1337).")\n    parser.add_argument("--overfit_patience", type=int, default=3)\n    parser.add_argument("--min_tokens_per_param", type=float, default=10.0,\n                        help="Warn about memorization below this corpus-token/parameter ratio (default: 10).")\n    parser.add_argument("--perplexity_memorization_floor", type=float, default=1.5,\n                        help="Flag low-capacity-ratio runs when train or validation perplexity falls below this value (default: 1.5).")\n    parser.add_argument("--early_stop_on_overfit", action="store_true")\n    # Neural-network training checkpoint frequency. This is deliberately\n    # separate from bulk_corpus.py\'s 500-FILE encoding checkpoint. At every\n    # 500 optimizer steps we save both a numbered checkpoint (for history)\n    # and `gamax1_latest.pt` (for automatic resume after Colab disconnects).\n    parser.add_argument("--checkpoint_interval", type=int, default=500,\n                        help="Save a resumable training checkpoint every N steps (default: 500).")\n    parser.add_argument("--resume_from", type=str, default=None,\n                        help="Checkpoint to resume. If omitted, automatically resumes checkpoints/gamax1_latest.pt when present.")\n    parser.add_argument("--experiment_dir", type=str, default="experiments",\n                        help="Directory containing one persistent subdirectory per training run.")\n    parser.add_argument("--run_name", type=str, default=None,\n                        help="Optional explicit experiment run directory name; never overwrite an existing metrics file when resuming.")\n    parser.add_argument("--tokenizer", choices=("char", "word", "bpe"), default="char",\n                        help="char: character-level; word: word-level with <unk>; "\n                             "bpe: dependency-free byte-level BPE (recommended for real corpora).")\n    parser.add_argument("--bpe_vocab_size", type=int, default=8000,\n                        help="BPE vocabulary size, including the 256 byte tokens (default: 8000).")\n    parser.add_argument("--bpe_sample_chars", type=int, default=3_000_000,\n                        help="Chars of training text scanned for BPE pair statistics (default: 3000000). "\n                             "Larger is slightly better but costs linear Python time.")\n    # 15K covers common English across several novels while preventing a huge\n    # vocabulary projection from consuming nearly all parameters in small LMs.\n    parser.add_argument("--max_vocab_size", type=int, default=15_000,\n                        help="Maximum word-tokenizer vocabulary size, including <unk> (default: 15000). "\n                             "Ignored by the character and BPE tokenizers.")\n    parser.add_argument("--hex_influence", action="store_true")\n    parser.add_argument("--device", type=str, default=None)\n    parser.add_argument("--no_amp", action="store_true",\n                        help="Disable CUDA mixed-precision training (uses more GPU memory).")\n    args = parser.parse_args()\n    if args.eval_batch_size is None:\n        # Match the training batch size by default rather than a fixed,\n        # potentially much larger value -- see the flag\'s help text for why\n        # a bigger-than-training eval batch is a common CUDA OOM trap.\n        args.eval_batch_size = args.batch_size\n    if args.data_dir and args.tokenizer != "bpe":\n        parser.error("--data_dir requires --tokenizer bpe")\n    if args.warmup_steps is None:\n        args.warmup_steps = max(100, args.max_steps // 20)\n\n    device = args.device or ("cuda" if torch.cuda.is_available() else "cpu")\n\n    # Cross-session default: if the user does not explicitly choose a\n    # checkpoint, continue from the latest saved training state when one\n    # exists.  This is independent of the corpus token cache.\n    latest_checkpoint = os.path.join(args.out_dir, "gamax1_latest.pt")\n    resume_path = args.resume_from or (latest_checkpoint if os.path.exists(latest_checkpoint) else None)\n    resume_ckpt = (\n        torch.load(resume_path, map_location=device, weights_only=False)\n        if resume_path else None\n    )\n    if resume_path and resume_ckpt:\n        args.resume_from = resume_path\n    if resume_ckpt:\n        saved_cfg = resume_ckpt["config"]\n        for key in ("d_model", "n_heads", "n_layers", "n_features", "block_size", "dropout",\n                    "hex_influence", "tokenizer", "max_vocab_size", "bpe_vocab_size", "bpe_sample_chars"):\n            if key in saved_cfg:\n                setattr(args, key, saved_cfg[key])\n    print(f"Using device: {device}")\n\n    bulk_store = None\n    if args.data_dir:\n        saved_tokenizer = (\n            BPETokenizer(merges=resume_ckpt["merges"])\n            if resume_ckpt and resume_ckpt.get("merges") is not None\n            else None\n        )\n        tok, bulk_store, bulk_metadata = build_or_load_bulk_tokens(\n            args.data_dir, args.bulk_cache_dir,\n            bpe_vocab_size=args.bpe_vocab_size,\n            bpe_sample_chars=args.bpe_sample_chars,\n            tokenizer=saved_tokenizer,\n            rebuild=args.rebuild_bulk_cache,\n        )\n        # Keep the int32 memory-map view. Converting here would duplicate the\n        # full cache as an 8-byte-per-token tensor in RAM.\n        data = bulk_store.tensor\n        text = None\n        corpus_description = f"{bulk_metadata[\'file_count\']:,} books"\n    else:\n        with open(args.data, "r", encoding="utf-8") as f:\n            text = f.read()\n        tok_cls = tokenizer_class(args.tokenizer)\n        if resume_ckpt:\n            if args.tokenizer == "bpe" and resume_ckpt.get("merges") is not None:\n                tok = BPETokenizer(merges=resume_ckpt["merges"])\n            else:\n                tok = tok_cls(vocab=resume_ckpt["vocab"])\n        elif args.tokenizer == "bpe":\n            tok = BPETokenizer(text, vocab_size=args.bpe_vocab_size, sample_chars=args.bpe_sample_chars)\n        elif args.tokenizer == "word":\n            tok = WordTokenizer(text, max_vocab_size=args.max_vocab_size)\n        else:\n            tok = CharTokenizer(text)\n        word_tokens = len(WordTokenizer._tokenize(text)) if args.tokenizer == "word" else 0\n        warning = word_tokenizer_warning(args.tokenizer, word_tokens, tok.vocab_size)\n        if warning:\n            print(warning)\n        if args.tokenizer == "bpe":\n            data_path = os.path.abspath(args.data)\n            cache_path = f"{data_path}.bpe{args.bpe_vocab_size}.encoded.pt"\n            meta_path = cache_path + ".meta.json"\n            # A token cache is only valid for the exact source text and exact\n            # tokenizer implementation/merge set. The old cache key used only\n            # path+vocab size, so changing tokenizer code could silently reuse\n            # stale token IDs.\n            source_hash = hashlib.sha256(text.encode("utf-8")).hexdigest()\n            tokenizer_hash = hashlib.sha256(\n                json.dumps([list(m) for m in tok.merges], separators=(",", ":")).encode("utf-8")\n            ).hexdigest()\n            expected_meta = {\n                "cache_version": 2,\n                "source_sha256": source_hash,\n                "tokenizer_merges_sha256": tokenizer_hash,\n                "vocab_size": tok.vocab_size,\n                "bpe_vocab_size": args.bpe_vocab_size,\n            }\n            use_cache = False\n            if os.path.exists(cache_path) and os.path.exists(meta_path):\n                try:\n                    with open(meta_path, "r", encoding="utf-8") as f:\n                        cached_meta = json.load(f)\n                    use_cache = cached_meta == expected_meta\n                except (OSError, ValueError, TypeError):\n                    use_cache = False\n            if use_cache:\n                data = torch.load(cache_path, map_location="cpu", weights_only=True)\n                if not isinstance(data, torch.Tensor) or data.ndim != 1 or data.dtype != torch.long:\n                    use_cache = False\n                elif data.numel() and (int(data.min()) < 0 or int(data.max()) >= tok.vocab_size):\n                    use_cache = False\n            if use_cache:\n                print(f"Using validated BPE encoding cache: {cache_path}")\n            else:\n                data = torch.tensor(tok.encode(text), dtype=torch.long)\n                tmp_cache = cache_path + ".tmp"\n                tmp_meta = meta_path + ".tmp"\n                torch.save(data, tmp_cache)\n                with open(tmp_meta, "w", encoding="utf-8") as f:\n                    json.dump(expected_meta, f, indent=2, sort_keys=True)\n                os.replace(tmp_cache, cache_path)\n                os.replace(tmp_meta, meta_path)\n                print(f"Performed fresh BPE encode and rebuilt cache: {cache_path}")\n        else:\n            data = torch.tensor(tok.encode(text), dtype=torch.long)\n        corpus_description = f"{len(text):,} chars"\n    if len(data) <= args.block_size + 1:\n        raise ValueError("corpus must contain more tokens than block_size + 1")\n    train_data, val_data, train_starts, val_starts = split_training_windows(\n        data, args.block_size, strategy=args.validation_split, seed=args.validation_split_seed,\n    )\n    print(f"Corpus: {corpus_description}, {len(data):,} tokens, vocab size {tok.vocab_size} ({args.tokenizer})")\n\n    auto_size_result = None\n    if args.auto_size_model and not resume_ckpt:\n        auto_size_result = auto_size_model(\n            len(data), tok.vocab_size, args.block_size, args.n_heads,\n            args.auto_size_target_tokens_per_param, args.min_tokens_per_param,\n        )\n        args.d_model = auto_size_result.d_model\n        args.n_heads = auto_size_result.n_heads\n        args.n_layers = auto_size_result.n_layers\n        args.n_features = auto_size_result.n_features\n        print("Auto-size selection: "\n              f"corpus_tokens={len(data):,} | d_model={args.d_model} | n_heads={args.n_heads} "\n              f"| n_layers={args.n_layers} | n_features={args.n_features} "\n              f"| parameters={auto_size_result.parameter_count:,} "\n              f"| tokens/parameter={auto_size_result.tokens_per_param:.3f}")\n        if auto_size_result.status == "hard_floor":\n            safe_parameter_budget = len(data) / args.min_tokens_per_param\n            print("[WARNING] Even the minimum viable GamaX1 architecture "\n                  f"(d_model={args.d_model}, n_layers={args.n_layers}, n_features={args.n_features}, "\n                  f"~{auto_size_result.parameter_count:,} parameters) exceeds the recommended token budget "\n                  f"for this corpus (~{len(data):,} tokens; {args.min_tokens_per_param:g} tokens/parameter "\n                  f"minimum recommends staying under ~{safe_parameter_budget:,.0f} parameters). Proceeding "\n                  "with the minimum viable architecture anyway, but expect some memorization risk with a corpus "\n                  "this small -- consider adding more training data.")\n        elif auto_size_result.status == "above_minimum_below_target":\n            print("Auto-size status: above the minimum safety threshold but below the requested target; "\n                  "manual validation is recommended.")\n        else:\n            print("Auto-size status: comfortably above the requested token-per-parameter target.")\n\n    model = GamaX1Model(\n        vocab_size=tok.vocab_size, d_model=args.d_model, n_heads=args.n_heads,\n        n_layers=args.n_layers, n_features=args.n_features, max_seq_len=args.block_size,\n        dropout=args.dropout,\n        hex_influence=args.hex_influence, sparsity_k_init=max(1, args.n_features // 2),\n        sparsity_k_min=max(1, args.n_features // 8),\n    ).to(device)\n    use_amp = device.startswith("cuda") and not args.no_amp\n    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)\n    if use_amp:\n        print("CUDA mixed precision: enabled (use --no_amp to disable)")\n    optimizer = create_optimizer(model, args.lr, args.weight_decay)\n    start_step = 0\n    if resume_ckpt:\n        model.load_state_dict(resume_ckpt["model_state"])\n        optimizer.load_state_dict(resume_ckpt["optimizer_state"])\n        model.sparsity_ctrl.load_state_dict(resume_ckpt["sparsity_controller_state"])\n        model.load_ptm_state_dicts(resume_ckpt.get("ptm_states"))\n        if resume_ckpt.get("scaler_state"):\n            scaler.load_state_dict(resume_ckpt["scaler_state"])\n        if resume_ckpt.get("rng_state") is not None:\n            # map_location=device (cuda) moves every tensor in the checkpoint\n            # onto the GPU, including rng_state -- but torch.set_rng_state()\n            # only accepts a CPU-resident ByteTensor ("This function only\n            # works for CPU" per its own docstring) and raises "RNG state\n            # must be a torch.ByteTensor" on a CUDA one, even though it\'s\n            # still a ByteTensor by dtype. Move it back to CPU explicitly.\n            torch.set_rng_state(resume_ckpt["rng_state"].cpu())\n        if torch.cuda.is_available() and resume_ckpt.get("cuda_rng_state_all") is not None:\n            # Same map_location issue as rng_state above: CUDA RNG state is\n            # conventionally stored as CPU tensors even for a CUDA\n            # generator (torch.cuda.get_rng_state_all() itself returns CPU\n            # tensors) -- move each one back to CPU defensively.\n            torch.cuda.set_rng_state_all(\n                [s.cpu() for s in resume_ckpt["cuda_rng_state_all"]]\n            )\n        start_step = int(resume_ckpt["step"])\n        print(f"Resuming from step {start_step}: {args.resume_from}")\n    parameter_count = sum(p.numel() for p in model.parameters() if p.requires_grad)\n    corpus_ratio = tokens_per_parameter(len(data), parameter_count)\n    print(f"Model parameters: {parameter_count:,}")\n    print(f"Corpus/model size check: ~{len(data):,} tokens, ~{parameter_count:,} parameters "\n          f"(~{corpus_ratio:.3f} tokens/parameter). Recommended minimum is roughly "\n          f"{args.min_tokens_per_param:g} tokens/parameter to reduce memorization risk.")\n\n    os.makedirs(args.out_dir, exist_ok=True)\n    run_name = args.run_name\n    if run_name is None:\n        run_name = "resume_" + time.strftime("%Y%m%d_%H%M%S") if resume_ckpt else None\n    tracker = ExperimentTracker(args.experiment_dir, run_name=run_name, config=vars(args), resume=False)\n    val_history, train_history = [], []\n    recent_training_losses = []\n    memorization_warning_printed = False\n    t0 = time.time()\n    for step in range(start_step + 1, args.max_steps + 1):\n        lr = get_lr_schedule(step, args.max_steps, args.lr, args.warmup_steps)\n        for group in optimizer.param_groups:\n            group["lr"] = lr\n        model.train()\n        xb, yb = get_batch(train_data, args.block_size, args.batch_size, device, train_starts)\n        k = model.sparsity_ctrl.k\n        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):\n            _, loss = model(xb, targets=yb, k=k)\n        optimizer.zero_grad(set_to_none=True)\n        scaler.scale(loss).backward()\n        scaler.unscale_(optimizer)\n        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)\n        scaler.step(optimizer)\n        scaler.update()\n        model.sparsity_ctrl.step(loss.item())\n        recent_training_losses.append(loss.item())\n        recent_training_losses = recent_training_losses[-args.eval_batches:]\n\n        should_eval = step % args.eval_interval == 0 or step == start_step + 1\n        overfit = False\n        memorization = False\n        if should_eval:\n            model.eval()\n            val_value = evaluate_loss(\n                model, val_data, args.block_size, args.eval_batch_size,\n                args.eval_batches, device, k=model.sparsity_ctrl.k, start_indices=val_starts,\n                use_amp=use_amp,\n            )\n            train_value = sum(recent_training_losses) / len(recent_training_losses)\n            train_history.append(train_value)\n            val_history.append(val_value)\n            val_history, train_history = val_history[-10:], train_history[-10:]\n            active = model.active_units_per_token()\n            ratio = args.n_features * args.n_layers / max(active, 1)\n            print(f"step {step:5d} | lr {lr:.2e} | train_loss {train_value:.4f} | train_ppl {perplexity(train_value):.2f} "\n                  f"| val_loss {val_value:.4f} | val_ppl {perplexity(val_value):.2f} "\n                  f"| sparsity_k {k} | active_units/token {active} | compute_ratio_vs_dense {ratio:.2f}x "\n                  f"| {time.time() - t0:.1f}s")\n            tracker.log(step=step, lr=lr, train_loss=train_value, val_loss=val_value,\n                         train_ppl=perplexity(train_value), val_ppl=perplexity(val_value),\n                         sparsity_k=k, active_units_per_token=active,\n                         compute_ratio_vs_dense=ratio, elapsed_sec=time.time()-t0)\n            overfit = is_overfitting(val_history, train_history, args.overfit_patience)\n            if overfit:\n                print(f"[WARNING] Validation loss has increased for {args.overfit_patience} consecutive evals while "\n                      "training loss keeps dropping — this usually means the model is starting to memorize the "\n                      "training data rather than generalize. Consider: a larger/more varied corpus, early stopping, "\n                      "or reducing model size.")\n            memorization = is_memorization_detected(\n                len(data), parameter_count, train_value, val_value,\n                args.min_tokens_per_param, args.perplexity_memorization_floor,\n            )\n            if memorization and not memorization_warning_printed:\n                observed_ppl = min(perplexity(train_value), perplexity(val_value))\n                print("[WARNING] MEMORIZATION DETECTED: perplexity is "\n                      f"{observed_ppl:.2f}, close to the theoretical minimum of 1.0, and the corpus has only "\n                      f"~{len(data):,} tokens against a ~{parameter_count:,}-parameter model "\n                      f"(~{corpus_ratio:.3f} tokens per parameter, below the --min_tokens_per_param threshold "\n                      f"of {args.min_tokens_per_param:g}). The model has likely memorized the training corpus "\n                      "verbatim rather than learning generalizable language patterns. Fix by: (a) using a much "\n                      "larger corpus, (b) reducing model size (fewer/smaller layers, --n_features), or (c) both.")\n                memorization_warning_printed = True\n\n        if args.checkpoint_interval and step % args.checkpoint_interval == 0:\n            timing = tracker.checkpoint_timing(step, checkpoint_kind="training")\n            tracker.log(step=step, checkpoint=True, checkpoint_interval_sec=timing.get("interval_sec"),\n                         checkpoint_steps=timing.get("interval_steps"), checkpoint_steps_per_sec=timing.get("steps_per_sec"))\n            tracker.plot()\n            save_checkpoint(\n                os.path.join(args.out_dir, f"gamax1_step_{step}.pt"),\n                model, optimizer, tok, vars(args), step, scaler=scaler,\n            )\n            save_checkpoint(\n                os.path.join(args.out_dir, "gamax1_latest.pt"),\n                model, optimizer, tok, vars(args), step, scaler=scaler,\n            )\n        if (overfit or memorization) and args.early_stop_on_overfit:\n            print("Early stopping because --early_stop_on_overfit was set.")\n            break\n\n    final_step = step if "step" in locals() else start_step\n    ckpt_path = os.path.join(args.out_dir, "gamax1.pt")\n    save_checkpoint(\n        ckpt_path, model, optimizer, tok, vars(args), final_step, scaler=scaler\n    )\n    save_checkpoint(\n        os.path.join(args.out_dir, "gamax1_latest.pt"),\n        model, optimizer, tok, vars(args), final_step, scaler=scaler\n    )\n    tracker.write_summary(final_step=final_step, parameter_count=parameter_count,\n                           corpus_tokens=len(data), tokens_per_parameter=corpus_ratio,\n                           final_train_loss=(train_history[-1] if train_history else None),\n                           final_val_loss=(val_history[-1] if val_history else None),\n                           final_train_ppl=(perplexity(train_history[-1]) if train_history else None),\n                           final_val_ppl=(perplexity(val_history[-1]) if val_history else None))\n    tracker.plot()\n    tok.save(os.path.join(args.out_dir, "tokenizer.json"))\n    if bulk_store is not None:\n        del data\n        bulk_store.close()\n\n\nif __name__ == "__main__":\n    main()', encoding="utf-8")
print("Wrote:", path)
print("Bytes:", path.stat().st_size)


In [ ]:
# FILE: gamax1/layers.py
from pathlib import Path

path = PROJECT_ROOT / 'gamax1/layers.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text('"""\ngamax1/layers.py\n================\nCore Aetherion mechanisms, translated from the numpy research prototype\n(see Aetherion Technical Report v4) into PyTorch nn.Modules suitable for\na real, GPU-trainable NLP model.\n\nEvery mechanism here is implemented the way it was *validated* to work\nin the research report, not the way it was first (and incorrectly)\nimplemented. Where a mechanism\'s benefit was found to be conditional\n(e.g. hexagonal neighbor influence only helps on clustered data), that\ncondition is documented and the module defaults to a safe setting.\n\nDesign note on loss functions: the research report\'s "loss-metric\nmismatch" finding (Section 6.2) is about plain binary cross-entropy on\na highly-imbalanced multi-label *sparse recovery* target -- it does\nNOT apply to standard categorical next-token language-model cross-\nentropy, which is a well-posed single-correct-class loss. GamaX1 uses\nstandard cross-entropy for language modeling; the pairwise-ranking-loss\nfix is deliberately NOT re-applied here, since the failure mode it\nfixes does not exist in this task shape.\n"""\n\nimport math\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\n\nclass DynamicSparsityController:\n    """Adaptive sparsity level S(t), fixed per Aetherion Section 5.2.\n\n    VALIDATED FIX: gate sparsity reduction on the *trend* of training\n    loss (a moving average comparison), never on instantaneous\n    interference/loss alone -- gating on instantaneous signal was the\n    root cause of the sparsity-collapse failure mode in the research\n    report. A separate `exploration_fraction` knob is kept independent\n    of the sparsity level itself (Design Principle 2): how sparse the\n    *representation* is and how much *extra random exploration* happens\n    during training are different concerns and must not share one\n    variable.\n    """\n\n    def __init__(self, k_init, k_min, k_max, trend_window=50, patience=200):\n        self.k = k_init\n        self.k_min = k_min\n        self.k_max = k_max\n        self.trend_window = trend_window\n        self.patience = patience\n        self.exploration_fraction = 0.10  # independent of k; decays separately\n        self._loss_history = []\n        self._steps_since_change = 0\n\n    def step(self, loss_value: float):\n        """Call once per training step with the current scalar loss.\n\n        The controller treats k_max as a true mathematical upper bound even\n        when loading a checkpoint produced by an older implementation.\n        """\n        self.k = max(self.k_min, min(self.k_max, int(self.k)))\n        self._loss_history.append(float(loss_value))\n        self._loss_history = self._loss_history[-(self.trend_window * 3):]\n        self._steps_since_change += 1\n\n        if len(self._loss_history) < self.trend_window * 2:\n            return self.k  # not enough history yet to judge a trend\n\n        recent = sum(self._loss_history[-self.trend_window:]) / self.trend_window\n        prior = sum(self._loss_history[-2 * self.trend_window:-self.trend_window]) / self.trend_window\n        improving = recent < prior * 0.995  # trend-based, not instantaneous\n\n        if improving and self._steps_since_change > self.patience and self.k > self.k_min:\n            self.k = max(self.k_min, int(self.k * 0.9))\n            self._steps_since_change = 0\n            self.exploration_fraction = max(0.02, self.exploration_fraction * 0.85)\n\n        return self.k\n\n    def state_dict(self):\n        return {\n            "k": self.k,\n            "exploration_fraction": self.exploration_fraction,\n            "loss_history": list(self._loss_history),\n            "steps_since_change": self._steps_since_change,\n        }\n\n    def load_state_dict(self, state):\n        self.k = max(self.k_min, min(self.k_max, int(state["k"])))\n        self.exploration_fraction = state["exploration_fraction"]\n        # .get() with a safe default: an older checkpoint saved before this\n        # fix won\'t have these two keys. Resume still works, just without\n        # the trend history -- see the fix note below for what this changes.\n        self._loss_history = list(state.get("loss_history", []))\n        self._steps_since_change = state.get("steps_since_change", 0)\n\n\nclass ProbationaryMemoryTracker:\n    """Dead-feature prevention, fixed per Aetherion Section 5.3.\n\n    Tracks, per hidden unit, how often it was among the top-K active\n    set. Units that go too long without activating are placed on\n    "probation" and forced into the active set periodically (a small\n    guaranteed nudge) until they demonstrate they can compete on merit\n    again.\n\n    VALIDATED FIX: only count events where a unit *should plausibly*\n    matter (i.e. only track under-activation, not simple absence);\n    true "not needed right now" cases are not penalized, avoiding the\n    bookkeeping bug that caused threshold flicker in the original\n    prototype. Entry requires `miss_threshold` consecutive misses; exit\n    requires `success_threshold` consecutive forced-successes -- an\n    explicit, asymmetric hysteresis band, not a single shared count.\n    """\n\n    def __init__(self, n_units, miss_threshold=4, success_threshold=3, nudge_period=20):\n        self.n_units = n_units\n        self.miss_threshold = miss_threshold\n        self.success_threshold = success_threshold\n        self.nudge_period = nudge_period\n        self.miss_count = torch.zeros(n_units, dtype=torch.long)\n        self.success_count = torch.zeros(n_units, dtype=torch.long)\n        self.on_probation = torch.zeros(n_units, dtype=torch.bool)\n        self._step_count = 0\n\n    def update(self, active_mask_batch: torch.Tensor):\n        """active_mask_batch: (batch, n_units) bool, True where a unit\n        was in the top-K active set for that sample this step."""\n        self._step_count += 1\n        active_any = active_mask_batch.any(dim=0).cpu()\n\n        missed = ~active_any & ~self.on_probation\n        self.miss_count[missed] += 1\n        self.miss_count[active_any] = 0\n        newly_probation = self.miss_count >= self.miss_threshold\n        self.on_probation |= newly_probation\n\n        got_forced_success = active_any & self.on_probation\n        self.success_count[got_forced_success] += 1\n        self.success_count[~got_forced_success & self.on_probation] = 0\n        exiting = self.on_probation & (self.success_count >= self.success_threshold)\n        self.on_probation[exiting] = False\n        self.miss_count[exiting] = 0\n        self.success_count[exiting] = 0\n\n    def nudge_indices(self):\n        """Units due for a forced-inclusion nudge this step."""\n        if self._step_count % self.nudge_period != 0:\n            return torch.empty(0, dtype=torch.long)\n        return self.on_probation.nonzero(as_tuple=True)[0]\n\n    def population(self):\n        return int(self.on_probation.sum().item())\n\n    def state_dict(self):\n        """Plain-object state, saved/restored explicitly by the caller\n        (this class is not an nn.Module, so ordinary checkpoint\n        save/load never touches it -- see the fix note in train.py)."""\n        return {\n            "miss_count": self.miss_count.clone(),\n            "success_count": self.success_count.clone(),\n            "on_probation": self.on_probation.clone(),\n            "step_count": self._step_count,\n        }\n\n    def load_state_dict(self, state):\n        # .cpu() is essential, not defensive-only: on a GPU run, the\n        # checkpoint these tensors came from was loaded via\n        # torch.load(..., map_location=device) with device=\'cuda\', which\n        # moves EVERY tensor in the checkpoint onto the GPU -- including\n        # these, even though PTM is a plain object (not an nn.Module) that\n        # model.to(device) never touches, so update() below still expects\n        # CPU tensors (it explicitly .cpu()s the mask it compares against).\n        # Without this, resume crashes with a cuda:0/cpu device mismatch\n        # the first time update() runs -- the same class of bug fixed for\n        # rng_state/cuda_rng_state_all in train.py\'s checkpoint resume.\n        self.miss_count = state["miss_count"].clone().cpu()\n        self.success_count = state["success_count"].clone().cpu()\n        self.on_probation = state["on_probation"].clone().cpu()\n        self._step_count = state["step_count"]\n\n\ndef build_hex_neighbor_table(n_features: int) -> torch.Tensor:\n    """Build six-neighbor indices; invalid positions use self as a gather-safe placeholder."""\n    side = max(1, int(math.ceil(math.sqrt(n_features))))\n    neighbors = torch.arange(n_features).unsqueeze(1).repeat(1, 6)\n    for idx in range(n_features):\n        r, c = divmod(idx, side)\n        parity = r % 2\n        offsets = [(-1, 0), (1, 0), (0, -1), (0, 1),\n                   (-1, 1 - 2 * parity), (1, 1 - 2 * parity)]\n        for k, (dr, dc) in enumerate(offsets):\n            nr, nc = r + dr, c + dc\n            nidx = nr * side + nc\n            if 0 <= nr and 0 <= nc < side and 0 <= nidx < n_features:\n                neighbors[idx, k] = nidx\n    return neighbors\n\n\ndef build_hex_neighbor_mask(n_features: int) -> torch.Tensor:\n    """Return True for real neighbors and False for boundary padding slots."""\n    side = max(1, int(math.ceil(math.sqrt(n_features))))\n    valid = torch.zeros((n_features, 6), dtype=torch.bool)\n    for idx in range(n_features):\n        r, c = divmod(idx, side)\n        parity = r % 2\n        offsets = [(-1, 0), (1, 0), (0, -1), (0, 1),\n                   (-1, 1 - 2 * parity), (1, 1 - 2 * parity)]\n        for k, (dr, dc) in enumerate(offsets):\n            nr, nc = r + dr, c + dc\n            nidx = nr * side + nc\n            if 0 <= nr and 0 <= nc < side and 0 <= nidx < n_features:\n                valid[idx, k] = True\n    return valid\n\n\nclass HexNeighborInfluence(nn.Module):\n    """Hexagonal-lattice neighbor influence, per Aetherion Section 3.2 / 5.7.\n\n    VALIDATED, CONDITIONAL finding: this mechanism measurably speeds up\n    early convergence only when the true underlying structure of the\n    hidden representation clusters at a scale >= the 6-neighbor\n    neighborhood; on unclustered/unstructured representations it can\n    inject noise rather than signal (Section 6.3). There is no way to\n    guarantee a language model\'s learned hidden features will cluster\n    at the right scale, so this module defaults to a *small* decay and\n    is explicitly OFF by default in GamaX1Block. Enable deliberately,\n    and evaluate its effect empirically on your task, rather than\n    assuming a benefit.\n    """\n\n    def __init__(self, n_features: int, decay: float = 0.05):\n        super().__init__()\n        table = build_hex_neighbor_table(n_features)\n        mask = build_hex_neighbor_mask(n_features)\n        self.register_buffer("neighbor_table", table)\n        self.register_buffer("neighbor_mask", mask)\n        self.decay = decay\n\n    def forward(self, activations: torch.Tensor) -> torch.Tensor:\n        # activations: (..., n_features). Self-index is only a gather-safe\n        # placeholder; invalid boundary positions are excluded from the mean.\n        neighbor_vals = activations[..., self.neighbor_table]  # (..., n_features, 6)\n        mask = self.neighbor_mask.to(dtype=activations.dtype)\n        denom = mask.sum(dim=-1).clamp_min(1.0)\n        influence = (neighbor_vals * mask).sum(dim=-1) / denom\n        return activations + influence * self.decay\n\n\nclass SparseSuperpositionLinear(nn.Module):\n    """The core efficiency mechanism, validated in Aetherion Sections\n    5.1 and 5.12: a wide feature space (n_features >> d_model) where\n    only the top-K units are kept active (ReLU\'d) per sample, the rest\n    zeroed. In the research report this matched 96-98% of a dense\n    baseline\'s accuracy at roughly half to a fifth of the compute.\n\n    `k` is supplied externally per step by a DynamicSparsityController\n    so representation sparsity can adapt over training.\n    """\n\n    def __init__(self, d_model: int, n_features: int, hex_influence: bool = False):\n        super().__init__()\n        self.d_model = d_model\n        self.n_features = n_features\n        self.in_proj = nn.Linear(d_model, n_features)\n        self.out_proj = nn.Linear(n_features, d_model)\n        self.hex = HexNeighborInfluence(n_features) if hex_influence else None\n        self.last_active_mask = None  # exposed for ProbationaryMemoryTracker\n\n    def forward(self, x: torch.Tensor, k: int, nudge_indices: torch.Tensor = None) -> torch.Tensor:\n        # x: (batch, seq, d_model)\n        pre = F.relu(self.in_proj(x))\n        if self.hex is not None:\n            pre = F.relu(self.hex(pre))\n\n        k = max(1, min(int(k), self.n_features))\n        topk_vals, topk_idx = pre.topk(k, dim=-1)\n        sparse = torch.zeros_like(pre)\n        sparse.scatter_(-1, topk_idx, topk_vals)\n\n        if nudge_indices is not None and nudge_indices.numel() > 0:\n            # PTM nudge: force-include probationary units at a small,\n            # non-disruptive magnitude so they keep receiving gradient.\n            # FIX: no .detach() here -- the previous version detached this\n            # value, which silently severed the gradient back to in_proj\n            # for exactly the features this mechanism exists to train,\n            # defeating its stated purpose (dead-feature prevention can\'t\n            # make a probationary feature competitive again if it never\n            # receives a gradient while forced active).\n            nudge_vals = pre[..., nudge_indices] * 0.5 + 1e-3\n            sparse[..., nudge_indices] = torch.maximum(sparse[..., nudge_indices], nudge_vals)\n\n        mask = sparse > 0\n        self.last_active_mask = mask.reshape(-1, self.n_features).detach()\n\n        return self.out_proj(sparse)\n\n\nclass RouterExpert(nn.Module):\n    """Layer-skip decision, validated per Aetherion Sections 5.4/5.11.\n\n    The research report found a small TRAINED classifier reached 100%\n    agreement with ground-truth difficulty, against 81% for a hand-\n    tuned heuristic -- so GamaX1\'s router is trained end-to-end (a\n    tiny linear-sigmoid head over pooled hidden state), not hand-tuned.\n    Used at inference time to skip deeper blocks for easy sequences.\n    """\n\n    def __init__(self, d_model: int):\n        super().__init__()\n        self.probe = nn.Linear(d_model, 1)\n\n    def forward(self, pooled_hidden: torch.Tensor) -> torch.Tensor:\n        return torch.sigmoid(self.probe(pooled_hidden)).squeeze(-1)  # P(needs deeper layer)\n\n\nclass ValidatorExpert(nn.Module):\n    """Confidence-gated early exit, validated per Aetherion Section 5.6.\n\n    VALIDATED FIX: a raw confidence/margin metric was found to trend\n    the *wrong way* as answer quality improves under iterative\n    refinement, so gating must be based on answer STABILITY (does the\n    predicted top-token set stop changing across refinement steps?),\n    not raw confidence. `is_stable` implements exactly that check.\n    """\n\n    def __init__(self, patience: int = 1):\n        super().__init__()\n        self.patience = patience\n        self._history = []\n\n    def is_stable(self, top_token_ids: torch.Tensor) -> bool:\n        self._history.append(top_token_ids.detach().cpu())\n        if len(self._history) <= self.patience:\n            return False\n        recent = self._history[-(self.patience + 1):]\n        stable = all(torch.equal(recent[i], recent[-1]) for i in range(len(recent) - 1))\n        return stable\n\n    def reset(self):\n        self._history = []', encoding="utf-8")
print("Wrote:", path)
print("Bytes:", path.stat().st_size)


In [ ]:
# FILE: gamax1/model.py
from pathlib import Path

path = PROJECT_ROOT / 'gamax1/model.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text('"""\ngamax1/model.py\n================\nGamaX1: first working version of Aetherion as a real, trainable NLP\nlanguage model.\n\nArchitecture note (honest design choice): the Aetherion research\nreport\'s core validated claim is about REPLACING A DENSE FEED-FORWARD\nLAYER with a sparse-superposition layer at large-but-sparse width,\nretaining ~96-98% accuracy at a fraction of the compute (Sections 5.1,\n5.12). It was not tested as a replacement for attention/token-mixing.\nGamaX1 therefore uses standard causal multi-head self-attention for\nsequence/token mixing (attention is a well-established, necessary\nmechanism for language modeling that Aetherion was never proposed as a\nreplacement for) and substitutes the Transformer\'s usual dense FFN\nwith an AetherionFFN block (SparseSuperpositionLinear + dynamic\nsparsity + probationary memory). This is the most defensible way to\nbring the *validated* Aetherion contributions into a real NLP model\nwithout overclaiming mechanisms that were never tested at this task.\n\nRouter/Validator experts (Section 5.4/5.6/5.11) are wired in as an\nINFERENCE-TIME compute-saving option (`use_hierarchical_exit=True` in\n`generate`), since dynamic per-sample depth is straightforward at\ninference (sequential decoding) but would require complex ragged-batch\nhandling to train efficiently -- a limitation stated plainly rather\nthan hidden.\n"""\n\nimport math\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\nfrom .layers import (\n    SparseSuperpositionLinear,\n    DynamicSparsityController,\n    ProbationaryMemoryTracker,\n    RouterExpert,\n    ValidatorExpert,\n)\n\n\ndef apply_repetition_penalty(logits: torch.Tensor, present: torch.Tensor, penalty: float) -> torch.Tensor:\n    """Suppress tokens already present in the sequence during sampling.\n\n    ``logits``: (batch, vocab); ``present``: (batch, vocab) bool mask, True\n    where that batch row\'s own sequence already contains the token -- or a\n    (vocab,) mask applied identically to every row, for a single shared\n    sequence. Positive logits are divided by ``penalty`` and negative ones\n    are multiplied by it, so both directions push the token\'s probability\n    down. A penalty of 1.0 is the identity. Uses ``torch.where`` (elementwise,\n    broadcasting-safe) rather than boolean fancy-indexing, so a per-row\n    ``present`` mask is applied per-row rather than collapsing the batch.\n    """\n    if penalty == 1.0:\n        return logits\n    scaled = torch.where(logits > 0, logits / penalty, logits * penalty)\n    return torch.where(present, scaled, logits)\n\n\nclass CausalSelfAttention(nn.Module):\n    def __init__(self, d_model: int, n_heads: int, max_seq_len: int, dropout: float = 0.1):\n        super().__init__()\n        assert d_model % n_heads == 0\n        self.n_heads = n_heads\n        self.head_dim = d_model // n_heads\n        self.qkv = nn.Linear(d_model, 3 * d_model)\n        self.out_proj = nn.Linear(d_model, d_model)\n        self.dropout = nn.Dropout(dropout)\n        mask = torch.tril(torch.ones(max_seq_len, max_seq_len, dtype=torch.bool)).view(\n            1, 1, max_seq_len, max_seq_len\n        )\n        self.register_buffer("causal_mask", mask)\n\n    def forward(self, x):\n        B, T, C = x.shape\n        qkv = self.qkv(x).view(B, T, 3, self.n_heads, self.head_dim).permute(2, 0, 3, 1, 4)\n        q, k, v = qkv[0], qkv[1], qkv[2]  # (B, n_heads, T, head_dim)\n\n        # SDPA selects FlashAttention or the memory-efficient CUDA kernel\n        # when available, avoiding materializing the full attention matrix.\n        # This is important for long contexts and larger training batches.\n        out = F.scaled_dot_product_attention(\n            q, k, v,\n            attn_mask=self.causal_mask[:, :, :T, :T],\n            dropout_p=self.dropout.p if self.training else 0.0,\n        )  # (B, n_heads, T, head_dim)\n        out = out.transpose(1, 2).contiguous().view(B, T, C)\n        return self.out_proj(out)\n\n\nclass AetherionFFN(nn.Module):\n    """Drop-in replacement for a Transformer block\'s dense FFN, using\n    the validated sparse-superposition mechanism instead of a dense\n    hidden layer."""\n\n    def __init__(self, d_model: int, n_features: int, dropout: float = 0.1,\n                 hex_influence: bool = False):\n        super().__init__()\n        self.sparse = SparseSuperpositionLinear(d_model, n_features, hex_influence=hex_influence)\n        self.ptm = ProbationaryMemoryTracker(n_features)\n        self.dropout = nn.Dropout(dropout)\n\n    def forward(self, x, k: int, use_ptm: bool = True):\n        nudge = self.ptm.nudge_indices() if use_ptm else None\n        out = self.sparse(x, k=k, nudge_indices=nudge)\n        if use_ptm and self.training:\n            self.ptm.update(self.sparse.last_active_mask)\n        return self.dropout(out)\n\n\nclass DenseFFN(nn.Module):\n    """Dense control FFN with the same projection shapes as AetherionFFN.\n\n    It exists solely to make sparse-versus-dense experiments fair: both paths\n    have the same feature width and parameter count, but this path evaluates\n    every hidden feature for every token.\n    """\n\n    def __init__(self, d_model: int, n_features: int, dropout: float = 0.1):\n        super().__init__()\n        self.in_proj = nn.Linear(d_model, n_features)\n        self.out_proj = nn.Linear(n_features, d_model)\n        self.dropout = nn.Dropout(dropout)\n\n    def forward(self, x, k: int = None, use_ptm: bool = True):\n        return self.dropout(self.out_proj(F.relu(self.in_proj(x))))\n\n\nclass GamaX1Block(nn.Module):\n    def __init__(self, d_model, n_heads, n_features, max_seq_len, dropout=0.1,\n                 hex_influence=False, dense_mode=False):\n        super().__init__()\n        self.ln1 = nn.LayerNorm(d_model)\n        self.attn = CausalSelfAttention(d_model, n_heads, max_seq_len, dropout)\n        self.ln2 = nn.LayerNorm(d_model)\n        self.ffn = (DenseFFN(d_model, n_features, dropout=dropout) if dense_mode else\n                    AetherionFFN(d_model, n_features, dropout=dropout,\n                                 hex_influence=hex_influence))\n        self.dropout = nn.Dropout(dropout)\n\n    def forward(self, x, k: int, use_ptm: bool = True):\n        x = x + self.dropout(self.attn(self.ln1(x)))\n        x = x + self.dropout(self.ffn(self.ln2(x), k=k, use_ptm=use_ptm))\n        return x\n\n\nclass GamaX1Model(nn.Module):\n    """First version ("GamaX1") of Aetherion as a real causal language\n    model. See module docstring for the architecture\'s relationship to\n    the research report\'s validated findings.\n    """\n\n    def __init__(\n        self,\n        vocab_size: int,\n        d_model: int = 256,\n        n_heads: int = 4,\n        n_layers: int = 4,\n        n_features: int = 1024,\n        max_seq_len: int = 256,\n        dropout: float = 0.1,\n        hex_influence: bool = False,\n        sparsity_k_init: int = 256,\n        sparsity_k_min: int = 64,\n        dense_mode: bool = False,\n    ):\n        super().__init__()\n        self.max_seq_len = max_seq_len\n        self.n_features = n_features\n        self.dense_mode = dense_mode\n        self.token_emb = nn.Embedding(vocab_size, d_model)\n        self.pos_emb = nn.Embedding(max_seq_len, d_model)\n        self.dropout = nn.Dropout(dropout)\n        self.blocks = nn.ModuleList([\n            GamaX1Block(d_model, n_heads, n_features, max_seq_len, dropout,\n                         hex_influence, dense_mode)\n            for _ in range(n_layers)\n        ])\n        self.ln_f = nn.LayerNorm(d_model)\n        self.head = nn.Linear(d_model, vocab_size, bias=False)\n\n        self.sparsity_ctrl = DynamicSparsityController(\n            k_init=sparsity_k_init, k_min=sparsity_k_min, k_max=n_features,\n        )\n        self.router = RouterExpert(d_model)\n        self.validator = ValidatorExpert(patience=1)\n\n        self.apply(self._init_weights)\n        # Standard language-model weight tying: input and output vocabulary\n        # representations share one matrix, avoiding a second vocab-sized\n        # parameter block that otherwise dominates small models with large\n        # word vocabularies.\n        self.head.weight = self.token_emb.weight\n\n    @staticmethod\n    def _init_weights(module):\n        if isinstance(module, nn.Linear):\n            nn.init.normal_(module.weight, mean=0.0, std=0.02)\n            if module.bias is not None:\n                nn.init.zeros_(module.bias)\n        elif isinstance(module, nn.Embedding):\n            nn.init.normal_(module.weight, mean=0.0, std=0.02)\n\n    def forward(self, idx, targets=None, k: int = None, use_ptm: bool = True):\n        B, T = idx.shape\n        assert T <= self.max_seq_len, "sequence length exceeds max_seq_len"\n        k = k if k is not None else self.sparsity_ctrl.k\n\n        pos = torch.arange(T, device=idx.device).unsqueeze(0)\n        x = self.dropout(self.token_emb(idx) + self.pos_emb(pos))\n\n        for block in self.blocks:\n            x = block(x, k=k, use_ptm=use_ptm)\n\n        x = self.ln_f(x)\n        logits = self.head(x)\n\n        loss = None\n        if targets is not None:\n            # Standard categorical cross-entropy for next-token prediction.\n            # (Deliberately NOT the pairwise ranking loss used elsewhere in\n            # the research report -- see module docstring / layers.py.)\n            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))\n\n        return logits, loss\n\n    def active_units_per_token(self, k: int = None) -> int:\n        """Compute-accounting helper: how many sparse hidden units are\n        actually evaluated per token, summed across all blocks --\n        directly comparable to a dense baseline\'s fixed n_features."""\n        if self.dense_mode:\n            return self.n_features * len(self.blocks)\n        k = k if k is not None else self.sparsity_ctrl.k\n        return k * len(self.blocks)\n\n    def ptm_state_dicts(self):\n        """List of each block\'s ProbationaryMemoryTracker state, in block\n        order (``None`` for a dense-mode block, which has no PTM). PTM\n        objects are plain Python objects, not nn.Modules, so they are\n        NOT captured by ``state_dict()``/``load_state_dict()`` -- without\n        explicitly saving/restoring this, a resumed run starts every\n        layer\'s dead-feature tracking from scratch (miss/success counts,\n        probation membership all reset), even though the model weights\n        themselves resumed correctly."""\n        return [\n            block.ffn.ptm.state_dict() if hasattr(block.ffn, "ptm") else None\n            for block in self.blocks\n        ]\n\n    def load_ptm_state_dicts(self, states):\n        """Inverse of ``ptm_state_dicts``. Tolerant of a shorter/None list\n        (e.g. an older checkpoint saved before this existed) -- restores\n        whatever is present and leaves the rest at their freshly-initialized\n        state rather than raising."""\n        if not states:\n            return\n        for block, state in zip(self.blocks, states):\n            if state is not None and hasattr(block.ffn, "ptm"):\n                block.ffn.ptm.load_state_dict(state)\n\n    @torch.no_grad()\n    def generate(self, idx, max_new_tokens: int, temperature: float = 1.0, top_k: int = None,\n                 repetition_penalty: float = 1.0, use_hierarchical_exit: bool = False,\n                 eos_id: int = None, repetition_penalty_start: int = 0):\n        """Autoregressive sampling. If use_hierarchical_exit, the Validator\n        (Section 5.6/5.11) decides per step whether the stack of blocks\n        processed so far is already stable, exiting early instead of always\n        running every block -- a genuine, inference-time-only use of the\n        hierarchical-exit mechanism (see module docstring for why this isn\'t\n        done at train time). Blocks are run incrementally (each block\n        evaluated exactly once per token, carrying its output forward to the\n        next depth check) rather than restarting the forward pass from the\n        embeddings at every depth, so early exit is never more expensive\n        than the full dense path.\n\n        If ``eos_id`` is given and the batch is a single sequence, sampling\n        stops as soon as that token is produced (a genuine end-of-document\n        signal from the tokenizer -- see tokenizer.py/bulk_corpus.py),\n        instead of always running to ``max_new_tokens``. Ignored for\n        batch>1, where sequences could finish at different steps and\n        per-sequence stopping isn\'t implemented here.\n\n        ``repetition_penalty_start`` marks the first token that belongs to the\n        generated continuation. Tokens before this position are prompt/context\n        and are intentionally excluded from the repetition mask. This matters\n        for chat generation: a word appearing in the user\'s question should\n        not become artificially "expensive" just because it appeared in the\n        prompt. The default 0 preserves the historical full-sequence behavior\n        for callers that do not provide a boundary.\n\n        Note: `self.router` (RouterExpert) is constructed but not currently\n        consulted here -- exit is decided purely by the Validator\'s\n        stability check. Router is a reserved hook for a future trained\n        difficulty-prediction signal (Section 5.4/5.11), not yet wired in;\n        it receives no gradient today since its output is unused in\n        `forward`/`generate`.\n        """\n        if idx.ndim != 2 or idx.size(1) == 0:\n            raise ValueError("idx must have shape (batch, sequence) with a non-empty sequence")\n        if max_new_tokens < 0:\n            raise ValueError("max_new_tokens must be non-negative")\n        if temperature <= 0:\n            raise ValueError("temperature must be > 0")\n        if top_k is not None and top_k <= 0:\n            raise ValueError("top_k must be positive when provided")\n        if not 0 <= repetition_penalty_start <= idx.size(1):\n            raise ValueError(\n                "repetition_penalty_start must be between 0 and the initial "\n                "prompt length"\n            )\n\n        self.eval()\n        for _ in range(max_new_tokens):\n            idx_cond = idx[:, -self.max_seq_len:]\n            k = self.sparsity_ctrl.k\n\n            if use_hierarchical_exit:\n                self.validator.reset()\n                x = self.token_emb(idx_cond) + self.pos_emb(\n                    torch.arange(idx_cond.size(1), device=idx_cond.device).unsqueeze(0)\n                )\n                logits = None\n                for n_blocks_used, block in enumerate(self.blocks, start=1):\n                    x = block(x, k=k, use_ptm=False)\n                    logits = self.head(self.ln_f(x))[:, -1, :]\n                    top_id = logits.argmax(dim=-1)\n                    if self.validator.is_stable(top_id) and n_blocks_used < len(self.blocks):\n                        break\n            else:\n                logits, _ = self.forward(idx_cond, k=k, use_ptm=False)\n                logits = logits[:, -1, :]\n\n            logits = logits / temperature\n            if repetition_penalty != 1.0:\n                # Per-row mask: (batch, vocab), True where THAT row\'s own\n                # sequence-so-far contains the token -- not just row 0\'s,\n                # so batched generation penalizes each sequence by its own\n                # history (previously all rows were penalized using only\n                # sequence 0\'s tokens).\n                # Only generated tokens participate in the penalty. The\n                # prompt remains pure context. At the first generation step\n                # this slice is empty, which means no prompt token is penalized.\n                generated_history = idx[:, repetition_penalty_start:]\n                present = torch.zeros(\n                    idx.size(0), logits.shape[-1], dtype=torch.bool, device=logits.device\n                )\n                if generated_history.size(1) > 0:\n                    present.scatter_(1, generated_history, True)\n                logits = apply_repetition_penalty(logits, present, float(repetition_penalty))\n            if top_k is not None:\n                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))\n                logits[logits < v[:, [-1]]] = float("-inf")\n            probs = F.softmax(logits, dim=-1)\n            next_id = torch.multinomial(probs, num_samples=1)\n            idx = torch.cat([idx, next_id], dim=1)\n\n            if eos_id is not None and idx.size(0) == 1 and next_id.item() == eos_id:\n                break\n        return idx', encoding="utf-8")
print("Wrote:", path)
print("Bytes:", path.stat().st_size)


In [ ]:
# FILE: gamax1/finetune.py
from pathlib import Path

path = PROJECT_ROOT / 'gamax1/finetune.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text('"""\ngamax1/finetune.py\n===================\nInstruction/Q&A fine-tuning stage, run AFTER bulk pretraining (train.py)\non a base checkpoint. This is a deliberately separate script/loop from\ntrain.py rather than a mode flag, because the two training regimes\ndiffer in ways that shouldn\'t share code paths silently:\n\n  - Data:   train.py samples random windows out of one continuous token\n            stream (bulk_corpus.py). This script trains on discrete,\n            padded (prompt, answer) examples (instruction_data.py).\n  - Loss:   train.py computes loss on every token. This script computes\n            loss ONLY on answer tokens (see instruction_data.py\'s\n            loss_mask) -- so it does NOT use GamaX1Model.forward()\'s\n            built-in loss; it takes the logits and applies its own\n            masked cross-entropy here.\n  - LR/optimizer: a fresh, low-LR AdamW -- NOT the pretraining\n            optimizer state (that state encodes a very different LR\n            regime and would fight a low fine-tuning LR). Only the\n            MODEL weights (and sparsity-controller/PTM state, so\n            generation behavior stays consistent) are carried over.\n  - Sparsity level: --freeze_sparsity (default True) keeps k fixed at\n            whatever the base checkpoint converged to, rather than\n            calling sparsity_ctrl.step() -- fine-tuning on a small\n            dataset is exactly the situation the DynamicSparsityController\n            was never validated for (its trend-window/patience logic\n            assumes the long, noisy loss curve of bulk pretraining), so\n            letting a short fine-tune run perturb k risked an\n            uncontrolled, unvalidated side effect for no benefit.\n\nOPTIONAL REPLAY (catastrophic-forgetting mitigation): pass\n--replay_data_dir pointing at the same plain-.txt corpus used for\npretraining (or a subset of it) and --replay_ratio (e.g. 0.2) to mix\nthat fraction of steps as ordinary bulk-style next-token training,\ninterleaved with the instruction steps. This is optional and off by\ndefault (ratio 0.0) -- turn it on if you observe the fine-tuned model\'s\ngeneral fluency degrading relative to the base checkpoint.\n"""\n\nimport argparse\nimport os\nimport time\n\nimport torch\nimport torch.nn.functional as F\n\nfrom .model import GamaX1Model\nfrom .tokenizer import BPETokenizer\nfrom .train import checkpoint_dict, save_checkpoint, perplexity, get_lr_schedule\nfrom .instruction_data import (\n    load_pairs, InstructionDataset, make_collate_fn, split_train_val,\n)\n\n\ndef masked_cross_entropy(logits: torch.Tensor, targets: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:\n    """Cross-entropy averaged only over positions where mask is True.\n\n    logits: (batch, seq, vocab); targets/mask: (batch, seq). If mask is\n    all-False for a batch (shouldn\'t happen -- every example has at\n    least one answer token -- but defensive against a pathological\n    all-truncated batch), returns 0 with a warning rather than NaN from\n    a zero-count division.\n    """\n    flat_logits = logits.reshape(-1, logits.size(-1))\n    flat_targets = targets.reshape(-1)\n    flat_mask = mask.reshape(-1)\n    if not flat_mask.any():\n        print("[WARNING] a batch had zero loss-mask positions (fully truncated?); contributing 0 loss.")\n        return logits.sum() * 0.0\n    per_token = F.cross_entropy(flat_logits, flat_targets, reduction="none")\n    return per_token[flat_mask].mean()\n\n\ndef load_base_checkpoint(ckpt_path: str, device: str):\n    """Load a train.py checkpoint and rebuild the exact model/tokenizer it\n    was saved with (same pattern as generate.py\'s load_model)."""\n    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)\n    cfg = ckpt["config"]\n    tok = BPETokenizer(merges=ckpt["merges"])\n    model = GamaX1Model(\n        vocab_size=tok.vocab_size,\n        d_model=cfg["d_model"], n_heads=cfg["n_heads"], n_layers=cfg["n_layers"],\n        n_features=cfg["n_features"], max_seq_len=cfg["block_size"],\n        hex_influence=cfg.get("hex_influence", False),\n        sparsity_k_init=max(1, cfg["n_features"] // 2),\n        sparsity_k_min=max(1, cfg["n_features"] // 8),\n    ).to(device)\n    model.load_state_dict(ckpt["model_state"])\n    model.sparsity_ctrl.load_state_dict(ckpt["sparsity_controller_state"])\n    model.load_ptm_state_dicts(ckpt.get("ptm_states"))\n    return model, tok, cfg\n\n\n@torch.no_grad()\ndef evaluate(model, val_loader, device, k, use_amp):\n    model.eval()\n    losses = []\n    for xb, yb, mask in val_loader:\n        xb, yb, mask = xb.to(device), yb.to(device), mask.to(device)\n        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):\n            logits, _ = model(xb, targets=None, k=k, use_ptm=False)\n            loss = masked_cross_entropy(logits, yb, mask)\n        losses.append(loss.item())\n    return sum(losses) / max(len(losses), 1)\n\n\ndef main():\n    parser = argparse.ArgumentParser(\n        description="Fine-tune a pretrained GamaX1 checkpoint on instruction/Q&A JSON data "\n                     "with answer-only masked loss."\n    )\n    parser.add_argument("--init_from", type=str, required=True,\n                         help="Path to the pretrained checkpoint to fine-tune (e.g. "\n                              "gamax1_step_20000.pt from train.py).")\n    parser.add_argument("--data", type=str, required=True,\n                         help="Path to a .json/.jsonl file, or a directory of them (recursive). "\n                              "See instruction_data.py for supported record formats.")\n    parser.add_argument("--out_dir", type=str, default="checkpoints_finetune")\n    parser.add_argument("--max_len", type=int, default=512,\n                         help="Max tokens per example (prompt+answer+tags). Must not exceed "\n                              "the base checkpoint\'s block_size.")\n    parser.add_argument("--batch_size", type=int, default=8)\n    parser.add_argument("--epochs", type=int, default=3)\n    parser.add_argument("--lr", type=float, default=3e-5,\n                         help="Fine-tuning LR. Deliberately far below pretraining LR (often "\n                              "1e-4 to 3e-4) so the model adjusts to answer facts/format "\n                              "without unlearning general fluency.")\n    parser.add_argument("--warmup_steps", type=int, default=50)\n    parser.add_argument("--weight_decay", type=float, default=0.0)\n    parser.add_argument("--val_fraction", type=float, default=0.05)\n    parser.add_argument("--eval_interval", type=int, default=200,\n                         help="Evaluate on the held-out split every N steps.")\n    parser.add_argument("--checkpoint_interval", type=int, default=500)\n    parser.add_argument("--freeze_sparsity", action="store_true", default=True,\n                         help="Keep sparsity_k fixed at the base checkpoint\'s converged value "\n                              "instead of letting DynamicSparsityController adapt further "\n                              "during fine-tuning (default: on -- see module docstring).")\n    parser.add_argument("--no_freeze_sparsity", dest="freeze_sparsity", action="store_false")\n    parser.add_argument("--replay_data_dir", type=str, default=None,\n                         help="Optional: a plain-.txt bulk corpus directory to interleave as "\n                              "ordinary next-token training, mitigating catastrophic "\n                              "forgetting of general fluency. Off by default.")\n    parser.add_argument("--replay_ratio", type=float, default=0.0,\n                         help="Fraction of steps drawn from --replay_data_dir instead of the "\n                              "instruction data (0.0-1.0). Only used if --replay_data_dir is set.")\n    parser.add_argument("--replay_cache_dir", type=str, default="data/finetune_replay_cache")\n    parser.add_argument("--seed", type=int, default=0)\n    parser.add_argument("--no_amp", action="store_true")\n    parser.add_argument("--device", type=str, default=None)\n    args = parser.parse_args()\n\n    device = args.device or ("cuda" if torch.cuda.is_available() else "cpu")\n    torch.manual_seed(args.seed)\n\n    model, tok, base_cfg = load_base_checkpoint(args.init_from, device)\n    if args.max_len > base_cfg["block_size"]:\n        raise ValueError(f"--max_len={args.max_len} exceeds base checkpoint\'s block_size="\n                          f"{base_cfg[\'block_size\']}; the model was never trained on sequences "\n                          "this long.")\n\n    print(f"Loaded base checkpoint from {args.init_from} (step {base_cfg.get(\'step\', \'?\')})")\n    print(f"Sparsity k: {model.sparsity_ctrl.k} (frozen: {args.freeze_sparsity})")\n\n    pairs, stats = load_pairs(args.data)\n    print(f"Instruction data: {stats[\'files\']} file(s), {stats[\'records\']} record(s) -> "\n          f"{stats[\'pairs\']} (prompt, answer) example(s); {stats[\'skipped_records\']} record(s) "\n          "skipped (unrecognized format)")\n    if stats["skipped_records"] and stats["unmatched_key_sets"]:\n        print("[WARNING] Sample unmatched record key sets (first few) -- if this is your real "\n              "data format, tell me these keys and I\'ll add support for them:")\n        for keys in stats["unmatched_key_sets"]:\n            print(f"    {list(keys)}")\n    if not pairs:\n        raise ValueError(\n            "No usable (prompt, answer) examples found in --data. See the format-detection "\n            "rules documented at the top of instruction_data.py, or the skipped-record key "\n            "sets printed above, and either reshape the data or tell me the actual key names."\n        )\n\n    train_pairs, val_pairs = split_train_val(pairs, args.val_fraction, seed=args.seed)\n    print(f"Train examples: {len(train_pairs)} | Val examples: {len(val_pairs)}")\n\n    collate = make_collate_fn(tok.pad_id)\n    train_loader = torch.utils.data.DataLoader(\n        InstructionDataset(train_pairs, tok, args.max_len),\n        batch_size=args.batch_size, shuffle=True, collate_fn=collate,\n    )\n    val_loader = torch.utils.data.DataLoader(\n        InstructionDataset(val_pairs, tok, args.max_len),\n        batch_size=args.batch_size, shuffle=False, collate_fn=collate,\n    )\n\n    use_replay = args.replay_data_dir is not None and args.replay_ratio > 0.0\n    replay_data = replay_starts = None\n    if use_replay:\n        # Reuses the exact same bulk-corpus token cache machinery train.py\n        # uses, so replay batches are drawn from real pretraining-format\n        # text (see bulk_corpus.py) with no separate code path to maintain.\n        from .bulk_corpus import build_or_load_bulk_tokens\n        from .train import get_batch\n        replay_data, replay_starts, _sources = build_or_load_bulk_tokens(\n            args.replay_data_dir, args.replay_cache_dir, tok, rebuild=False,\n        )\n        print(f"Replay corpus: {len(replay_data):,} tokens from {args.replay_data_dir} "\n              f"(mixed in at ratio {args.replay_ratio:g})")\n\n    use_amp = device.startswith("cuda") and not args.no_amp\n    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)\n    optimizer = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)\n\n    steps_per_epoch = max(1, len(train_loader))\n    max_steps = steps_per_epoch * args.epochs\n    os.makedirs(args.out_dir, exist_ok=True)\n\n    step = 0\n    t0 = time.time()\n    best_val = float("inf")\n    k = model.sparsity_ctrl.k\n\n    for epoch in range(1, args.epochs + 1):\n        for xb, yb, mask in train_loader:\n            step += 1\n            lr = get_lr_schedule(step, max_steps, args.lr, args.warmup_steps)\n            for group in optimizer.param_groups:\n                group["lr"] = lr\n\n            model.train()\n            xb, yb, mask = xb.to(device), yb.to(device), mask.to(device)\n\n            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):\n                logits, _ = model(xb, targets=None, k=k, use_ptm=not args.freeze_sparsity)\n                loss = masked_cross_entropy(logits, yb, mask)\n\n            optimizer.zero_grad(set_to_none=True)\n            scaler.scale(loss).backward()\n            scaler.unscale_(optimizer)\n            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)\n            scaler.step(optimizer)\n            scaler.update()\n            if not args.freeze_sparsity:\n                model.sparsity_ctrl.step(loss.item())\n                k = model.sparsity_ctrl.k\n\n            if use_replay and torch.rand(1).item() < args.replay_ratio:\n                xb_r, yb_r = get_batch(replay_data, args.max_len, args.batch_size, device, replay_starts)\n                with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):\n                    _, replay_loss = model(xb_r, targets=yb_r, k=k)\n                optimizer.zero_grad(set_to_none=True)\n                scaler.scale(replay_loss).backward()\n                scaler.unscale_(optimizer)\n                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)\n                scaler.step(optimizer)\n                scaler.update()\n\n            if step % args.eval_interval == 0 or step == 1:\n                val_loss = evaluate(model, val_loader, device, k, use_amp)\n                print(f"epoch {epoch} | step {step:5d}/{max_steps} | lr {lr:.2e} | "\n                      f"train_loss {loss.item():.4f} | val_loss {val_loss:.4f} | "\n                      f"val_ppl {perplexity(val_loss):.2f} | {time.time() - t0:.1f}s")\n                if val_loss < best_val:\n                    best_val = val_loss\n                    save_checkpoint(\n                        os.path.join(args.out_dir, "gamax1_finetune_best.pt"),\n                        model, optimizer, tok, base_cfg, step, scaler=scaler,\n                    )\n\n            if args.checkpoint_interval and step % args.checkpoint_interval == 0:\n                save_checkpoint(\n                    os.path.join(args.out_dir, f"gamax1_finetune_step_{step}.pt"),\n                    model, optimizer, tok, base_cfg, step, scaler=scaler,\n                )\n\n    save_checkpoint(\n        os.path.join(args.out_dir, "gamax1_finetune_latest.pt"),\n        model, optimizer, tok, base_cfg, step, scaler=scaler,\n    )\n    tok.save(os.path.join(args.out_dir, "tokenizer.json"))\n    print(f"\\nDone. Best val_loss: {best_val:.4f} (val_ppl {perplexity(best_val):.2f}). "\n          f"Checkpoints saved under {args.out_dir}/")\n\n\nif __name__ == "__main__":\n    main()\n', encoding="utf-8")
print("Wrote:", path)
print("Bytes:", path.stat().st_size)


In [ ]:
# FILE: gamax1/generate.py
from pathlib import Path

path = PROJECT_ROOT / 'gamax1/generate.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text('"""\ngamax1/generate.py\n====================\nLoad a trained GamaX1 checkpoint and generate text.\n"""\n\nimport argparse\nimport os\n\nimport torch\n\nfrom .model import GamaX1Model\nfrom .tokenizer import BPETokenizer, CharTokenizer, WordTokenizer\n\n\ndef load_model(ckpt_path: str, device: str):\n    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)\n    cfg = ckpt["config"]\n    if cfg.get("tokenizer") == "bpe":\n        tok = BPETokenizer(merges=ckpt["merges"])\n    else:\n        tok_cls = WordTokenizer if cfg.get("tokenizer") == "word" else CharTokenizer\n        tok = tok_cls(vocab=ckpt["vocab"])\n    model = GamaX1Model(\n        vocab_size=tok.vocab_size,\n        d_model=cfg["d_model"],\n        n_heads=cfg["n_heads"],\n        n_layers=cfg["n_layers"],\n        n_features=cfg["n_features"],\n        max_seq_len=cfg["block_size"],\n        hex_influence=cfg.get("hex_influence", False),\n        sparsity_k_init=max(1, cfg["n_features"] // 2),\n        sparsity_k_min=max(1, cfg["n_features"] // 8),\n    ).to(device)\n    model.load_state_dict(ckpt["model_state"])\n    # Restore the trained sparsity level. DynamicSparsityController is a\n    # plain Python object (not an nn.Module), so its state is NOT captured\n    # by model.state_dict()/load_state_dict() above -- it\'s saved/restored\n    # separately, exactly like train.py already does on resume. Without\n    # this, generation would silently use the INITIAL k (n_features // 2)\n    # instead of whatever k the controller actually converged to during\n    # training, which can meaningfully change generation behavior since the\n    # sparse layer\'s active-feature count would no longer match what the\n    # model was trained and evaluated under.\n    model.sparsity_ctrl.load_state_dict(ckpt["sparsity_controller_state"])\n    model.eval()\n    return model, tok\n\n\ndef main():\n    parser = argparse.ArgumentParser(description="Generate text with a trained GamaX1 model.")\n    parser.add_argument("--ckpt", type=str, default="checkpoints/gamax1.pt")\n    parser.add_argument("--prompt", type=str, default="\\n")\n    parser.add_argument("--max_new_tokens", type=int, default=300)\n    parser.add_argument("--temperature", type=float, default=0.8)\n    parser.add_argument("--top_k", type=int, default=20)\n    parser.add_argument("--repetition_penalty", type=float, default=1.0,\n                        help="Penalize reusing tokens already in the sequence. >1.0 suppresses "\n                             "repetition (e.g. 1.2); 1.0 disables it (default: 1.0).")\n    parser.add_argument("--hierarchical_exit", action="store_true",\n                         help="Use Router/Validator hierarchical early exit at inference time.")\n    parser.add_argument("--stop_at_eos", action="store_true",\n                         help="Stop generation as soon as the tokenizer\'s reserved "\n                              "document-boundary token (eos_id) is produced, instead of "\n                              "always generating max_new_tokens. Only meaningful for a "\n                              "BPE tokenizer trained with the eos_id document-boundary fix "\n                              "(bulk_corpus.py inserting eos_id between files instead of "\n                              "\\"\\\\n\\\\n\\"); has no effect otherwise since the model will "\n                              "essentially never produce that id.")\n    parser.add_argument("--chat", action="store_true",\n                         help="Wrap --prompt as a user turn using the tokenizer\'s reserved "\n                              "<|user|>/<|assistant|> ids (instead of the literal text "\n                              "\\"User:\\"/\\"Assistant:\\") and generate the assistant\'s reply. "\n                              "Only meaningful for a checkpoint trained on the role-tagged "\n                              "corpus format (bulk_corpus.py\'s user_assistant source format); "\n                              "on any other checkpoint the model was never shown these ids "\n                              "and this will not produce a sensible reply. Use "\n                              "--stop_at_eos explicitly when the training data contains EOS turn boundaries.")\n    parser.add_argument("--no_stop_at_eos", action="store_true",\n                         help="With --chat, generate the full --max_new_tokens instead of "\n                              "stopping at the assistant turn\'s eos_id.")\n    parser.add_argument("--device", type=str, default=None)\n    args = parser.parse_args()\n\n    if args.max_new_tokens <= 0:\n        parser.error("--max_new_tokens must be > 0")\n    if args.temperature <= 0:\n        parser.error("--temperature must be > 0")\n    if args.top_k < 0:\n        parser.error("--top_k must be >= 0")\n    if args.repetition_penalty <= 0:\n        parser.error("--repetition_penalty must be > 0")\n    if args.no_stop_at_eos and not args.chat and not args.stop_at_eos:\n        parser.error("--no_stop_at_eos is only meaningful with --chat")\n\n    device = args.device or ("cuda" if torch.cuda.is_available() else "cpu")\n    model, tok = load_model(args.ckpt, device)\n\n    stop_at_eos = args.stop_at_eos and not args.no_stop_at_eos\n    eos_id = tok.eos_id if (stop_at_eos and hasattr(tok, "eos_id")) else None\n\n    if args.chat:\n        if not hasattr(tok, "user_id") or not hasattr(tok, "assistant_id"):\n            parser.error(\n                "--chat requires a tokenizer with reserved <|user|>/<|assistant|> ids "\n                "(this checkpoint\'s tokenizer does not have them -- it predates the "\n                "role-tagged corpus format, or was trained without it)."\n            )\n        prompt_ids = [tok.user_id] + tok.encode(args.prompt) + [tok.assistant_id]\n    else:\n        prompt_ids = tok.encode(args.prompt)\n\n    idx = torch.tensor([prompt_ids], dtype=torch.long, device=device)\n    # The prompt length is the boundary between user-provided context and the\n    # model\'s own answer. Passing it into generate() makes repetition penalty\n    # affect only generated tokens, not words that happened to occur in the\n    # user\'s question.\n    out = model.generate(\n        idx, max_new_tokens=args.max_new_tokens, temperature=args.temperature,\n        top_k=args.top_k, repetition_penalty=args.repetition_penalty,\n        repetition_penalty_start=idx.size(1),\n        use_hierarchical_exit=args.hierarchical_exit, eos_id=eos_id,\n    )\n    if args.chat:\n        # Only the newly generated continuation is the assistant\'s reply;\n        # decode_with_boundaries makes the prompt\'s own role tags visible\n        # too, for inspection.\n        print(tok.decode_with_boundaries(out[0].tolist()))\n    else:\n        print(tok.decode(out[0].tolist()))\n\n\nif __name__ == "__main__":\n    main()', encoding="utf-8")
print("Wrote:", path)
print("Bytes:", path.stat().st_size)


In [ ]:
# FILE: gamax1/mechanism_audit.py
from pathlib import Path

path = PROJECT_ROOT / 'gamax1/mechanism_audit.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text('"""Deterministic mathematical/mechanism invariants for Aetherion."""\nfrom dataclasses import dataclass, asdict\nimport json\n\n@dataclass\nclass AuditResult:\n    name:str\n    passed:bool\n    expected:object\n    observed:object\n    note:str=""\n\ndef compute_ratio_vs_dense(d_model:int,n_features:int,k:int)->float:\n    """Transparent local work proxy: sparse/dense = k/n_features."""\n    if not (0 <= k <= n_features): raise ValueError("k must satisfy 0 <= k <= n_features")\n    if n_features <= 0: raise ValueError("n_features must be positive")\n    return k/n_features\n\ndef audit_k_bounds(k,n_features):\n    return AuditResult("k_bounds",0<=k<=n_features,{"min":0,"max":n_features},k)\n\ndef audit_compute_ratio(d_model,n_features,k):\n    expected=k/n_features; observed=compute_ratio_vs_dense(d_model,n_features,k)\n    return AuditResult("compute_ratio_vs_dense",abs(expected-observed)<1e-12,expected,observed,\n                       "This is a transparent work proxy, not measured GPU speedup.")\n\ndef audit_topk_mask(mask,k,n_features):\n    import torch\n    counts=mask.bool().sum(-1)\n    expected=max(0,min(k,n_features))\n    lo=int(counts.min()) if counts.numel() else 0\n    hi=int(counts.max()) if counts.numel() else 0\n    return AuditResult("topk_active_count",lo==expected and hi==expected,\n                       expected,{"min":lo,"max":hi},\n                       "Nudge/override paths can intentionally violate exact-k.")\n\ndef audit_active_fraction(mask,n_features):\n    active=mask.bool().float().sum(-1).mean().item()\n    observed=active/n_features\n    return AuditResult("active_fraction",0<=observed<=1, "active/n_features", observed)\n\ndef run_basic_audit(d_model,n_features,k,batch_tokens=32):\n    import torch\n    mask=torch.zeros(batch_tokens,n_features,dtype=torch.bool)\n    if k: mask[:,:k]=True\n    return [audit_k_bounds(k,n_features),audit_compute_ratio(d_model,n_features,k),\n            audit_topk_mask(mask,k,n_features),audit_active_fraction(mask,n_features)]\n\ndef results_to_json(results): return [asdict(x) for x in results]\n\nif __name__=="__main__":\n    import argparse\n    p=argparse.ArgumentParser(); p.add_argument("--d-model",type=int,default=64)\n    p.add_argument("--n-features",type=int,default=256); p.add_argument("--k",type=int,default=64)\n    a=p.parse_args(); r=run_basic_audit(a.d_model,a.n_features,a.k)\n    print(json.dumps({"all_passed":all(x.passed for x in r),"results":results_to_json(r)},indent=2))\n    raise SystemExit(0 if all(x.passed for x in r) else 1)\n', encoding="utf-8")
print("Wrote:", path)
print("Bytes:", path.stat().st_size)


In [ ]:
# FILE: gamax1/experiment_tracker.py
from pathlib import Path

path = PROJECT_ROOT / 'gamax1/experiment_tracker.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text('"""Persistent experiment tracking for GamaX1/Aetherion.\n\nEach run gets its own directory. Metrics are append-only JSONL, while plots are\nregenerated from that run\'s metrics. This deliberately records measurements\nrather than inventing a speedup: the Aetherion compute ratio is kept separate\nfrom measured wall-clock/GPU throughput.\n"""\nfrom __future__ import annotations\nimport json, math, time\nfrom pathlib import Path\n\nclass ExperimentTracker:\n    def __init__(self, base_dir="experiments", run_name=None, config=None, resume=False):\n        self.base = Path(base_dir)\n        self.base.mkdir(parents=True, exist_ok=True)\n        if run_name:\n            self.run_dir = self.base / run_name\n        else:\n            stamp = time.strftime("run_%Y%m%d_%H%M%S")\n            self.run_dir = self.base / stamp\n            i=2\n            while self.run_dir.exists():\n                self.run_dir = self.base / f"{stamp}_{i}"; i+=1\n        self.run_dir.mkdir(parents=True, exist_ok=True)\n        (self.run_dir / "plots").mkdir(exist_ok=True)\n        self.metrics_path=self.run_dir/"metrics.jsonl"\n        self.timing_path=self.run_dir/"checkpoint_timing.jsonl"\n        if config is not None and not resume:\n            (self.run_dir/"config.json").write_text(json.dumps(config,indent=2,default=str))\n        self.start_wall=time.time()\n        self.last_checkpoint_step=None\n        self.last_checkpoint_wall=None\n        self.last_checkpoint_metrics=None\n\n    def log(self, **metrics):\n        record={"timestamp":time.time(),"elapsed_sec":time.time()-self.start_wall,**metrics}\n        with self.metrics_path.open("a",encoding="utf-8") as f:\n            f.write(json.dumps(record,sort_keys=True,default=float)+"\\n")\n        return record\n\n    def checkpoint_timing(self, step:int, checkpoint_kind="training", extra=None):\n        now=time.time()\n        interval_steps=None if self.last_checkpoint_step is None else step-self.last_checkpoint_step\n        interval_sec=None if self.last_checkpoint_wall is None else now-self.last_checkpoint_wall\n        steps_per_sec=(interval_steps/interval_sec) if interval_sec and interval_steps is not None and interval_sec>0 else None\n        rec={"timestamp":now,"checkpoint_kind":checkpoint_kind,"step":step,\n             "interval_steps":interval_steps,"interval_sec":interval_sec,\n             "steps_per_sec":steps_per_sec}\n        if extra: rec.update(extra)\n        with self.timing_path.open("a",encoding="utf-8") as f:\n            f.write(json.dumps(rec,sort_keys=True,default=float)+"\\n")\n        self.last_checkpoint_step=step; self.last_checkpoint_wall=now\n        self.last_checkpoint_metrics=rec\n        return rec\n\n    def write_summary(self, **summary):\n        (self.run_dir/"summary.json").write_text(json.dumps(summary,indent=2,default=str))\n\n    def plot(self):\n        try:\n            import matplotlib.pyplot as plt\n        except Exception as exc:\n            (self.run_dir/"plot_warning.txt").write_text(f"matplotlib unavailable: {exc}\\n")\n            return\n        rows=[]\n        if self.metrics_path.exists():\n            for line in self.metrics_path.read_text().splitlines():\n                try: rows.append(json.loads(line))\n                except json.JSONDecodeError: pass\n        if not rows: return\n        def series(key):\n            xs=[]; ys=[]\n            for r in rows:\n                if key in r and r[key] is not None:\n                    xs.append(r.get("step",len(xs))); ys.append(r[key])\n            return xs,ys\n        for key,title,ylabel in [("train_loss","Training loss","loss"),("val_loss","Validation loss","loss"),\n                                 ("train_ppl","Training perplexity","PPL"),("val_ppl","Validation perplexity","PPL"),\n                                 ("lr","Learning rate","LR"),("active_units_per_token","Active units/token","units")]:\n            x,y=series(key)\n            if not y: continue\n            fig=plt.figure(figsize=(7,4)); ax=fig.add_subplot(111); ax.plot(x,y); ax.set_title(title); ax.set_xlabel("step"); ax.set_ylabel(ylabel); fig.tight_layout(); fig.savefig(self.run_dir/"plots"/(key+".png")); plt.close(fig)\n        # Checkpoint interval timing as its own graph.\n        if self.timing_path.exists():\n            ts=[]\n            for line in self.timing_path.read_text().splitlines():\n                try:\n                    r=json.loads(line)\n                    if r.get("interval_sec") is not None: ts.append(r)\n                except json.JSONDecodeError: pass\n            if ts:\n                fig=plt.figure(figsize=(7,4)); ax=fig.add_subplot(111); ax.plot([r["step"] for r in ts],[r["interval_sec"] for r in ts],marker="o"); ax.set_title("Checkpoint interval time"); ax.set_xlabel("checkpoint step"); ax.set_ylabel("seconds since previous checkpoint"); fig.tight_layout(); fig.savefig(self.run_dir/"plots"/"checkpoint_interval_seconds.png"); plt.close(fig)\n\ndef compare_runs(experiments_dir="experiments", output="comparison.json"):\n    base=Path(experiments_dir); result=[]\n    if not base.exists(): return result\n    for d in sorted(p for p in base.iterdir() if p.is_dir()):\n        summary={"run":d.name}\n        sp=d/"summary.json"\n        if sp.exists():\n            try: summary.update(json.loads(sp.read_text()))\n            except Exception: pass\n        result.append(summary)\n    (base/output).write_text(json.dumps(result,indent=2,default=str))\n    return result\n', encoding="utf-8")
print("Wrote:", path)
print("Bytes:", path.stat().st_size)


## 4. Install dependencies and validate every embedded file


In [ ]:
%cd /content/GamaX1_Aetherion
!pip install -q -r requirements.txt

import sys, py_compile
from pathlib import Path

py_files = sorted(PROJECT_ROOT.rglob("*.py"))
for p in py_files:
    py_compile.compile(str(p), doraise=True)

print(f"Python files compiled successfully: {len(py_files)}")
print("Project tree:")
for p in sorted(PROJECT_ROOT.rglob("*")):
    if p.is_file():
        print(" ", p.relative_to(PROJECT_ROOT))


## 5. Sanity checks before touching the real corpus


In [ ]:
%cd /content/GamaX1_Aetherion
!python -m gamax1.mechanism_audit

from gamax1.layers import DynamicSparsityController
c = DynamicSparsityController(16, 2, 16)
c.k = 999
c.step(1.0)
assert c.k == 16
print("Dynamic sparsity upper-bound check: PASSED")

from gamax1.tokenizer import BPETokenizer
sample = "Aetherion test: sparse features, language modeling, and reproducible experiments."
tok = BPETokenizer(sample, vocab_size=300)
ids = tok.encode(sample)
decoded = tok.decode(ids)
assert isinstance(ids, list) and len(ids) > 0
print("BPE smoke round-trip: PASSED")
print("Decoded sample:", decoded[:120])


## 6. Verify the four real corpus sources


In [ ]:
from pathlib import Path

DATASETS = {
    "books_cleaned_v1": DATA_ROOT / "books_cleaned_v1",
    "Math_Reasoning_train": DATA_ROOT / "Math_Reasoning" / "train" / "books",
    "Conversations-200k_clean": DATA_ROOT / "Conversations-200k_clean",
    "Q&A": DATA_ROOT / "QnA",
}

total = 0
for name, path in DATASETS.items():
    files = [p for p in path.rglob("*") if p.is_file()] if path.exists() else []
    total += len(files)
    print(f"{name:28s} exists={path.exists()} files={len(files):,} path={path}")

print("Total files discovered:", f"{total:,}")
assert all(p.exists() for p in DATASETS.values()), "One or more corpus paths are missing."


## 7. Inspect/resume the bulk BPE cache — READ ONLY


In [ ]:
from pathlib import Path
import json

BULK_CACHE.mkdir(parents=True, exist_ok=True)

for name in [
    "tokenizer.json",
    "tokens.int32.bin",
    "metadata.json",
    "encode_progress.json",
    "encoding_checkpoint_timing.jsonl",
]:
    p = BULK_CACHE / name
    print(f"{name:32s} exists={p.exists()} size={p.stat().st_size if p.exists() else 0:,}")

meta = BULK_CACHE / "metadata.json"
if meta.exists():
    try:
        info = json.loads(meta.read_text(encoding="utf-8"))
        print("\nCache metadata keys:", sorted(info.keys()))
        for key in ("file_count", "token_count", "cache_version"):
            if key in info:
                print(f"{key}: {info[key]}")
    except Exception as exc:
        print("Could not parse metadata:", exc)

progress = BULK_CACHE / "encode_progress.json"
if progress.exists():
    try:
        info = json.loads(progress.read_text(encoding="utf-8"))
        print("\nEncoding progress:")
        for key in ("completed_files", "total_files", "last_checkpoint_files"):
            if key in info:
                print(f"{key}: {info[key]}")
    except Exception as exc:
        print("Could not parse progress:", exc)


## 8. OPTIONAL: force a clean corpus rebuild


In [ ]:
# DO NOT run this unless you intentionally want to rebuild the token cache.
# Normal training/resume should NOT use --rebuild_bulk_cache.
#
# The cache code itself protects old artifacts by quarantining stale state.
#
# Uncomment to run:
#
# !python -m gamax1.train --tokenizer bpe --data_dir "{DATA_ROOT}" \
#     --bulk_cache_dir "{BULK_CACHE}" --out_dir "{CKPT_DIR}/rebuild_test" \
#     --d_model 64 --n_heads 2 --n_layers 2 --n_features 256 \
#     --block_size 128 --batch_size 4 --max_steps 1 \
#     --checkpoint_interval 1 --experiment_dir "{EXPERIMENT_DIR}" \
#     --run_name rebuild_smoke --rebuild_bulk_cache


## 9. GPU smoke training — first actual training run


In [ ]:
%cd /content/GamaX1_Aetherion

SMOKE_CMD = (
    f'python -m gamax1.train --tokenizer bpe '
    f'--data_dir "{DATA_ROOT}" '
    f'--bulk_cache_dir "{BULK_CACHE}" '
    f'--out_dir "{CKPT_DIR / "smoke"}" '
    f'--d_model {SMOKE["d_model"]} --n_heads {SMOKE["n_heads"]} '
    f'--n_layers {SMOKE["n_layers"]} --n_features {SMOKE["n_features"]} '
    f'--block_size {SMOKE["block_size"]} --batch_size {SMOKE["batch_size"]} '
    f'--bpe_vocab_size 8000 '
    f'--max_steps {SMOKE["max_steps"]} '
    f'--eval_interval {SMOKE["eval_interval"]} '
    f'--checkpoint_interval {SMOKE["checkpoint_interval"]} '
    f'--experiment_dir "{EXPERIMENT_DIR}" '
    f'--run_name smoke_master'
)
print(SMOKE_CMD)
!{SMOKE_CMD}


## 10. 🚀 START FULL TRAINING — this is the main training cell


In [ ]:
%cd /content/GamaX1_Aetherion

# IMPORTANT:
# - Run the smoke test above first.
# - For normal/resume training, do NOT add --rebuild_bulk_cache.
# - Checkpoints are saved every 500 optimizer steps.
# - gamax1_latest.pt is the automatic resume point.
# - Numbered gamax1_step_N.pt files preserve checkpoint history.
#
# This cell is intentionally explicit: THIS is where the full training starts.

TRAIN_CMD = (
    f'python -m gamax1.train --tokenizer bpe '
    f'--data_dir "{DATA_ROOT}" '
    f'--bulk_cache_dir "{BULK_CACHE}" '
    f'--out_dir "{CKPT_DIR}" '
    f'--d_model {MODEL["d_model"]} '
    f'--n_heads {MODEL["n_heads"]} '
    f'--n_layers {MODEL["n_layers"]} '
    f'--n_features {MODEL["n_features"]} '
    f'--block_size {MODEL["block_size"]} '
    f'--batch_size {MODEL["batch_size"]} '
    f'--bpe_vocab_size {MODEL["bpe_vocab_size"]} '
    f'--lr {MODEL["lr"]} '
    f'--dropout {MODEL["dropout"]} '
    f'--max_steps {MODEL["max_steps"]} '
    f'--eval_interval {MODEL["eval_interval"]} '
    f'--checkpoint_interval {MODEL["checkpoint_interval"]} '
    f'--experiment_dir "{EXPERIMENT_DIR}" '
    f'--run_name full_v7_master'
)

print("FULL TRAINING COMMAND:")
print(TRAIN_CMD)
print("\nStarting training now...")
!{TRAIN_CMD}


## 11. Resume training after Colab disconnect


In [ ]:
%cd /content/GamaX1_Aetherion

# The trainer automatically resumes CKPT_DIR/gamax1_latest.pt when it exists.
# Keep the same architecture/tokenizer/cache settings as the original run.
#
# If the previous run stopped at step 7,500, this command continues from that
# checkpoint toward max_steps=30,000. It does NOT restart from step 0.

RESUME_CMD = TRAIN_CMD
print(RESUME_CMD)
!{RESUME_CMD}


## 12. Inspect checkpoints, experiment logs, timing and plots


In [ ]:
from pathlib import Path
import json

print("=== CHECKPOINTS ===")
if CKPT_DIR.exists():
    for p in sorted(CKPT_DIR.glob("gamax1*.pt")):
        print(p.name, f"{p.stat().st_size/1024/1024:.1f} MB")

print("\n=== EXPERIMENT RUNS ===")
if EXPERIMENT_DIR.exists():
    for p in sorted([p for p in EXPERIMENT_DIR.iterdir() if p.is_dir()]):
        print(p.name)

run_dir = EXPERIMENT_DIR / "full_v7_master"
if run_dir.exists():
    print("\n=== FILES IN FULL RUN ===")
    for p in sorted(run_dir.rglob("*")):
        if p.is_file():
            print(p.relative_to(run_dir))

    timing = run_dir / "checkpoint_timing.jsonl"
    if timing.exists():
        print("\n=== LAST CHECKPOINT TIMING RECORDS ===")
        rows = timing.read_text(encoding="utf-8").splitlines()
        for line in rows[-10:]:
            print(line)

    summary = run_dir / "summary.json"
    if summary.exists():
        print("\n=== SUMMARY ===")
        print(summary.read_text(encoding="utf-8")[:8000])


## 13. Generate from the latest checkpoint


In [ ]:
%cd /content/GamaX1_Aetherion

latest = CKPT_DIR / "gamax1_latest.pt"
if not latest.exists():
    raise FileNotFoundError(f"No latest checkpoint yet: {latest}")

GEN_CMD = (
    f'python -m gamax1.generate '
    f'--ckpt "{latest}" '
    f'--chat '
    f'--prompt "Explain what a language model learns during pretraining." '
    f'--max_new_tokens 200 '
    f'--stop_at_eos'
)
print(GEN_CMD)
!{GEN_CMD}


## 14. Optional instruction/Q&A fine-tuning


In [ ]:
%cd /content/GamaX1_Aetherion

# This stage is separate from bulk next-token pretraining.
# It uses answer-only masked loss and freezes sparsity by default.
#
# Point --data at the actual JSON/JSONL instruction corpus.
# Example using the QnA directory:
#
# FINETUNE_CMD = (
#     f'python -m gamax1.finetune '
#     f'--init_from "{CKPT_DIR / "gamax1_latest.pt"}" '
#     f'--data "{DATA_ROOT / "QnA"}" '
#     f'--out_dir "{CKPT_DIR / "finetune"}" '
#     f'--max_len 512 --batch_size 8 --epochs 3 '
#     f'--lr 3e-5 --checkpoint_interval 500 '
#     f'--replay_data_dir "{DATA_ROOT}" --replay_ratio 0.10 '
# )
# print(FINETUNE_CMD)
# !{FINETUNE_CMD}

print("Fine-tuning cell is prepared but intentionally does not start until you enable it.")


## 15. Controlled sparse-vs-dense comparison


In [ ]:
%cd /content/GamaX1_Aetherion

# This is a small paired mechanism experiment, not the full language-model run.
# It uses identical sampled batches and disables PTM/dropout for a cleaner
# sparse-FFN versus dense-FFN comparison.

DENSE_CMD = (
    'python compare_dense.py '
    '--steps 300 --d_model 64 --n_heads 2 --n_layers 2 '
    '--n_features 256 --block_size 64 --batch_size 16 '
    '--lr 3e-4 --dropout 0 --eval_batches 20'
)
print(DENSE_CMD)
!{DENSE_CMD}


## 16. Mathematical/mechanism audit


In [ ]:
%cd /content/GamaX1_Aetherion
!python -m gamax1.mechanism_audit

print("\n=== MATH CORRECTION LOG ===")
print((PROJECT_ROOT / "MATH_CORRECTION_LOG.md").read_text(encoding="utf-8"))


## 17. Final reproducibility / integrity report


In [ ]:
from pathlib import Path
import hashlib, json, time

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

print("=== PROJECT INTEGRITY ===")
for p in sorted(PROJECT_ROOT.rglob("*.py")):
    print(f"{p.relative_to(PROJECT_ROOT)} | sha256={sha256_file(p)[:16]}")

print("\n=== TRAINING ARTIFACTS ===")
for p in [
    CKPT_DIR / "gamax1_latest.pt",
    CKPT_DIR / "gamax1.pt",
    EXPERIMENT_DIR / "full_v7_master" / "metrics.jsonl",
    EXPERIMENT_DIR / "full_v7_master" / "checkpoint_timing.jsonl",
]:
    print(p, "EXISTS" if p.exists() else "missing")

print("\n=== INTERPRETATION ===")
print("- k / n_features is a local sparse-work proxy, not a guaranteed GPU speedup.")
print("- Wall-clock time, tokens/sec, memory, and checkpoint timing are measurements.")
print("- Do not change formulas merely to improve a benchmark.")
print("- Negative or inconclusive experiment results remain part of the research record.")
